# Current official-response GLSD-90 cross-pipeline configuration transfer

This notebook transfers only `theta=(a0, rho, tau)` within the same dataset. It never transfers `k`, response data, event geometry, boundary rules, candidate order, conflict handling, evaluator, or recognition/result synergy. The four exact replay gates must pass before a transfer cell can run. Existing evidence directories are read-only.


## 0. Runtime requirement

Select a Colab GPU runtime. Environment commands below are copied verbatim from the previously successful clean notebooks; package versions are not upgraded or substituted.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

print('DRIVE_MOUNT = PASS')


In [ ]:
from pathlib import Path

RUN_NAME = 'current_protocol_cross_pipeline_transfer_v1'
RUN_ROOT = Path('/content/drive/MyDrive') / RUN_NAME
if RUN_ROOT.exists():
    raise RuntimeError(f'Refusing to overwrite existing run: {RUN_ROOT}')
RUN_ROOT.mkdir(parents=True)
Path('/content/transfer_run_root.txt').write_text(str(RUN_ROOT), encoding='utf-8')
print('NEW_OUTPUT_ROOT =', RUN_ROOT)


## 1. BoostingVRME environment

Source: `BoostingVRME_environment_clean_v2.ipynb`. The successful configuration cells are retained in their original order and content.


## 2. 恢复官方 BoostingVRME 仓库并固定 commit


In [ ]:
%%bash
set -e

apt-get -qq update
apt-get -qq install -y git git-lfs curl bzip2 build-essential cmake ninja-build

cd /content
rm -rf /content/BoostingVRME

git clone https://github.com/zizheng-guo/BoostingVRME.git /content/BoostingVRME
cd /content/BoostingVRME

git checkout 689a7baa5edcb5831f6f2a911e22b4673fc70ac3
git lfs install
git lfs pull

echo "COMMIT=$(git rev-parse HEAD)"
test "$(git rev-parse HEAD)" = "689a7baa5edcb5831f6f2a911e22b4673fc70ac3"
echo "BOOSTINGVRME_REPO = PASS"


## 3. 创建原实验使用的 Python 3.8 / PyTorch 2.1 环境


In [ ]:
%%bash
set -e

cd /content
rm -rf /content/micromamba /content/boostvrme-env
mkdir -p /content/micromamba

curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest   | tar -xj -C /content/micromamba bin/micromamba

MM=/content/micromamba/bin/micromamba
ENV=/content/boostvrme-env

$MM create -y   -p "$ENV"   -c pytorch   -c nvidia   -c conda-forge   python=3.8   pip=23.3.1   pytorch=2.1.0   torchvision=0.16.0   torchaudio=2.1.0   pytorch-cuda=12.1

$MM run -p "$ENV" python - <<'PY'
import sys, torch, torchvision, torchaudio
print("Python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
assert sys.version.startswith("3.8")
assert torch.__version__.startswith("2.1.0")
assert torchvision.__version__.startswith("0.16.0")
assert torch.cuda.is_available()
print("BASE_TORCH_ENV = PASS")
PY


## 4. 安装 dlib 19.21.1


In [ ]:
%%bash
set -e

MM=/content/micromamba/bin/micromamba
ENV=/content/boostvrme-env

$MM install -y -p "$ENV" -c conda-forge "dlib=19.21.1"

$MM run -p "$ENV" python - <<'PY'
import dlib
print("dlib:", dlib.__version__)
assert dlib.__version__ == "19.21.1"
print("DLIB = PASS")
PY


## 5. 安装 BoostingVRME requirements（排除 dlib / Mamba 两包）

这一步沿用原 notebook 的做法。它可能暂时改动 torch 版本；
**下一步会按原成功流程强制恢复官方 PyTorch 2.1 stack**。


In [ ]:
%%bash
set -e

MM=/content/micromamba/bin/micromamba
ENV=/content/boostvrme-env

$MM run -p "$ENV" pip install "setuptools<70" wheel packaging ninja

cd /content/BoostingVRME

grep -v '^dlib' requirements.txt   | grep -v '^causal-conv1d'   | grep -v '^mamba-ssm'   > /content/requirements_boosting_base.txt

echo "=== Requirements ==="
cat /content/requirements_boosting_base.txt

$MM run -p "$ENV" pip install -r /content/requirements_boosting_base.txt

echo "BOOSTING_BASE_DEPENDENCIES = PASS"


## 6. 按原成功链恢复官方 PyTorch stack

这是关键步骤：项目 requirements 可能把 torch/torchvision 改成别的版本。
这里恢复到最终成功实验使用的版本。


In [ ]:
%%bash
set -e

MM=/content/micromamba/bin/micromamba
ENV=/content/boostvrme-env

# 先移除可能由依赖安装带来的 Mamba 二进制，避免 ABI 混用
$MM run -p "$ENV" pip uninstall -y mamba-ssm causal-conv1d || true

$MM install -y   --force-reinstall   -p "$ENV"   -c pytorch   -c nvidia   -c conda-forge   pytorch=2.1.0   torchvision=0.16.0   torchaudio=2.1.0   pytorch-cuda=12.1

$MM run -p "$ENV" python - <<'PY'
import torch, torchvision, torchaudio
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
assert torch.__version__.startswith("2.1.0")
assert torchvision.__version__.startswith("0.16.0")
assert torchaudio.__version__.startswith("2.1.0")
assert torch.version.cuda == "12.1"
assert torch.cuda.is_available()
print("OFFICIAL_PYTORCH_STACK_RESTORED = PASS")
PY


## 7. 清理旧 CUDA 扩展并安装与 Torch 2.1 ABI 匹配的 wheel


In [ ]:
%%bash
set -e

MM=/content/micromamba/bin/micromamba
ENV=/content/boostvrme-env

SITE=$($MM run -p "$ENV" python - <<'PY'
import site
print(site.getsitepackages()[0])
PY
)

rm -f "$SITE"/selective_scan_cuda*.so
rm -f "$SITE"/causal_conv1d_cuda*.so
$MM run -p "$ENV" pip cache purge || true

ABI=$($MM run -p "$ENV" python - <<'PY'
import torch
print("TRUE" if torch._C._GLIBCXX_USE_CXX11_ABI else "FALSE")
PY
)

echo "Detected CXX11 ABI = $ABI"

CAUSAL_WHEEL="causal_conv1d-1.4.0+cu122torch2.1cxx11abi${ABI}-cp38-cp38-linux_x86_64.whl"
MAMBA_WHEEL="mamba_ssm-2.2.2+cu122torch2.1cxx11abi${ABI}-cp38-cp38-linux_x86_64.whl"

CAUSAL_URL="https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/${CAUSAL_WHEEL}"
MAMBA_URL="https://github.com/state-spaces/mamba/releases/download/v2.2.2/${MAMBA_WHEEL}"

$MM run -p "$ENV" pip install --no-deps --force-reinstall "$CAUSAL_URL"
$MM run -p "$ENV" pip install --no-deps --force-reinstall "$MAMBA_URL"

echo "EXACT_MAMBA_WHEELS = PASS"


## 8. 固定 Transformers 兼容版本


In [ ]:
%%bash
set -e

MM=/content/micromamba/bin/micromamba
ENV=/content/boostvrme-env

$MM run -p "$ENV" pip install   --force-reinstall   "transformers==4.36.2"   "tokenizers==0.15.2"

echo "TRANSFORMERS_DOWNGRADE = PASS"


## 9. 最终严格检查


In [ ]:
%%bash
set -e

MM=/content/micromamba/bin/micromamba
ENV=/content/boostvrme-env

cd /content/BoostingVRME

$MM run -p "$ENV" python - <<'PY'
import sys
import torch
import torchvision
import torchaudio
import transformers
import tokenizers
import dlib
import causal_conv1d
import mamba_ssm
from mamba_ssm.modules.mamba_simple import Mamba

print("Python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("torchaudio:", torchaudio.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("dlib:", dlib.__version__)
print("mamba_ssm:", mamba_ssm.__version__)
print("causal_conv1d:", causal_conv1d.__version__)

assert sys.version.startswith("3.8")
assert torch.__version__.startswith("2.1.0")
assert torchvision.__version__.startswith("0.16.0")
assert torchaudio.__version__.startswith("2.1.0")
assert torch.version.cuda == "12.1"
assert torch.cuda.is_available()
assert transformers.__version__ == "4.36.2"
assert tokenizers.__version__ == "0.15.2"
assert dlib.__version__ == "19.21.1"

model = Mamba(d_model=16, d_state=16, d_conv=4, expand=2).cuda()
x = torch.randn(2, 32, 16, device="cuda")
with torch.no_grad():
    y = model(x)
assert y.shape == x.shape

print("Mamba forward:", tuple(y.shape))
print("STRICT_BOOSTINGVRME_ENVIRONMENT = PASS")
PY


## 2. ME-TST environment

Source: `GLSD_ME-TST_environment_clean.ipynb`. The successful configuration cells are retained in their original order and content.


## 2. 干净恢复 ME-TST 仓库

这里会删除 `/content/ME-TST` 的**临时 Colab 目录**后重新 clone。
不会删除 Google Drive 中的任何文件。
这样可以避免之前残留的不完整仓库导致 `Utils` 缺失。


In [ ]:
%cd /content
!rm -rf /content/ME-TST
!git clone https://github.com/zizheng-guo/ME-TST.git /content/ME-TST

from pathlib import Path

repo = Path("/content/ME-TST")
assert (repo / "Utils").exists(), "ME-TST/Utils not found after clean clone"
assert (repo / "Utils/mean_average_precision_str").exists(), "mean_average_precision_str folder missing"

print("METST_REPO_CLONE = PASS")
print("repo =", repo)


## 3. 安装 micromamba


In [ ]:
%cd /content
!rm -rf /content/bin
!wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj

from pathlib import Path
mamba = Path("/content/bin/micromamba")
assert mamba.exists(), "micromamba binary missing"
print("MICROMAMBA_INSTALL = PASS")
print("micromamba =", mamba)


## 4. 创建最终成功的 ME-TST 环境

对应原 notebook 最终采用的环境：
- Python 3.8
- PyTorch 2.1.0
- torchvision 0.16.0
- torchaudio 2.1.0
- CUDA 12.1


In [ ]:
!/content/bin/micromamba env remove -n ME-TST -y || true

!/content/bin/micromamba create -y -n ME-TST   -c pytorch -c nvidia -c conda-forge   python=3.8   pytorch=2.1.0   torchvision=0.16.0   torchaudio=2.1.0   pytorch-cuda=12.1   pip

!/content/bin/micromamba run -n ME-TST python --version


## 5. 验证 PyTorch / CUDA


In [ ]:
!/content/bin/micromamba run -n ME-TST python -c "import torch; print('torch =', torch.__version__); print('torch cuda =', torch.version.cuda); print('cuda available =', torch.cuda.is_available())"


## 6. 安装原 notebook 最终成功的基础依赖


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install   natsort==7.1.1   einops==0.8.0   tqdm==4.66.4   pandas==1.1.3   scikit-learn==0.23.2   scipy==1.5.4   xlrd==1.2.0   opencv-python-headless==4.5.1.48

!/content/bin/micromamba run -n ME-TST pip install --no-deps timm==1.0.7


## 7. 验证基础依赖与 ME-TST Utils


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST python -c "import torch,natsort,einops,timm,pandas,sklearn,scipy,cv2; print('torch =',torch.__version__); print('cuda =',torch.cuda.is_available()); print('BASIC_OK')"

!/content/bin/micromamba run -n ME-TST python -c "from Utils.mean_average_precision_str.mean_average_precision import MeanAveragePrecision2d; print('METST_UTILS_IMPORT = PASS')"


## 8. 安装 causal-conv1d


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install     causal-conv1d==1.4.0     --no-build-isolation


## 9. 安装 mamba-ssm


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install     mamba-ssm==1.2.0.post1     --no-build-isolation


## 10. 最终环境检查


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST python -c "import sys, torch, pandas, mamba_ssm, causal_conv1d; from Utils.mean_average_precision_str.mean_average_precision import MeanAveragePrecision2d; print('Python:', sys.version); print('torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print('pandas:', pandas.__version__); print('mamba_ssm: OK'); print('causal_conv1d: OK'); print('Utils metric: OK'); print('ME_TST_ENVIRONMENT = PASS')"


## 3. Restore sealed-compatible runners and transfer package

The two BoostingVRME runners below are exact snapshots from the audited formal experiment chain. ME-TST runners and GLSD cores are loaded directly from their frozen Drive evidence directories.


In [ ]:
%%writefile /content/BoostingVRME/boosting_official_glds_full.py
"""Official-response GLSD evaluation for BoostingVRME/SAMMLV.

This program never runs a backbone or training.  It consumes the recorder-made
official response cache and calls the repository's official spotting,
recognition, metric and NMS code.  Run it from the root of the official
BoostingVRME repository, for example::

    python boosting_official_glds_full.py \
        --cache /content/sammlv_official_full_responses.pkl \
        --output_dir /content/boosting_glsd_results

``--mode audit`` stops after the mandatory native exact-replay gate.  ``full``
(the default) runs the nested-LOSO GLSD evaluation only after that gate passes.
"""

from __future__ import annotations

import argparse
import csv
from datetime import datetime, timezone
import hashlib
import json
import pickle
import platform
import subprocess
import sys
from collections import Counter
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
from scipy.signal import find_peaks, peak_prominences


# Locked GLSD-90 protocol.  The ordering is also the final deterministic
# selection tie-break after F1, precision and FP.
SCALES = (1.0, 1.5, 2.0)
LOCAL_RADII = (1.0, 2.0, 3.0)
THRESHOLDS = (0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.75)
EXPECTED_RAW = (51, 144, 108)
EXPECTED_FULL = (51, 141, 108)


@dataclass(frozen=True)
class GLSDConfig:
    """One member of the locked 3 x 3 x 10 GLSD search grid."""

    reference_scale: float
    local_radius: float
    threshold: float

    @property
    def identifier(self) -> str:
        return (
            f"reference_scale={self.reference_scale:g}|"
            f"local_radius={self.local_radius:g}|threshold={self.threshold:g}"
        )


def configuration_grid() -> list[GLSDConfig]:
    return [
        GLSDConfig(reference, radius, threshold)
        for reference in SCALES
        for radius in LOCAL_RADII
        for threshold in THRESHOLDS
    ]


def metrics(counts) -> dict:
    tp, fp, fn = (int(value) for value in counts)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "F1": f1,
    }


def cache_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def file_sha256(path: Path) -> str:
    """Hash an existing artifact without changing its contents."""
    return cache_sha256(path)


def utc_timestamp() -> str:
    return datetime.now(timezone.utc).isoformat()


def official_git_commit() -> str:
    root = Path(__file__).resolve().parent
    return subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=root, text=True
    ).strip()


def as_counts(value) -> tuple[int, int, int]:
    return int(value["TP"]), int(value["FP"]), int(value["FN"])


def prediction_lists(predictions) -> list:
    return [np.asarray(video_predictions).tolist() for video_predictions in predictions]


def load_cache(path: Path) -> dict:
    with path.open("rb") as handle:
        cache = pickle.load(handle)
    if cache.get("schema_version") != 1 or cache.get("dataset") != "SAMMLV":
        raise RuntimeError("cache is not a schema-v1 SAMMLV official response cache")
    config = cache.get("config", {})
    required_config = {
        "emotion_type", "frame_skip", "k", "k_p", "synergy_strategy",
        "interval_strategy", "penalty_strategy",
    }
    missing = sorted(required_config.difference(config))
    if missing:
        raise RuntimeError("cache config missing: " + ", ".join(missing))
    if config["synergy_strategy"] != 1 or config["interval_strategy"] != 1:
        raise RuntimeError("this evaluator requires official SAMMLV synergy=1 and interval_strategy=1")
    direct_recognition = cache.get("direct_recognition", {})
    if set(direct_recognition) != {"pred_list", "gt_tp_list"}:
        raise RuntimeError("cache direct-recognition audit values are missing")
    direct_counts = cache.get("direct_inference_counts", {})
    if set(direct_counts) != {"raw", "full"}:
        raise RuntimeError("cache direct-inference count audit values are missing")
    videos = cache.get("videos", [])
    assert len(videos) == 79
    required_video = {
        "subject_id", "subject_index", "video_index_within_subject", "video_name",
        "spot_response", "recognition_sequence", "gt_intervals", "emotion_labels",
        "direct_spot_predictions",
    }
    for global_index, video in enumerate(videos):
        if video.get("video_index_global") != global_index:
            raise RuntimeError(f"cache global-video ordering failure at {global_index}")
        missing = sorted(required_video.difference(video))
        if missing:
            raise RuntimeError(f"video {global_index} missing: {', '.join(missing)}")
    return cache


def group_videos(videos: list[dict]) -> tuple[list[str], list[list[dict]]]:
    """Preserve the recorder's official subject/video ordering exactly."""
    subjects: list[str] = []
    grouped: list[list[dict]] = []
    current_index = None
    for video in videos:
        subject_index = int(video["subject_index"])
        if subject_index != current_index:
            if subject_index != len(grouped):
                raise RuntimeError(f"non-contiguous subject order at video {video['video_index_global']}")
            current_index = subject_index
            subjects.append(str(video["subject_id"]))
            grouped.append([])
        if int(video["video_index_within_subject"]) != len(grouped[-1]):
            raise RuntimeError(f"non-contiguous video order for subject {subject_index}")
        grouped[-1].append(video)
    assert len(subjects) == 29
    return subjects, grouped


def official_functions():
    """Import the exact repository implementations only after cache checks."""
    from Utils.mean_average_precision_str.mean_average_precision import MeanAveragePrecision2d
    import training_utils

    return MeanAveragePrecision2d, training_utils


def samples_and_emotions(grouped: list[list[dict]]) -> tuple[list, list]:
    samples = [[video["gt_intervals"] for video in subject] for subject in grouped]
    emotions = [[video["emotion_labels"] for video in subject] for subject in grouped]
    return samples, emotions


def full_counts_from_official_synergy(raw_counts, pred_list, gt_tp_list, config) -> tuple[int, int, int]:
    """The official synergy=1 neutral handling, without any new rule."""
    neutral = int(config["emotion_type"]) - 1
    tp_neutral = sum(prediction == neutral and target != -1 for prediction, target in zip(pred_list, gt_tp_list))
    fp_neutral = sum(prediction == neutral and target == -1 for prediction, target in zip(pred_list, gt_tp_list))
    tp, fp, fn = raw_counts
    return int(tp - tp_neutral), int(fp - fp_neutral), int(fn + tp_neutral)


def native_exact_replay(cache: dict, grouped: list[list[dict]]):
    """Replay the complete native official response path before GLSD is allowed."""
    MeanAveragePrecision2d, official = official_functions()
    config = cache["config"]
    final_samples, final_emotions = samples_and_emotions(grouped)
    metric_final = MeanAveragePrecision2d(num_classes=1)
    total_gt = 0
    pred_list, gt_tp_list = [], []
    pred_windows, pred_single = [], []
    decoded_predictions = []
    for subject_index, videos in enumerate(grouped):
        result_all = [np.asarray(video["spot_response"]) for video in videos]
        recognition_sequence = [np.asarray(video["recognition_sequence"]) for video in videos]
        predictions, _, total_gt, metric_video, metric_final = official.spotting(
            final_samples, subject_index, result_all, total_gt, metric_final,
            config["k_p"], config["interval_strategy"], "SAMMLV",
        )
        decoded_predictions.extend(prediction_lists(predictions))
        pred_list, _, gt_tp_list, pred_windows, pred_single = official.recognition(
            recognition_sequence, predictions, metric_video, final_emotions, subject_index,
            pred_list, gt_tp_list, final_samples, pred_windows, pred_single,
            config["frame_skip"], config["penalty_strategy"],
        )
    raw_counts = tuple(int(value) for value in official.sequence_evaluation(total_gt, metric_final))
    full_counts = full_counts_from_official_synergy(raw_counts, pred_list, gt_tp_list, config)
    direct_predictions = prediction_lists([video["direct_spot_predictions"] for video in cache["videos"]])
    return {
        "raw_counts": raw_counts,
        "full_counts": full_counts,
        "decoded_predictions_match": direct_predictions == decoded_predictions,
        "recognition_predictions_match": cache["direct_recognition"]["pred_list"] == pred_list,
        "matched_gt_sequence_match": cache["direct_recognition"]["gt_tp_list"] == gt_tp_list,
    }


class GLSDFeatures:
    """Locked label-free GLSD G/L evidence for one official spotting response."""

    def __init__(self, response, k: int):
        self.response = np.asarray(response, dtype=float)
        if self.response.ndim != 1 or len(self.response) < 3 or not np.isfinite(self.response).all():
            raise ValueError("official spotting response must be finite one-dimensional data")
        self.k = int(k)
        if self.k < 1:
            raise ValueError("k_p must be positive")
        self.scale_data = {}
        self.local_cache = {}
        physical = {}
        for scale in SCALES:
            width = min(len(self.response), max(1, round(scale * self.k)))
            if width not in physical:
                smooth = np.convolve(self.response, np.ones(width, dtype=float) / width, mode="same")
                peaks = find_peaks(smooth, distance=self.k)[0] if np.ptp(self.response) > 0 else np.empty(0, dtype=int)
                spread = max(float(np.ptp(smooth)), 1e-12)
                global_prominence = peak_prominences(smooth, peaks)[0] / spread
                physical[width] = (peaks, smooth, spread, global_prominence)
            self.scale_data[scale] = physical[width]
        # A duplicate integer smoothing width is one physical scale/vote.
        self.effective_scales = physical

    def evidence(self, reference: float, radius: float) -> tuple[np.ndarray, np.ndarray]:
        peaks, _, _, global_values = self.scale_data[reference]
        window = max(1, round(radius * self.k))
        tolerance = max(1, round(0.5 * self.k))
        for width, (other_peaks, smooth, spread, _) in self.effective_scales.items():
            key = width, window
            if key not in self.local_cache:
                values = []
                for peak in other_peaks:
                    left = float(np.min(smooth[max(0, peak - window):peak + 1]))
                    right = float(np.min(smooth[peak:min(len(smooth), peak + window + 1)]))
                    values.append(max(0.0, float(smooth[peak]) - max(left, right)) / spread)
                self.local_cache[key] = np.asarray(values, dtype=float)
        local_values = []
        for peak in peaks:
            aligned = []
            for width, (other_peaks, _, _, _) in self.effective_scales.items():
                values = self.local_cache[width, window]
                candidates = np.flatnonzero(np.abs(other_peaks - peak) <= tolerance)
                if not len(candidates):
                    aligned.append(0.0)  # locked missing-scale policy
                    continue
                chosen = min(candidates, key=lambda index: (abs(int(other_peaks[index]) - int(peak)), -values[index]))
                aligned.append(float(values[chosen]))
            local_values.append(float(np.median(aligned)))  # locked median aggregation
        evidence = np.column_stack((global_values, np.asarray(local_values, dtype=float)))
        return np.asarray(peaks, dtype=int), evidence

    def selected_peaks(self, config: GLSDConfig) -> np.ndarray:
        peaks, evidence = self.evidence(config.reference_scale, config.local_radius)
        if not len(peaks):
            return peaks
        global_values, local_values = evidence.T
        score = (global_values + local_values) / 2.0  # locked S(c)=(G(c)+L(c))/2
        return peaks[score >= config.threshold]


@contextmanager
def official_glsd_candidate_source(official, selected_by_video: list[np.ndarray]):
    """Feed GLSD-retained peaks to the *unmodified* official spotting() decoder.

    ``spotting()`` owns the boundary search, ``peak_new`` construction, candidate
    representation, metric registration and its direct call to official ``nms()``.
    Only its candidate source is substituted: GLSD has already constructed and
    thresholded the reference-scale candidates.  The original finder is restored
    even if the official function raises.
    """
    original_find_peaks = official.find_peaks
    position = 0

    def glsd_find_peaks(signal, *args, **kwargs):
        nonlocal position
        if position >= len(selected_by_video):
            raise RuntimeError("official spotting requested more candidate lists than cached videos")
        peaks = np.asarray(selected_by_video[position], dtype=int)
        position += 1
        if np.any(peaks < 0) or np.any(peaks >= len(signal)):
            raise RuntimeError("GLSD candidate lies outside its official response")
        return peaks, {}

    official.find_peaks = glsd_find_peaks
    try:
        yield
        if position != len(selected_by_video):
            raise RuntimeError("official spotting did not consume every GLSD candidate list")
    finally:
        official.find_peaks = original_find_peaks


def decode_glsd_subject(
    grouped, final_samples, subject_index: int, glsd_config: GLSDConfig,
    official, MeanAveragePrecision2d, cache_config, with_recognition: bool,
):
    """Decode one subject through official spotting; optionally official recognition."""
    videos = grouped[subject_index]
    responses = [np.asarray(video["spot_response"]) for video in videos]
    candidates = [GLSDFeatures(response, int(cache_config["k_p"])).selected_peaks(glsd_config) for response in responses]
    metric_final = MeanAveragePrecision2d(num_classes=1)
    total_gt = 0
    with official_glsd_candidate_source(official, candidates):
        predictions, _, total_gt, metric_video, metric_final = official.spotting(
            final_samples, subject_index, responses, total_gt, metric_final,
            cache_config["k_p"], cache_config["interval_strategy"], "SAMMLV",
        )
    raw_counts = tuple(int(value) for value in official.sequence_evaluation(total_gt, metric_final))
    prediction_matches = sorted(
        metric_video.value(iou_thresholds=0.5)[0.5][0]["pred_match_gt"].items()
    )
    if not with_recognition:
        return raw_counts, predictions, None, None, prediction_matches, metric_video
    _, final_emotions = samples_and_emotions(grouped)
    sequences = [np.asarray(video["recognition_sequence"]) for video in videos]
    pred_list, _, gt_tp_list, _, _ = official.recognition(
        sequences, predictions, metric_video, final_emotions, subject_index,
        [], [], final_samples, [], [], cache_config["frame_skip"], cache_config["penalty_strategy"],
    )
    return raw_counts, predictions, pred_list, gt_tp_list, prediction_matches, metric_video


def selection_order(inner_counts: np.ndarray) -> np.ndarray:
    """Locked selection order: F1, precision, fewer FP, fixed GLSD-grid order."""
    tp, fp, fn = inner_counts.astype(float).T
    denominator = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, denominator, out=np.zeros_like(tp), where=denominator > 0)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    grid_order = np.arange(len(inner_counts))
    return np.lexsort((grid_order, fp, -precision, -f1))


def choose_configuration(inner_counts: np.ndarray) -> int:
    """Return the winner under the unchanged locked selection order."""
    return int(selection_order(inner_counts)[0])


def plain_scalar(value):
    """Convert NumPy scalar labels to portable CSV/JSON/Pickle values."""
    return value.item() if isinstance(value, np.generic) else value


def auditable_prediction_evidence(
    videos, predictions, prediction_matches, metric_video, pred_list, gt_tp_list, official, cache_config,
):
    """Persist evaluator/recognition outputs without participating in evaluation.

    The supplied match map is read directly from the official metric object and
    the recognition labels are returned by the official recognition() call.
    These rows are evidence only: they never feed candidate selection, spotting,
    recognition, synergy, or metric computation.
    """
    matches_by_video = dict(prediction_matches)
    if set(matches_by_video).difference(range(len(videos))):
        raise RuntimeError("official metric returned an invalid video index")
    neutral_label = int(cache_config["emotion_type"]) - 1
    label_index = 0
    raw = np.zeros(3, dtype=np.int64)
    full = np.zeros(3, dtype=np.int64)
    csv_rows = []
    pickle_videos = []
    for video_index, (video, video_predictions) in enumerate(zip(videos, predictions)):
        video_predictions = np.asarray(video_predictions).tolist()
        matched_indices = list(matches_by_video.get(video_index, []))
        if len(matched_indices) != len(video_predictions):
            raise RuntimeError("official metric/prediction cardinality mismatch")
        evaluator_rows = metric_video.match_table[0]
        evaluator_rows = evaluator_rows[evaluator_rows["img_id"] == video_index]
        if len(evaluator_rows) != len(video_predictions):
            raise RuntimeError("official evaluator evidence cardinality mismatch")
        video_rows = []
        matched_gt = set()
        for prediction_index, (prediction, gt_index) in enumerate(zip(video_predictions, matched_indices)):
            if label_index >= len(pred_list) or label_index >= len(gt_tp_list):
                raise RuntimeError("official recognition cardinality mismatch")
            recognition_prediction = plain_scalar(pred_list[label_index])
            gt_recognition_label = plain_scalar(gt_tp_list[label_index])
            label_index += 1
            # The official metric may serialize unmatched indexes as either -1
            # or a one-element [-1] container; normalize only for persistence.
            gt_index = int(np.asarray(gt_index).reshape(-1)[0])
            if gt_index >= 0:
                matched_gt.add(gt_index)
            interval = [int(prediction[0]), int(prediction[2])]
            evaluator_iou = np.asarray(evaluator_rows.iloc[prediction_index]["iou"], dtype=float)
            best_iou = float(evaluator_iou.max()) if evaluator_iou.size else 0.0
            raw_status = "TP" if gt_index >= 0 else "FP"
            is_neutral = recognition_prediction == neutral_label
            if raw_status == "TP":
                raw += (1, 0, 0)
                if is_neutral:
                    synergy_action = "neutral_tp_to_fn"
                    final_contribution = (0, 0, 1)
                else:
                    synergy_action = "preserve_tp"
                    final_contribution = (1, 0, 0)
            else:
                raw += (0, 1, 0)
                if is_neutral:
                    synergy_action = "neutral_fp_removed"
                    final_contribution = (0, 0, 0)
                else:
                    synergy_action = "preserve_fp"
                    final_contribution = (0, 1, 0)
            full += final_contribution
            row = {
                "record_type": "prediction",
                "subject": str(video["subject_id"]),
                "subject_index": int(video["subject_index"]),
                "video": str(video["video_name"]),
                "video_index_global": int(video["video_index_global"]),
                "prediction_index": prediction_index,
                "onset": interval[0],
                "offset": interval[1],
                "peak": int(prediction[6]),
                "config_id": None,
                "matched_gt_index": gt_index,
                "best_iou": best_iou,
                "raw_status": raw_status,
                "recognition_prediction": recognition_prediction,
                "matched_gt_recognition_label": gt_recognition_label,
                "classified_neutral": bool(is_neutral),
                "synergy_action": synergy_action,
                "raw_TP_contribution": 1 if raw_status == "TP" else 0,
                "raw_FP_contribution": 1 if raw_status == "FP" else 0,
                "raw_FN_contribution": 0,
                "final_TP_contribution": final_contribution[0],
                "final_FP_contribution": final_contribution[1],
                "final_FN_contribution": final_contribution[2],
            }
            video_rows.append(row)
            csv_rows.append(row)
        for gt_index, gt in enumerate(video["gt_intervals"]):
            if gt_index in matched_gt:
                continue
            raw += (0, 0, 1)
            full += (0, 0, 1)
            row = {
                "record_type": "unmatched_gt",
                "subject": str(video["subject_id"]),
                "subject_index": int(video["subject_index"]),
                "video": str(video["video_name"]),
                "video_index_global": int(video["video_index_global"]),
                "prediction_index": None,
                "onset": int(gt[0]),
                "offset": int(gt[2]),
                "peak": int(gt[1]),
                "config_id": None,
                "matched_gt_index": gt_index,
                "best_iou": 0.0,
                "raw_status": "FN",
                "recognition_prediction": None,
                "matched_gt_recognition_label": plain_scalar(official.convertLabel(video["emotion_labels"][gt_index])),
                "classified_neutral": False,
                "synergy_action": "preserve_fn",
                "raw_TP_contribution": 0,
                "raw_FP_contribution": 0,
                "raw_FN_contribution": 1,
                "final_TP_contribution": 0,
                "final_FP_contribution": 0,
                "final_FN_contribution": 1,
            }
            video_rows.append(row)
            csv_rows.append(row)
        pickle_videos.append({
            "subject_id": str(video["subject_id"]),
            "subject_index": int(video["subject_index"]),
            "video_name": str(video["video_name"]),
            "video_index_global": int(video["video_index_global"]),
            "evidence": video_rows,
        })
    if label_index != len(pred_list) or label_index != len(gt_tp_list):
        raise RuntimeError("unconsumed official recognition records")
    return tuple(int(value) for value in raw), tuple(int(value) for value in full), csv_rows, pickle_videos


def make_run_manifest(cache_path: Path, output_dir: Path, video_count: int, subject_count: int, started_at: str) -> dict:
    """Create machine-generated provenance; hashes are never entered manually."""
    import scipy
    import sklearn

    script_path = Path(__file__).resolve()
    log_path = output_dir / "run_stdout_stderr.log"
    return {
        "cache_absolute_path": str(cache_path.resolve()),
        "cache_sha256": file_sha256(cache_path),
        "script_absolute_path": str(script_path),
        "script_sha256": file_sha256(script_path),
        "official_boostingvrme_git_commit": official_git_commit(),
        "python_version": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "numpy_version": np.__version__,
        "scipy_version": scipy.__version__,
        "sklearn_version": sklearn.__version__,
        "command_line": list(sys.argv),
        "dataset": "SAMMLV",
        "video_count": video_count,
        "subject_count": subject_count,
        "glsd_grid_size": len(configuration_grid()),
        "glsd_grid_constants": {
            "reference_scales": list(SCALES),
            "local_radii": list(LOCAL_RADII),
            "thresholds": list(THRESHOLDS),
        },
        "native_expected_counts": {"raw": list(EXPECTED_RAW), "full": list(EXPECTED_FULL)},
        "execution_started_at_utc": started_at,
        "execution_finished_at_utc": utc_timestamp(),
        "run_stdout_stderr_path": str(log_path.resolve()),
        "run_stdout_stderr_sha256": None,
        "run_stdout_stderr_status": "PENDING_FINALIZE_LOG",
    }


def finalize_log(output_dir: Path, log_path: Path) -> None:
    """Bind the externally tee'd complete log to an already completed run."""
    manifest_path = output_dir / "run_manifest.json"
    if not manifest_path.is_file():
        raise RuntimeError("run_manifest.json is required before log finalization")
    if not log_path.is_file():
        raise RuntimeError(f"log does not exist: {log_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    digest = file_sha256(log_path)
    digest_path = output_dir / "run_stdout_stderr.sha256"
    digest_path.write_text(f"{digest}  {log_path.name}\n", encoding="utf-8")
    manifest.update({
        "run_stdout_stderr_path": str(log_path.resolve()),
        "run_stdout_stderr_sha256": digest,
        "run_stdout_stderr_status": "FINALIZED",
        "log_finalized_at_utc": utc_timestamp(),
    })
    write_json(manifest_path, manifest)
    print(f"run_stdout_stderr.sha256 = {digest}")


def write_json(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def write_csv(path: Path, rows: list[dict]) -> None:
    with path.open("w", newline="", encoding="utf-8-sig") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def write_audit(path: Path, audit: dict, selections: list[dict] | None = None) -> None:
    lines = [
        "# BoostingVRME SAMMLV GLSD Full-Pipeline Audit",
        "",
        "## Cache and native gate",
        "",
        f"- Cache path: `{audit['cache_path']}`",
        f"- Cache SHA-256: `{audit['cache_sha256']}`",
        f"- Videos: {audit['video_count']} / 79.",
        f"- Subjects: {audit['subject_count']} / 29.",
        f"- Native raw: `{audit['native']['raw_counts']}`; F1 = {audit['native']['raw']['F1']:.10f}.",
        f"- Native full: `{audit['native']['full_counts']}`; F1 = {audit['native']['full']['F1']:.10f}.",
        f"- Decoded spotting predictions exact match: `{audit['native']['decoded_predictions_match']}`.",
        f"- Recognition prediction sequence exact match: `{audit['native']['recognition_predictions_match']}`.",
        f"- Matched-GT sequence exact match: `{audit['native']['matched_gt_sequence_match']}`.",
        f"- Native gate: `{audit['native']['status']}`.",
        "",
        "## Locked implementation",
        "",
        "- GLSD grid: 90 = 3 reference scales x 3 local radii x 10 thresholds.",
        "- GLSD score: `S(c) = (G(c) + L(c)) / 2`; missing-scale support is zero and multi-scale aggregation is median.",
        "- Official BoostingVRME interval geometry is reused by direct calls to `training_utils.spotting()` with `interval_strategy=1`; its native `nms()` remains in that call path.",
        "- Official recognition aggregation is reused by direct calls to `training_utils.recognition()` over each newly decoded GLSD interval.",
        "- Official neutral/result synergy is applied with the cached official `emotion_type` convention.",
        "- Nested selection pools each of the 28 non-outer subjects as an inner held-out validation subject; outer subject is asserted absent.",
    ]
    if selections is not None:
        lines.extend(["", "## Per-outer selected configuration", ""])
        for row in selections:
            lines.append(
                f"- Subject `{row['outer_subject']}`: `{row['config']}`; "
                f"inner TP/FP/FN = {row['inner_TP']}/{row['inner_FP']}/{row['inner_FN']}; "
                f"inner F1 = {row['inner_F1']:.10f}."
            )
        glsd = audit["glsd"]
        lines.extend([
            "", "## GLSD official-response result", "",
            f"- Raw: `{tuple(glsd['raw_counts'])}`; F1 = {glsd['raw']['F1']:.10f}.",
            f"- Full: `{tuple(glsd['full_counts'])}`; F1 = {glsd['full']['F1']:.10f}.",
            f"- Delta raw: {glsd['delta_raw']:.10f}.",
            f"- Delta full: {glsd['delta_full']:.10f}.",
            "", "## Verdict", "", "`BOOSTING_SAMMLV_GLSD_FULL_VALIDATED = PASS`",
        ])
    else:
        lines.extend([
            "", "## Verdict", "",
            f"`NATIVE_EXACT_REPLAY = {audit['native']['status']}`"
            + (" (audit mode; GLSD was not executed)." if audit["native"]["status"] == "PASS" else ""),
        ])
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def run_full(
    cache: dict, cache_path: Path, output_dir: Path, native: dict, started_at: str,
) -> None:
    MeanAveragePrecision2d, official = official_functions()
    subjects, grouped = group_videos(cache["videos"])
    final_samples, _ = samples_and_emotions(grouped)
    grid = configuration_grid()
    assert len(grid) == 90

    # One raw official decode per (configuration, subject).  This table contains
    # no model fitting and no outer labels are read during a fold's selection.
    raw_table = np.zeros((len(grid), len(subjects), 3), dtype=np.int64)
    for subject_index in range(len(subjects)):
        for config_index, glsd_config in enumerate(grid):
            raw_counts, _, _, _, _, _ = decode_glsd_subject(
                grouped, final_samples, subject_index, glsd_config,
                official, MeanAveragePrecision2d, cache["config"], with_recognition=False,
            )
            raw_table[config_index, subject_index] = raw_counts

    choices: list[int] = []
    selection_rows: list[dict] = []
    search_rows: list[dict] = []
    inner_fold_rows: list[dict] = []
    selection_traces: list[dict] = []
    for outer_index, outer_subject in enumerate(subjects):
        inner_indexes = [index for index in range(len(subjects)) if index != outer_index]
        inner_selection_subjects = [subjects[index] for index in inner_indexes]
        assert outer_subject not in inner_selection_subjects
        # These 28 validation subjects are the inner-LOSO folds.  Pooling their
        # validation counts is the locked pooled-inner selection objective.
        pooled_inner = raw_table[:, inner_indexes, :].sum(axis=1)
        order = selection_order(pooled_inner)
        selected = int(order[0])
        ranks = np.empty(len(grid), dtype=int)
        ranks[order] = np.arange(1, len(grid) + 1)
        choices.append(selected)
        selected_metrics = metrics(pooled_inner[selected])
        selection_rows.append({
            "outer_subject": outer_subject,
            "config_id": selected,
            "config": grid[selected].identifier,
            **asdict(grid[selected]),
            "inner_subject_count": len(inner_selection_subjects),
            "inner_subjects": inner_selection_subjects,
            "inner_TP": selected_metrics["TP"],
            "inner_FP": selected_metrics["FP"],
            "inner_FN": selected_metrics["FN"],
            "inner_precision": selected_metrics["precision"],
            "inner_recall": selected_metrics["recall"],
            "inner_F1": selected_metrics["F1"],
        })
        trace_candidates = []
        for config_index, glsd_config in enumerate(grid):
            current_metrics = metrics(pooled_inner[config_index])
            selected_flag = config_index == selected
            search_rows.append({
                "outer_subject": outer_subject,
                "config_id": config_index,
                **asdict(glsd_config),
                "pooled_inner_TP": current_metrics["TP"],
                "pooled_inner_FP": current_metrics["FP"],
                "pooled_inner_FN": current_metrics["FN"],
                "precision": current_metrics["precision"],
                "recall": current_metrics["recall"],
                "F1": current_metrics["F1"],
                "grid_order": config_index,
                "final_rank": int(ranks[config_index]),
                "selected": selected_flag,
            })
            trace_candidates.append({
                "config_id": config_index,
                **asdict(glsd_config),
                "pooled_inner_counts": list(map(int, pooled_inner[config_index])),
                "precision": current_metrics["precision"],
                "recall": current_metrics["recall"],
                "F1": current_metrics["F1"],
                "selection_key": {
                    "higher_F1": current_metrics["F1"],
                    "higher_precision": current_metrics["precision"],
                    "fewer_FP": current_metrics["FP"],
                    "fixed_grid_order": config_index,
                },
                "final_rank": int(ranks[config_index]),
                "selected": selected_flag,
            })
            for inner_index in inner_indexes:
                inner_metrics = metrics(raw_table[config_index, inner_index])
                inner_fold_rows.append({
                    "outer_subject": outer_subject,
                    "inner_validation_subject": subjects[inner_index],
                    "config_id": config_index,
                    **asdict(glsd_config),
                    "TP": inner_metrics["TP"],
                    "FP": inner_metrics["FP"],
                    "FN": inner_metrics["FN"],
                })
        selection_traces.append({
            "outer_subject": outer_subject,
            "inner_subjects": inner_selection_subjects,
            "tie_break_order": [
                "higher F1", "higher precision", "fewer FP", "fixed grid order",
            ],
            "candidates": trace_candidates,
            "winner_config_id": selected,
            "winner_rank": int(ranks[selected]),
        })

    glsd_raw = np.zeros(3, dtype=np.int64)
    glsd_full = np.zeros(3, dtype=np.int64)
    final_predictions = []
    auditable_predictions = []
    per_video_rows = []
    per_subject_rows = []
    for subject_index, videos in enumerate(grouped):
        selected = choices[subject_index]
        raw_counts, predictions, pred_list, gt_tp_list, prediction_matches, metric_video = decode_glsd_subject(
            grouped, final_samples, subject_index, grid[selected], official,
            MeanAveragePrecision2d, cache["config"], with_recognition=True,
        )
        # The selected config's separately evaluated outer raw count must replay exactly.
        assert tuple(raw_counts) == tuple(raw_table[selected, subject_index])
        full_counts = full_counts_from_official_synergy(raw_counts, pred_list, gt_tp_list, cache["config"])
        evidence_raw, evidence_full, video_rows, video_evidence = auditable_prediction_evidence(
            videos, predictions, prediction_matches, metric_video, pred_list, gt_tp_list, official, cache["config"],
        )
        assert evidence_raw == tuple(raw_counts)
        assert evidence_full == tuple(full_counts)
        glsd_raw += np.asarray(raw_counts, dtype=np.int64)
        glsd_full += np.asarray(full_counts, dtype=np.int64)
        raw_metrics = metrics(raw_counts)
        full_metrics = metrics(full_counts)
        per_subject_rows.append({
            "outer_subject": subjects[subject_index],
            "config_id": selected,
            "raw_TP": raw_metrics["TP"],
            "raw_FP": raw_metrics["FP"],
            "raw_FN": raw_metrics["FN"],
            "raw_F1": raw_metrics["F1"],
            "full_TP": full_metrics["TP"],
            "full_FP": full_metrics["FP"],
            "full_FN": full_metrics["FN"],
            "full_F1": full_metrics["F1"],
        })
        for row in video_rows:
            row["config_id"] = selected
            row["selected_config"] = grid[selected].identifier
        for evidence in video_evidence:
            evidence["selected_config_id"] = selected
            evidence["selected_config"] = asdict(grid[selected])
            for row in evidence["evidence"]:
                row["config_id"] = selected
        per_video_rows.extend(video_rows)
        auditable_predictions.extend(video_evidence)
        for video, predictions_for_video in zip(videos, predictions):
            final_predictions.append({
                "subject_id": video["subject_id"],
                "subject_index": int(video["subject_index"]),
                "video_name": video["video_name"],
                "video_index_global": int(video["video_index_global"]),
                "selected_config_id": selected,
                "selected_config": asdict(grid[selected]),
                "predictions": np.asarray(predictions_for_video).tolist(),
            })

    raw_counts = tuple(int(value) for value in glsd_raw)
    full_counts = tuple(int(value) for value in glsd_full)
    assert tuple(sum(row[f"raw_{name}_contribution"] for row in per_video_rows) for name in ("TP", "FP", "FN")) == raw_counts
    assert tuple(sum(row[f"final_{name}_contribution"] for row in per_video_rows) for name in ("TP", "FP", "FN")) == full_counts
    assert tuple(sum(row[f"raw_{name}"] for row in per_subject_rows) for name in ("TP", "FP", "FN")) == raw_counts
    assert tuple(sum(row[f"full_{name}"] for row in per_subject_rows) for name in ("TP", "FP", "FN")) == full_counts
    audit = {
        "cache_path": str(cache_path),
        "cache_sha256": cache_sha256(cache_path),
        "video_count": len(cache["videos"]),
        "subject_count": len(subjects),
        "native": {**native, "raw": metrics(native["raw_counts"]), "full": metrics(native["full_counts"]), "status": "PASS"},
        "glsd": {
            "raw_counts": raw_counts,
            "full_counts": full_counts,
            "raw": metrics(raw_counts),
            "full": metrics(full_counts),
            "delta_raw": metrics(raw_counts)["F1"] - metrics(native["raw_counts"])["F1"],
            "delta_full": metrics(full_counts)["F1"] - metrics(native["full_counts"])["F1"],
        },
    }
    json_rows = selection_rows
    csv_rows = [{**row, "inner_subjects": ";".join(row["inner_subjects"])} for row in selection_rows]
    write_json(output_dir / "outer_selected_configs.json", json_rows)
    write_csv(output_dir / "outer_selected_configs.csv", csv_rows)
    write_csv(output_dir / "glsd_search_all_90x29.csv", search_rows)
    write_csv(output_dir / "glsd_search_inner_fold_counts.csv", inner_fold_rows)
    write_json(output_dir / "outer_selection_trace.json", selection_traces)
    with (output_dir / "glsd_final_predictions.pkl").open("wb") as handle:
        pickle.dump(final_predictions, handle, protocol=pickle.HIGHEST_PROTOCOL)
    with (output_dir / "glsd_final_predictions_auditable.pkl").open("wb") as handle:
        pickle.dump(auditable_predictions, handle, protocol=pickle.HIGHEST_PROTOCOL)
    write_csv(output_dir / "per_video_evaluation.csv", per_video_rows)
    write_csv(output_dir / "per_subject_counts.csv", per_subject_rows)
    write_json(
        output_dir / "run_manifest.json",
        make_run_manifest(cache_path, output_dir, len(cache["videos"]), len(subjects), started_at),
    )
    write_audit(output_dir / "BOOSTING_SAMMLV_GLSD_FULL_AUDIT.md", audit, selection_rows)

    print("==============================")
    print("GLSD official-response")
    print("==============================")
    print("Raw  : TP/FP/FN = %d/%d/%d, F1 = %.4f" % (*raw_counts, audit["glsd"]["raw"]["F1"]))
    print("Full : TP/FP/FN = %d/%d/%d, F1 = %.4f" % (*full_counts, audit["glsd"]["full"]["F1"]))
    print("Delta raw  = %.10f" % audit["glsd"]["delta_raw"])
    print("Delta full = %.10f" % audit["glsd"]["delta_full"])
    print("BOOSTING_SAMMLV_GLSD_FULL_VALIDATED = PASS")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--cache", type=Path, help="recorder-produced SAMMLV official response cache")
    parser.add_argument("--output_dir", required=True, type=Path, help="directory for audit and evaluation outputs")
    parser.add_argument("--mode", choices=("audit", "full"), default="full")
    parser.add_argument(
        "--finalize_log", type=Path,
        help="hash an externally tee'd completed run log and update run_manifest.json; no evaluation is run",
    )
    args = parser.parse_args()
    args.output_dir.mkdir(parents=True, exist_ok=True)
    if args.finalize_log is not None:
        if args.cache is not None:
            parser.error("--finalize_log does not accept --cache")
        finalize_log(args.output_dir, args.finalize_log)
        return
    if args.cache is None:
        parser.error("--cache is required unless --finalize_log is used")
    if not args.cache.is_file():
        parser.error(f"cache does not exist: {args.cache}")
    started_at = utc_timestamp()

    cache = load_cache(args.cache)
    subjects, grouped = group_videos(cache["videos"])
    assert len(cache["videos"]) == 79
    assert len(subjects) == 29
    native = native_exact_replay(cache, grouped)
    expected_direct_raw = as_counts(cache["direct_inference_counts"]["raw"])
    expected_direct_full = as_counts(cache["direct_inference_counts"]["full"])
    native_passed = (
        native["raw_counts"] == EXPECTED_RAW == expected_direct_raw
        and native["full_counts"] == EXPECTED_FULL == expected_direct_full
        and native["decoded_predictions_match"]
        and native["recognition_predictions_match"]
        and native["matched_gt_sequence_match"]
    )
    print("==============================")
    print("Native exact replay")
    print("==============================")
    print("Raw  : TP/FP/FN = %d/%d/%d, F1 = %.4f" % (*native["raw_counts"], metrics(native["raw_counts"])["F1"]))
    print("Full : TP/FP/FN = %d/%d/%d, F1 = %.4f" % (*native["full_counts"], metrics(native["full_counts"])["F1"]))
    print("decoded spotting predictions exact match =", native["decoded_predictions_match"])
    print("recognition prediction sequence exact match =", native["recognition_predictions_match"])
    print("matched-GT sequence exact match =", native["matched_gt_sequence_match"])
    audit = {
        "cache_path": str(args.cache),
        "cache_sha256": cache_sha256(args.cache),
        "video_count": len(cache["videos"]),
        "subject_count": len(subjects),
        "native": {
            **native,
            "raw": metrics(native["raw_counts"]),
            "full": metrics(native["full_counts"]),
            "status": "PASS" if native_passed else "FAIL",
        },
    }
    if not native_passed:
        write_audit(args.output_dir / "BOOSTING_SAMMLV_GLSD_FULL_AUDIT.md", audit)
        print("NATIVE_EXACT_REPLAY = FAIL", file=sys.stderr)
        raise SystemExit(1)
    print("NATIVE_EXACT_REPLAY = PASS")
    if args.mode == "audit":
        write_audit(args.output_dir / "BOOSTING_SAMMLV_GLSD_FULL_AUDIT.md", audit)
        return
    run_full(cache, args.cache, args.output_dir, native, started_at)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/BoostingVRME/boosting_official_glds_full_casme3.py

#!/usr/bin/env python3

import argparse
import contextlib
import csv
import hashlib
import io
import json
import os
import pickle
import platform
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import scipy
import sklearn

from scipy.signal import (
    find_peaks,
    peak_prominences,
)

import training_utils as tu


# ============================================================
# Locked protocol
# ============================================================

DATASET = "CASME_3"

EXPECTED_SUBJECTS = 94
EXPECTED_VIDEOS = 462
EXPECTED_GT = 853

EXPECTED_NATIVE_RAW = (
    84, 793, 769
)

EXPECTED_NATIVE_FULL = (
    84, 786, 769
)

OFFICIAL_COMMIT = (
    "689a7baa5edcb5831f6f2a911e22b4673fc70ac3"
)

REFERENCE_SCALES = [
    1.0,
    1.5,
    2.0,
]

LOCAL_RADII = [
    1.0,
    2.0,
    3.0,
]

THRESHOLDS = [
    0.05,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.55,
    0.60,
    0.65,
    0.75,
]


# ============================================================
# Basic helpers
# ============================================================

def utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()


def sha256_file(path):

    path = Path(path)

    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def normalize(obj):

    if isinstance(obj, np.ndarray):
        return normalize(obj.tolist())

    if isinstance(obj, np.generic):
        return obj.item()

    if isinstance(obj, tuple):
        return [
            normalize(x)
            for x in obj
        ]

    if isinstance(obj, list):
        return [
            normalize(x)
            for x in obj
        ]

    if isinstance(obj, dict):
        return {
            str(k): normalize(v)
            for k, v in obj.items()
        }

    return obj


def metrics(counts):

    tp, fp, fn = [
        int(x)
        for x in counts
    ]

    p = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0.0
    )

    r = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0.0
    )

    f1 = (
        2 * tp / (
            2 * tp + fp + fn
        )
        if 2 * tp + fp + fn > 0
        else 0.0
    )

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": p,
        "recall": r,
        "f1": f1,
    }


def add_counts(items):

    tp = sum(
        int(x[0])
        for x in items
    )

    fp = sum(
        int(x[1])
        for x in items
    )

    fn = sum(
        int(x[2])
        for x in items
    )

    return (
        tp,
        fp,
        fn,
    )


def write_csv(
    path,
    rows,
    fieldnames,
):

    path = Path(path)

    with path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
        )

        writer.writeheader()

        for row in rows:
            writer.writerow(row)


def official_metric_counts(
    metric,
    total_gt,
):

    result = metric.value(
        iou_thresholds=0.5
    )[0.5][0]

    tp = int(
        sum(result["tp"])
    )

    fp = int(
        sum(result["fp"])
    )

    fn = int(total_gt - tp)

    return (
        tp,
        fp,
        fn,
    )


def interval_iou_inclusive(
    pred,
    gt,
):

    ps = int(pred[0])
    pe = int(pred[2])

    gs = int(gt[0])
    ge = int(gt[2])

    inter = max(
        0,
        min(pe, ge)
        - max(ps, gs)
        + 1,
    )

    if inter <= 0:
        return 0.0

    plen = pe - ps + 1
    glen = ge - gs + 1

    union = (
        plen
        + glen
        - inter
    )

    return (
        inter / union
        if union > 0
        else 0.0
    )


# ============================================================
# Load cache
# ============================================================

def load_cache(path):

    with open(path, "rb") as f:
        cache = pickle.load(f)

    assert (
        cache["dataset"]
        == DATASET
    )

    assert (
        cache["video_count"]
        == EXPECTED_VIDEOS
    )

    assert (
        cache["subject_count"]
        == EXPECTED_SUBJECTS
    )

    videos = sorted(
        cache["videos"],
        key=lambda x:
            int(
                x[
                    "video_index_global"
                ]
            ),
    )

    assert [
        int(
            v[
                "video_index_global"
            ]
        )
        for v in videos
    ] == list(
        range(
            EXPECTED_VIDEOS
        )
    )

    gt_count = sum(
        len(v["gt_intervals"])
        for v in videos
    )

    assert gt_count == EXPECTED_GT, (
        gt_count
    )

    return (
        cache,
        videos,
    )


def group_videos(videos):

    grouped = [
        []
        for _ in range(
            EXPECTED_SUBJECTS
        )
    ]

    for video in videos:

        idx = int(
            video[
                "subject_index"
            ]
        )

        grouped[idx].append(
            video
        )

    assert all(
        len(x) > 0
        for x in grouped
    )

    return grouped


# ============================================================
# Native exact replay hard gate
# ============================================================

def native_exact_replay(
    cache,
    grouped,
):

    config = cache["config"]

    final_samples = [
        [
            v["gt_intervals"]
            for v in subject
        ]
        for subject in grouped
    ]

    final_emotions = [
        [
            v["emotion_labels"]
            for v in subject
        ]
        for subject in grouped
    ]

    metric_final = (
        tu.MeanAveragePrecision2d(
            num_classes=1
        )
    )

    total_gt = 0

    pred_list = []
    gt_tp_list = []

    pred_window_list = []
    pred_single_list = []

    replay_predictions = []

    # Ensure official find_peaks
    original_find_peaks = (
        tu.find_peaks
    )

    try:

        for subject_index in range(
            EXPECTED_SUBJECTS
        ):

            subject = grouped[
                subject_index
            ]

            result_all = [
                np.asarray(
                    v["spot_response"],
                    dtype=float,
                )
                for v in subject
            ]

            result1_all = [
                np.asarray(
                    v[
                        "recognition_sequence"
                    ]
                )
                for v in subject
            ]

            (
                preds,
                _,
                total_gt,
                metric_video,
                metric_final,
            ) = tu.spotting(
                final_samples,
                subject_index,
                result_all,
                total_gt,
                metric_final,
                int(
                    config["k_p"]
                ),
                int(
                    config[
                        "interval_strategy"
                    ]
                ),
                DATASET,
            )

            replay_predictions.extend(
                [
                    normalize(x)
                    for x in preds
                ]
            )

            with contextlib.redirect_stdout(
                io.StringIO()
            ):

                (
                    pred_list,
                    _,
                    gt_tp_list,
                    pred_window_list,
                    pred_single_list,
                ) = tu.recognition(
                    result1_all,
                    preds,
                    metric_video,
                    final_emotions,
                    subject_index,
                    pred_list,
                    gt_tp_list,
                    final_samples,
                    pred_window_list,
                    pred_single_list,
                    int(
                        config[
                            "frame_skip"
                        ]
                    ),
                    int(
                        config[
                            "penalty_strategy"
                        ]
                    ),
                )

    finally:

        tu.find_peaks = (
            original_find_peaks
        )

    raw_counts = (
        official_metric_counts(
            metric_final,
            total_gt,
        )
    )

    tp_neutral = 0
    fp_neutral = 0

    neutral_label = (
        int(
            config[
                "emotion_type"
            ]
        )
        - 1
    )

    for prediction, target in zip(
        pred_list,
        gt_tp_list,
    ):

        if (
            int(prediction)
            == neutral_label
        ):

            if int(target) == -1:
                fp_neutral += 1

            else:
                tp_neutral += 1

    full_counts = (
        raw_counts[0]
        - tp_neutral,

        raw_counts[1]
        - fp_neutral,

        raw_counts[2]
        + tp_neutral,
    )

    direct_predictions = [
        normalize(
            v[
                "direct_spot_predictions"
            ]
        )
        for subject in grouped
        for v in subject
    ]

    prediction_exact = (
        direct_predictions
        == replay_predictions
    )

    recognition_exact = (
        normalize(
            cache[
                "direct_recognition"
            ]["pred_list"]
        )
        ==
        normalize(
            pred_list
        )
    )

    gt_exact = (
        normalize(
            cache[
                "direct_recognition"
            ]["gt_tp_list"]
        )
        ==
        normalize(
            gt_tp_list
        )
    )

    passed = (
        raw_counts
        == EXPECTED_NATIVE_RAW

        and full_counts
        == EXPECTED_NATIVE_FULL

        and prediction_exact
        and recognition_exact
        and gt_exact
    )

    return {
        "passed": passed,

        "raw_counts":
            raw_counts,

        "full_counts":
            full_counts,

        "prediction_exact":
            prediction_exact,

        "recognition_exact":
            recognition_exact,

        "matched_gt_exact":
            gt_exact,
    }


# ============================================================
# Locked GLSD feature construction
# ============================================================

class GLSDFeatures:

    def __init__(
        self,
        response,
        k,
    ):

        self.x = np.asarray(
            response,
            dtype=float,
        ).reshape(-1)

        self.T = len(
            self.x
        )

        self.k = max(
            1,
            int(k),
        )

        self.scale_to_width = {}

        self.curves = {}
        self.peaks = {}
        self.ranges = {}

        self.local_cache = {}
        self.score_cache = {}

        if self.T == 0:
            return

        # Constant original curve -> no candidates
        self.constant = (
            float(
                np.ptp(
                    self.x
                )
            )
            <= 1e-12
        )

        for scale in REFERENCE_SCALES:

            width = min(
                self.T,
                max(
                    1,
                    int(
                        round(
                            scale
                            * self.k
                        )
                    ),
                ),
            )

            self.scale_to_width[
                float(scale)
            ] = int(width)

            if width in self.curves:
                continue

            kernel = (
                np.ones(
                    width,
                    dtype=float,
                )
                / float(width)
            )

            y = np.convolve(
                self.x,
                kernel,
                mode="same",
            )

            self.curves[
                width
            ] = y

            self.ranges[
                width
            ] = max(
                float(
                    np.ptp(y)
                ),
                1e-12,
            )

            if self.constant:

                peaks = np.array(
                    [],
                    dtype=int,
                )

            else:

                peaks, _ = find_peaks(
                    y,
                    distance=self.k,
                )

                peaks = np.asarray(
                    peaks,
                    dtype=int,
                )

            self.peaks[
                width
            ] = peaks

    def local_values(
        self,
        width,
        rho,
    ):

        key = (
            int(width),
            float(rho),
        )

        if key in self.local_cache:
            return self.local_cache[
                key
            ]

        y = self.curves[
            width
        ]

        peaks = self.peaks[
            width
        ]

        dynamic_range = (
            self.ranges[
                width
            ]
        )

        radius = max(
            1,
            int(
                round(
                    float(rho)
                    * self.k
                )
            ),
        )

        q = {}

        for peak in peaks:

            p = int(peak)

            left_start = max(
                0,
                p - radius,
            )

            right_end = min(
                self.T - 1,
                p + radius,
            )

            left_base = float(
                np.min(
                    y[
                        left_start:
                        p + 1
                    ]
                )
            )

            right_base = float(
                np.min(
                    y[
                        p:
                        right_end + 1
                    ]
                )
            )

            local = max(
                0.0,
                float(y[p])
                - max(
                    left_base,
                    right_base,
                ),
            )

            q[p] = (
                local
                / dynamic_range
            )

        self.local_cache[
            key
        ] = q

        return q

    def scores(
        self,
        reference_scale,
        rho,
    ):

        key = (
            float(reference_scale),
            float(rho),
        )

        if key in self.score_cache:
            return self.score_cache[
                key
            ]

        if self.T == 0 or self.constant:

            result = {
                "peaks": np.array(
                    [],
                    dtype=int,
                ),
                "G": np.array(
                    [],
                    dtype=float,
                ),
                "L": np.array(
                    [],
                    dtype=float,
                ),
                "S": np.array(
                    [],
                    dtype=float,
                ),
            }

            self.score_cache[
                key
            ] = result

            return result

        ref_width = (
            self.scale_to_width[
                float(
                    reference_scale
                )
            ]
        )

        ref_curve = (
            self.curves[
                ref_width
            ]
        )

        ref_peaks = (
            self.peaks[
                ref_width
            ]
        )

        if len(ref_peaks) == 0:

            result = {
                "peaks": ref_peaks,
                "G": np.array(
                    [],
                    dtype=float,
                ),
                "L": np.array(
                    [],
                    dtype=float,
                ),
                "S": np.array(
                    [],
                    dtype=float,
                ),
            }

            self.score_cache[
                key
            ] = result

            return result

        prominence = (
            peak_prominences(
                ref_curve,
                ref_peaks,
            )[0]
        )

        G = (
            prominence
            / self.ranges[
                ref_width
            ]
        )

        delta = max(
            1,
            int(
                round(
                    0.5
                    * self.k
                )
            ),
        )

        local_by_width = {
            width:
                self.local_values(
                    width,
                    rho,
                )

            for width in self.curves
        }

        L_values = []

        effective_widths = list(
            self.curves.keys()
        )

        for candidate in ref_peaks:

            c = int(candidate)

            aligned_values = []

            for width in effective_widths:

                peaks = (
                    self.peaks[
                        width
                    ]
                )

                q = (
                    local_by_width[
                        width
                    ]
                )

                valid = [
                    int(p)
                    for p in peaks
                    if abs(
                        int(p) - c
                    ) <= delta
                ]

                if not valid:

                    aligned_values.append(
                        0.0
                    )

                    continue

                best = min(
                    valid,
                    key=lambda p: (
                        abs(p - c),
                        -float(
                            q[p]
                        ),
                        p,
                    ),
                )

                aligned_values.append(
                    float(
                        q[best]
                    )
                )

            L_values.append(
                float(
                    np.median(
                        aligned_values
                    )
                )
            )

        L = np.asarray(
            L_values,
            dtype=float,
        )

        S = (
            np.asarray(
                G,
                dtype=float,
            )
            + L
        ) / 2.0

        result = {
            "peaks":
                np.asarray(
                    ref_peaks,
                    dtype=int,
                ),

            "G":
                np.asarray(
                    G,
                    dtype=float,
                ),

            "L":
                L,

            "S":
                S,
        }

        self.score_cache[
            key
        ] = result

        return result

    def selected_peaks(
        self,
        reference_scale,
        rho,
        tau,
    ):

        evidence = self.scores(
            reference_scale,
            rho,
        )

        peaks = (
            evidence[
                "peaks"
            ]
        )

        score = (
            evidence[
                "S"
            ]
        )

        if len(peaks) == 0:
            return []

        selected = peaks[
            score
            >= float(tau)
        ]

        return [
            int(x)
            for x in selected
        ]


# ============================================================
# Grid
# ============================================================

def build_grid():

    grid = []

    config_id = 0

    for a0 in REFERENCE_SCALES:

        for rho in LOCAL_RADII:

            for tau in THRESHOLDS:

                grid.append({
                    "config_id":
                        config_id,

                    "reference_scale":
                        float(a0),

                    "local_radius":
                        float(rho),

                    "threshold":
                        float(tau),

                    "grid_order":
                        config_id,
                })

                config_id += 1

    assert len(grid) == 90

    return grid


# ============================================================
# Official Boosting geometry adapter
# ============================================================

@contextlib.contextmanager
def patched_candidate_source(
    selected_per_video,
):

    original = (
        tu.find_peaks
    )

    state = {
        "index": 0
    }

    def fake_find_peaks(
        *_args,
        **_kwargs,
    ):

        i = state["index"]

        if i >= len(
            selected_per_video
        ):
            raise RuntimeError(
                "GLSD candidate source "
                "called too many times"
            )

        selected = (
            selected_per_video[
                i
            ]
        )

        state["index"] += 1

        return (
            np.asarray(
                selected,
                dtype=int,
            ),
            {},
        )

    tu.find_peaks = (
        fake_find_peaks
    )

    try:
        yield

    finally:

        tu.find_peaks = (
            original
        )

        if (
            state["index"]
            != len(
                selected_per_video
            )
        ):
            raise RuntimeError(
                "GLSD candidate-source "
                "call count mismatch: "
                f'{state["index"]} vs '
                f'{len(selected_per_video)}'
            )


# ============================================================
# Decode one subject under one GLSD configuration
# ============================================================

def decode_subject(
    subject_index,
    config_item,
    grouped,
    feature_bank,
    final_samples,
    final_emotions,
    cache_config,
    need_recognition=False,
):

    subject = grouped[
        subject_index
    ]

    selected_per_video = [
        feature_bank[
            subject_index
        ][video_index].selected_peaks(
            config_item[
                "reference_scale"
            ],
            config_item[
                "local_radius"
            ],
            config_item[
                "threshold"
            ],
        )
        for video_index in range(
            len(subject)
        )
    ]

    result_all = [
        np.asarray(
            v["spot_response"],
            dtype=float,
        )
        for v in subject
    ]

    metric_final = (
        tu.MeanAveragePrecision2d(
            num_classes=1
        )
    )

    total_gt = 0

    with patched_candidate_source(
        selected_per_video
    ):

        (
            preds,
            _,
            total_gt,
            metric_video,
            metric_final,
        ) = tu.spotting(
            final_samples,
            subject_index,
            result_all,
            total_gt,
            metric_final,
            int(
                cache_config[
                    "k_p"
                ]
            ),
            int(
                cache_config[
                    "interval_strategy"
                ]
            ),
            DATASET,
        )

    raw_counts = (
        official_metric_counts(
            metric_final,
            total_gt,
        )
    )

    if not need_recognition:

        return {
            "raw_counts":
                raw_counts,

            "preds":
                preds,

            "selected_peaks":
                selected_per_video,

            "metric_video":
                metric_video,
        }

    result1_all = [
        np.asarray(
            v[
                "recognition_sequence"
            ]
        )
        for v in subject
    ]

    with contextlib.redirect_stdout(
        io.StringIO()
    ):

        (
            pred_list,
            _,
            gt_tp_list,
            _,
            _,
        ) = tu.recognition(
            result1_all,
            preds,
            metric_video,
            final_emotions,
            subject_index,
            [],
            [],
            final_samples,
            [],
            [],
            int(
                cache_config[
                    "frame_skip"
                ]
            ),
            int(
                cache_config[
                    "penalty_strategy"
                ]
            ),
        )

    neutral_label = (
        int(
            cache_config[
                "emotion_type"
            ]
        )
        - 1
    )

    tp_neutral = 0
    fp_neutral = 0

    for prediction, target in zip(
        pred_list,
        gt_tp_list,
    ):

        if (
            int(prediction)
            == neutral_label
        ):

            if int(target) == -1:
                fp_neutral += 1
            else:
                tp_neutral += 1

    full_counts = (
        raw_counts[0]
        - tp_neutral,

        raw_counts[1]
        - fp_neutral,

        raw_counts[2]
        + tp_neutral,
    )

    return {
        "raw_counts":
            raw_counts,

        "full_counts":
            full_counts,

        "preds":
            preds,

        "selected_peaks":
            selected_per_video,

        "metric_video":
            metric_video,

        "recognition_pred":
            [
                int(x)
                for x in pred_list
            ],

        "recognition_gt":
            [
                int(x)
                for x in gt_tp_list
            ],
    }


# ============================================================
# Prediction-level audit extraction
# ============================================================

def subject_prediction_evidence(
    subject_index,
    subject_name,
    config_item,
    decoded,
    grouped,
    final_samples,
    cache_config,
):

    preds = decoded["preds"]

    metric_video = (
        decoded[
            "metric_video"
        ]
    )

    value = metric_video.value(
        iou_thresholds=0.5
    )[0.5][0]

    match_map = (
        value[
            "pred_match_gt"
        ]
    )

    recognition_pred = (
        decoded[
            "recognition_pred"
        ]
    )

    recognition_gt = (
        decoded[
            "recognition_gt"
        ]
    )

    neutral_label = (
        int(
            cache_config[
                "emotion_type"
            ]
        )
        - 1
    )

    prediction_rows = []
    video_rows = []
    video_prediction_records = []

    flat_recognition_index = 0

    subject_raw_tp = 0
    subject_raw_fp = 0
    subject_raw_fn = 0

    subject_full_tp = 0
    subject_full_fp = 0
    subject_full_fn = 0

    subject_videos = (
        grouped[
            subject_index
        ]
    )

    for video_index, video in enumerate(
        subject_videos
    ):

        video_preds = preds[
            video_index
        ]

        gt_list = (
            final_samples[
                subject_index
            ][video_index]
        )

        matches = list(
            match_map.get(
                video_index,
                [],
            )
        )

        if len(video_preds) == 0:
            matches = []

        if (
            len(matches)
            != len(video_preds)
        ):
            raise RuntimeError(
                "pred_match_gt length mismatch "
                f"subject={subject_index}, "
                f"video={video_index}: "
                f"{len(matches)} vs "
                f"{len(video_preds)}"
            )

        raw_tp = 0
        raw_fp = 0

        neutral_tp = 0
        neutral_fp = 0

        this_video_records = []

        for pred_index, pred in enumerate(
            video_preds
        ):

            sample_index = int(
                matches[
                    pred_index
                ]
            )

            is_tp = (
                sample_index
                != -1
            )

            if is_tp:
                raw_tp += 1
            else:
                raw_fp += 1

            if (
                flat_recognition_index
                >= len(
                    recognition_pred
                )
            ):
                raise RuntimeError(
                    "recognition index overflow"
                )

            recog_pred = int(
                recognition_pred[
                    flat_recognition_index
                ]
            )

            recog_gt = int(
                recognition_gt[
                    flat_recognition_index
                ]
            )

            flat_recognition_index += 1

            is_neutral = (
                recog_pred
                == neutral_label
            )

            if is_neutral:

                if is_tp:
                    neutral_tp += 1
                    synergy_action = (
                        "TP_TO_FN"
                    )

                else:
                    neutral_fp += 1
                    synergy_action = (
                        "REMOVE_FP"
                    )

            else:
                synergy_action = "KEEP"

            gt_index = (
                sample_index
                if is_tp
                else None
            )

            matched_gt = (
                gt_list[
                    sample_index
                ]
                if is_tp
                else None
            )

            all_ious = [
                interval_iou_inclusive(
                    pred,
                    gt,
                )
                for gt in gt_list
            ]

            best_iou = (
                max(all_ious)
                if all_ious
                else 0.0
            )

            matched_iou = (
                interval_iou_inclusive(
                    pred,
                    matched_gt,
                )
                if matched_gt
                is not None
                else 0.0
            )

            row = {
                "subject_index":
                    subject_index,

                "subject":
                    subject_name,

                "video_index_within_subject":
                    video_index,

                "video_index_global":
                    int(
                        video[
                            "video_index_global"
                        ]
                    ),

                "video":
                    str(
                        video["video"]
                    ),

                "config_id":
                    int(
                        config_item[
                            "config_id"
                        ]
                    ),

                "prediction_index":
                    pred_index,

                "onset":
                    int(pred[0]),

                "offset":
                    int(pred[2]),

                "peak":
                    int(pred[-1]),

                "matched_gt_index":
                    (
                        sample_index
                        if is_tp
                        else -1
                    ),

                "matched_gt":
                    (
                        json.dumps(
                            normalize(
                                matched_gt
                            )
                        )
                        if matched_gt
                        is not None
                        else ""
                    ),

                "best_iou_inclusive":
                    float(best_iou),

                "matched_iou_inclusive":
                    float(matched_iou),

                "raw_status":
                    (
                        "TP"
                        if is_tp
                        else "FP"
                    ),

                "recognition_prediction":
                    recog_pred,

                "recognition_target":
                    recog_gt,

                "neutral":
                    bool(
                        is_neutral
                    ),

                "synergy_action":
                    synergy_action,
            }

            prediction_rows.append(
                row
            )

            this_video_records.append(
                row
            )

        raw_fn = (
            len(gt_list)
            - raw_tp
        )

        full_tp = (
            raw_tp
            - neutral_tp
        )

        full_fp = (
            raw_fp
            - neutral_fp
        )

        full_fn = (
            raw_fn
            + neutral_tp
        )

        video_row = {
            "subject_index":
                subject_index,

            "subject":
                subject_name,

            "video_index_within_subject":
                video_index,

            "video_index_global":
                int(
                    video[
                        "video_index_global"
                    ]
                ),

            "video":
                str(
                    video["video"]
                ),

            "config_id":
                int(
                    config_item[
                        "config_id"
                    ]
                ),

            "gt_count":
                len(gt_list),

            "prediction_count":
                len(video_preds),

            "raw_tp":
                raw_tp,

            "raw_fp":
                raw_fp,

            "raw_fn":
                raw_fn,

            "neutral_tp":
                neutral_tp,

            "neutral_fp":
                neutral_fp,

            "full_tp":
                full_tp,

            "full_fp":
                full_fp,

            "full_fn":
                full_fn,
        }

        video_rows.append(
            video_row
        )

        video_prediction_records.append({
            "video_index_global":
                int(
                    video[
                        "video_index_global"
                    ]
                ),

            "subject_index":
                subject_index,

            "subject":
                subject_name,

            "video":
                str(
                    video["video"]
                ),

            "config":
                normalize(
                    config_item
                ),

            "selected_reference_peaks":
                normalize(
                    decoded[
                        "selected_peaks"
                    ][video_index]
                ),

            "decoded_predictions":
                normalize(
                    video_preds
                ),

            "prediction_evidence":
                this_video_records,
        })

        subject_raw_tp += raw_tp
        subject_raw_fp += raw_fp
        subject_raw_fn += raw_fn

        subject_full_tp += full_tp
        subject_full_fp += full_fp
        subject_full_fn += full_fn

    assert (
        flat_recognition_index
        == len(
            recognition_pred
        )
    )

    evidence_raw = (
        subject_raw_tp,
        subject_raw_fp,
        subject_raw_fn,
    )

    evidence_full = (
        subject_full_tp,
        subject_full_fp,
        subject_full_fn,
    )

    assert (
        evidence_raw
        == tuple(
            decoded[
                "raw_counts"
            ]
        )
    ), (
        evidence_raw,
        decoded[
            "raw_counts"
        ],
    )

    assert (
        evidence_full
        == tuple(
            decoded[
                "full_counts"
            ]
        )
    ), (
        evidence_full,
        decoded[
            "full_counts"
        ],
    )

    return (
        prediction_rows,
        video_rows,
        video_prediction_records,
    )


# ============================================================
# Main
# ============================================================

def main():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--cache",
        required=True,
    )

    parser.add_argument(
        "--output_dir",
        required=True,
    )

    args = parser.parse_args()

    cache_path = Path(
        args.cache
    ).resolve()

    out = Path(
        args.output_dir
    ).resolve()

    out.mkdir(
        parents=True,
        exist_ok=True,
    )

    started = utc_now()

    # --------------------------------------------------------
    # Identity / manifest
    # --------------------------------------------------------

    script_path = Path(
        __file__
    ).resolve()

    repo_root = (
        script_path.parent
    )

    git_commit = subprocess.check_output(
        [
            "git",
            "-C",
            str(repo_root),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()

    if git_commit != OFFICIAL_COMMIT:

        raise RuntimeError(
            "Official repo commit mismatch: "
            f"{git_commit}"
        )

    cache, videos = (
        load_cache(
            cache_path
        )
    )

    grouped = group_videos(
        videos
    )

    config_cache = (
        cache["config"]
    )

    k = int(
        config_cache[
            "k_p"
        ]
    )

    if k < 1:
        raise RuntimeError(
            "invalid k_p"
        )

    subject_names = [
        str(
            subject[0][
                "subject"
            ]
        )
        for subject in grouped
    ]

    gt_count = sum(
        len(
            v[
                "gt_intervals"
            ]
        )
        for v in videos
    )

    assert gt_count == EXPECTED_GT

    grid = build_grid()

    assert len(grid) == 90

    manifest = {
        "dataset":
            DATASET,

        "protocol":
            "official-response full-pipeline GLSD-90",

        "cache_absolute_path":
            str(cache_path),

        "cache_sha256":
            sha256_file(
                cache_path
            ),

        "script_absolute_path":
            str(script_path),

        "script_sha256":
            sha256_file(
                script_path
            ),

        "official_boostingvrme_git_commit":
            git_commit,

        "python_version":
            sys.version,

        "python_executable":
            sys.executable,

        "platform":
            platform.platform(),

        "numpy_version":
            np.__version__,

        "scipy_version":
            scipy.__version__,

        "sklearn_version":
            sklearn.__version__,

        "command_line":
            sys.argv,

        "subject_count":
            EXPECTED_SUBJECTS,

        "video_count":
            EXPECTED_VIDEOS,

        "gt_event_count":
            EXPECTED_GT,

        "important_protocol_note":
            (
                "This official-response "
                "reconstruction contains "
                "853 GT events. It is not "
                "silently relabeled as the "
                "858-event controlled cache."
            ),

        "official_cache_k_p":
            k,

        "base_scale_source":
            (
                "fixed official response-cache "
                "k_p for apples-to-apples "
                "full-pipeline comparison"
            ),

        "glsd_grid_size":
            90,

        "glsd_grid_constants": {
            "reference_scales":
                REFERENCE_SCALES,

            "local_radii":
                LOCAL_RADII,

            "thresholds":
                THRESHOLDS,
        },

        "selection_tie_break": [
            "higher_F1",
            "higher_precision",
            "fewer_FP",
            "fixed_grid_order",
        ],

        "native_expected_counts": {
            "raw":
                list(
                    EXPECTED_NATIVE_RAW
                ),

            "full":
                list(
                    EXPECTED_NATIVE_FULL
                ),
        },

        "published_context_f1":
            0.0997,

        "execution_started_at_utc":
            started,

        "run_stdout_stderr_path":
            str(
                out
                / "run_stdout_stderr.log"
            ),

        "run_stdout_stderr_sha256":
            None,

        "run_stdout_stderr_status":
            "PENDING_FINALIZE_LOG",
    }

    # --------------------------------------------------------
    # HARD GATE: Native exact replay
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("NATIVE EXACT REPLAY HARD GATE")
    print("=" * 70)

    native = native_exact_replay(
        cache,
        grouped,
    )

    print(
        "Native Raw :",
        native[
            "raw_counts"
        ],
    )

    print(
        "Native Full:",
        native[
            "full_counts"
        ],
    )

    print(
        "Spot predictions exact =",
        native[
            "prediction_exact"
        ],
    )

    print(
        "Recognition exact =",
        native[
            "recognition_exact"
        ],
    )

    print(
        "Matched-GT exact =",
        native[
            "matched_gt_exact"
        ],
    )

    if not native["passed"]:

        print(
            "NATIVE_EXACT_REPLAY = FAIL"
        )

        raise RuntimeError(
            "Native exact replay "
            "hard gate failed"
        )

    print(
        "NATIVE_EXACT_REPLAY = PASS"
    )

    # --------------------------------------------------------
    # Build fixed subject structures
    # --------------------------------------------------------

    final_samples = [
        [
            v[
                "gt_intervals"
            ]
            for v in subject
        ]
        for subject in grouped
    ]

    final_emotions = [
        [
            v[
                "emotion_labels"
            ]
            for v in subject
        ]
        for subject in grouped
    ]

    # --------------------------------------------------------
    # Precompute GLSD evidence objects
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("BUILD GLSD FEATURE BANK")
    print("=" * 70)

    feature_bank = []

    for subject_index, subject in enumerate(
        grouped
    ):

        feature_bank.append(
            [
                GLSDFeatures(
                    v[
                        "spot_response"
                    ],
                    k,
                )
                for v in subject
            ]
        )

        if (
            (subject_index + 1)
            % 10 == 0
            or
            subject_index + 1
            == EXPECTED_SUBJECTS
        ):

            print(
                "feature subjects:",
                subject_index + 1,
                "/",
                EXPECTED_SUBJECTS,
            )

    # --------------------------------------------------------
    # 90 × 94 subject/config raw-count table
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("PRECOMPUTE SUBJECT × CONFIG RAW COUNTS")
    print("=" * 70)

    subject_config_counts = {
        int(c["config_id"]):
            [None]
            * EXPECTED_SUBJECTS

        for c in grid
    }

    for config_item in grid:

        cid = int(
            config_item[
                "config_id"
            ]
        )

        for subject_index in range(
            EXPECTED_SUBJECTS
        ):

            decoded = decode_subject(
                subject_index,
                config_item,
                grouped,
                feature_bank,
                final_samples,
                final_emotions,
                config_cache,
                need_recognition=False,
            )

            subject_config_counts[
                cid
            ][subject_index] = tuple(
                int(x)
                for x in decoded[
                    "raw_counts"
                ]
            )

        print(
            f"config {cid + 1:02d}/90 complete"
        )

    # --------------------------------------------------------
    # Full inner-fold evidence table:
    # 94 × 90 × 93 = 786,780
    # --------------------------------------------------------

    inner_path = (
        out
        / "glsd_search_inner_fold_counts.csv"
    )

    inner_fields = [
        "outer_subject_index",
        "outer_subject",
        "config_id",
        "reference_scale",
        "local_radius",
        "threshold",
        "inner_subject_index",
        "inner_subject",
        "tp",
        "fp",
        "fn",
    ]

    with inner_path.open(
        "w",
        newline="",
        encoding="utf-8",
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=inner_fields,
        )

        writer.writeheader()

        for outer in range(
            EXPECTED_SUBJECTS
        ):

            for config_item in grid:

                cid = int(
                    config_item[
                        "config_id"
                    ]
                )

                for inner in range(
                    EXPECTED_SUBJECTS
                ):

                    if inner == outer:
                        continue

                    assert (
                        inner
                        != outer
                    )

                    tp, fp, fn = (
                        subject_config_counts[
                            cid
                        ][inner]
                    )

                    writer.writerow({
                        "outer_subject_index":
                            outer,

                        "outer_subject":
                            subject_names[
                                outer
                            ],

                        "config_id":
                            cid,

                        "reference_scale":
                            config_item[
                                "reference_scale"
                            ],

                        "local_radius":
                            config_item[
                                "local_radius"
                            ],

                        "threshold":
                            config_item[
                                "threshold"
                            ],

                        "inner_subject_index":
                            inner,

                        "inner_subject":
                            subject_names[
                                inner
                            ],

                        "tp":
                            tp,

                        "fp":
                            fp,

                        "fn":
                            fn,
                    })

    # --------------------------------------------------------
    # Outer selection
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("OUTER LOSO CONFIGURATION SELECTION")
    print("=" * 70)

    search_rows = []
    selected_rows = []
    selection_trace = []

    selected_by_outer = {}

    for outer in range(
        EXPECTED_SUBJECTS
    ):

        inner_subjects = [
            i
            for i in range(
                EXPECTED_SUBJECTS
            )
            if i != outer
        ]

        assert len(
            inner_subjects
        ) == 93

        assert (
            outer
            not in inner_subjects
        )

        candidates = []

        for config_item in grid:

            cid = int(
                config_item[
                    "config_id"
                ]
            )

            pooled = add_counts(
                [
                    subject_config_counts[
                        cid
                    ][inner]
                    for inner
                    in inner_subjects
                ]
            )

            m = metrics(
                pooled
            )

            candidates.append({
                **config_item,

                "pooled_inner_tp":
                    m["tp"],

                "pooled_inner_fp":
                    m["fp"],

                "pooled_inner_fn":
                    m["fn"],

                "precision":
                    m[
                        "precision"
                    ],

                "recall":
                    m[
                        "recall"
                    ],

                "f1":
                    m["f1"],
            })

        ranked = sorted(
            candidates,
            key=lambda row: (
                -float(
                    row["f1"]
                ),

                -float(
                    row[
                        "precision"
                    ]
                ),

                int(
                    row[
                        "pooled_inner_fp"
                    ]
                ),

                int(
                    row[
                        "grid_order"
                    ]
                ),
            ),
        )

        for rank, row in enumerate(
            ranked,
            start=1,
        ):
            row["rank"] = rank

        winner = ranked[0]

        selected_by_outer[
            outer
        ] = dict(
            winner
        )

        rank_lookup = {
            int(row["config_id"]):
                int(row["rank"])
            for row in ranked
        }

        for row in candidates:

            cid = int(
                row[
                    "config_id"
                ]
            )

            search_rows.append({
                "outer_subject_index":
                    outer,

                "outer_subject":
                    subject_names[
                        outer
                    ],

                **row,

                "final_rank":
                    rank_lookup[cid],

                "selected":
                    int(
                        cid
                        == int(
                            winner[
                                "config_id"
                            ]
                        )
                    ),
            })

        selected_rows.append({
            "outer_subject_index":
                outer,

            "outer_subject":
                subject_names[
                    outer
                ],

            "inner_subject_count":
                len(
                    inner_subjects
                ),

            "inner_subject_indices":
                json.dumps(
                    inner_subjects
                ),

            "inner_subjects":
                json.dumps(
                    [
                        subject_names[i]
                        for i
                        in inner_subjects
                    ]
                ),

            **winner,
        })

        selection_trace.append({
            "outer_subject_index":
                outer,

            "outer_subject":
                subject_names[
                    outer
                ],

            "inner_subject_indices":
                inner_subjects,

            "inner_subjects":
                [
                    subject_names[i]
                    for i
                    in inner_subjects
                ],

            "tie_break": [
                "higher_F1",
                "higher_precision",
                "fewer_FP",
                "fixed_grid_order",
            ],

            "winner_config_id":
                int(
                    winner[
                        "config_id"
                    ]
                ),

            "winner_rank":
                int(
                    winner["rank"]
                ),

            "candidates":
                ranked,
        })

        print(
            "outer",
            f"{outer + 1:02d}/94",
            "subject",
            subject_names[
                outer
            ],
            "winner config",
            winner[
                "config_id"
            ],
            "F1",
            f'{winner["f1"]:.6f}',
        )

    # --------------------------------------------------------
    # Persist selection evidence BEFORE outer test decoding
    # --------------------------------------------------------

    search_fields = [
        "outer_subject_index",
        "outer_subject",
        "config_id",
        "reference_scale",
        "local_radius",
        "threshold",
        "grid_order",
        "pooled_inner_tp",
        "pooled_inner_fp",
        "pooled_inner_fn",
        "precision",
        "recall",
        "f1",
        "rank",
        "final_rank",
        "selected",
    ]

    write_csv(
        out
        / "glsd_search_all_90x94.csv",
        search_rows,
        search_fields,
    )

    selected_fields = [
        "outer_subject_index",
        "outer_subject",
        "inner_subject_count",
        "inner_subject_indices",
        "inner_subjects",
        "config_id",
        "reference_scale",
        "local_radius",
        "threshold",
        "grid_order",
        "pooled_inner_tp",
        "pooled_inner_fp",
        "pooled_inner_fn",
        "precision",
        "recall",
        "f1",
        "rank",
    ]

    write_csv(
        out
        / "outer_selected_configs.csv",
        selected_rows,
        selected_fields,
    )

    (
        out
        / "outer_selected_configs.json"
    ).write_text(
        json.dumps(
            normalize(
                selected_rows
            ),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    (
        out
        / "outer_selection_trace.json"
    ).write_text(
        json.dumps(
            normalize(
                selection_trace
            ),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    # --------------------------------------------------------
    # Decode the 94 held-out outer subjects
    # --------------------------------------------------------

    print()
    print("=" * 70)
    print("FINAL OUTER-TEST GLSD DECODING")
    print("=" * 70)

    per_subject_rows = []
    per_video_rows = []
    per_prediction_rows = []
    final_video_records = []

    outer_raw_counts = []
    outer_full_counts = []

    for outer in range(
        EXPECTED_SUBJECTS
    ):

        config_item = (
            selected_by_outer[
                outer
            ]
        )

        decoded = decode_subject(
            outer,
            config_item,
            grouped,
            feature_bank,
            final_samples,
            final_emotions,
            config_cache,
            need_recognition=True,
        )

        raw = tuple(
            int(x)
            for x in decoded[
                "raw_counts"
            ]
        )

        full = tuple(
            int(x)
            for x in decoded[
                "full_counts"
            ]
        )

        (
            prediction_rows,
            video_rows,
            video_records,
        ) = subject_prediction_evidence(
            outer,
            subject_names[
                outer
            ],
            config_item,
            decoded,
            grouped,
            final_samples,
            config_cache,
        )

        # independent sum from video evidence
        video_raw = (
            sum(
                x["raw_tp"]
                for x in video_rows
            ),

            sum(
                x["raw_fp"]
                for x in video_rows
            ),

            sum(
                x["raw_fn"]
                for x in video_rows
            ),
        )

        video_full = (
            sum(
                x["full_tp"]
                for x in video_rows
            ),

            sum(
                x["full_fp"]
                for x in video_rows
            ),

            sum(
                x["full_fn"]
                for x in video_rows
            ),
        )

        assert video_raw == raw
        assert video_full == full

        mr = metrics(raw)
        mf = metrics(full)

        per_subject_rows.append({
            "outer_subject_index":
                outer,

            "outer_subject":
                subject_names[
                    outer
                ],

            "config_id":
                int(
                    config_item[
                        "config_id"
                    ]
                ),

            "reference_scale":
                config_item[
                    "reference_scale"
                ],

            "local_radius":
                config_item[
                    "local_radius"
                ],

            "threshold":
                config_item[
                    "threshold"
                ],

            "raw_tp":
                mr["tp"],

            "raw_fp":
                mr["fp"],

            "raw_fn":
                mr["fn"],

            "raw_precision":
                mr[
                    "precision"
                ],

            "raw_recall":
                mr["recall"],

            "raw_f1":
                mr["f1"],

            "full_tp":
                mf["tp"],

            "full_fp":
                mf["fp"],

            "full_fn":
                mf["fn"],

            "full_precision":
                mf[
                    "precision"
                ],

            "full_recall":
                mf["recall"],

            "full_f1":
                mf["f1"],
        })

        per_video_rows.extend(
            video_rows
        )

        per_prediction_rows.extend(
            prediction_rows
        )

        final_video_records.extend(
            video_records
        )

        outer_raw_counts.append(
            raw
        )

        outer_full_counts.append(
            full
        )

        print(
            "outer",
            f"{outer + 1:02d}/94",
            subject_names[
                outer
            ],
            "raw",
            raw,
            "full",
            full,
        )

    final_raw = add_counts(
        outer_raw_counts
    )

    final_full = add_counts(
        outer_full_counts
    )

    raw_metrics = metrics(
        final_raw
    )

    full_metrics = metrics(
        final_full
    )

    # --------------------------------------------------------
    # Persist final evidence
    # --------------------------------------------------------

    subject_fields = [
        "outer_subject_index",
        "outer_subject",
        "config_id",
        "reference_scale",
        "local_radius",
        "threshold",
        "raw_tp",
        "raw_fp",
        "raw_fn",
        "raw_precision",
        "raw_recall",
        "raw_f1",
        "full_tp",
        "full_fp",
        "full_fn",
        "full_precision",
        "full_recall",
        "full_f1",
    ]

    write_csv(
        out
        / "per_subject_counts.csv",
        per_subject_rows,
        subject_fields,
    )

    video_fields = [
        "subject_index",
        "subject",
        "video_index_within_subject",
        "video_index_global",
        "video",
        "config_id",
        "gt_count",
        "prediction_count",
        "raw_tp",
        "raw_fp",
        "raw_fn",
        "neutral_tp",
        "neutral_fp",
        "full_tp",
        "full_fp",
        "full_fn",
    ]

    write_csv(
        out
        / "per_video_evaluation.csv",
        per_video_rows,
        video_fields,
    )

    prediction_fields = [
        "subject_index",
        "subject",
        "video_index_within_subject",
        "video_index_global",
        "video",
        "config_id",
        "prediction_index",
        "onset",
        "offset",
        "peak",
        "matched_gt_index",
        "matched_gt",
        "best_iou_inclusive",
        "matched_iou_inclusive",
        "raw_status",
        "recognition_prediction",
        "recognition_target",
        "neutral",
        "synergy_action",
    ]

    write_csv(
        out
        / "per_prediction_evaluation.csv",
        per_prediction_rows,
        prediction_fields,
    )

    with (
        out
        / "glsd_final_predictions.pkl"
    ).open(
        "wb"
    ) as f:

        pickle.dump(
            final_video_records,
            f,
            protocol=4,
        )

    auditable_payload = {
        "dataset":
            DATASET,

        "subject_count":
            EXPECTED_SUBJECTS,

        "video_count":
            EXPECTED_VIDEOS,

        "gt_event_count":
            EXPECTED_GT,

        "grid":
            grid,

        "selected_configs":
            selected_rows,

        "per_subject_counts":
            per_subject_rows,

        "per_video_evaluation":
            per_video_rows,

        "per_prediction_evaluation":
            per_prediction_rows,

        "final_raw_counts":
            final_raw,

        "final_full_counts":
            final_full,

        "videos":
            final_video_records,
    }

    with (
        out
        / "glsd_final_predictions_auditable.pkl"
    ).open(
        "wb"
    ) as f:

        pickle.dump(
            auditable_payload,
            f,
            protocol=4,
        )

    # --------------------------------------------------------
    # Manifest final scientific result
    # --------------------------------------------------------

    native_raw_m = metrics(
        EXPECTED_NATIVE_RAW
    )

    native_full_m = metrics(
        EXPECTED_NATIVE_FULL
    )

    delta_raw = (
        raw_metrics["f1"]
        - native_raw_m["f1"]
    )

    delta_full = (
        full_metrics["f1"]
        - native_full_m["f1"]
    )

    manifest.update({
        "execution_finished_at_utc":
            utc_now(),

        "native_exact_replay":
            normalize(
                native
            ),

        "glsd_final_raw_counts":
            list(
                final_raw
            ),

        "glsd_final_raw_metrics":
            raw_metrics,

        "glsd_final_full_counts":
            list(
                final_full
            ),

        "glsd_final_full_metrics":
            full_metrics,

        "delta_raw_vs_native":
            delta_raw,

        "delta_full_vs_native":
            delta_full,

        "glsd_full_minus_published_0_0997":
            (
                full_metrics["f1"]
                - 0.0997
            ),
    })

    manifest_path = (
        out
        / "run_manifest.json"
    )

    manifest_path.write_text(
        json.dumps(
            normalize(
                manifest
            ),
            indent=2,
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )

    # --------------------------------------------------------
    # Main audit Markdown
    # --------------------------------------------------------

    audit = f"""# BoostingVRME / CAS(ME)3 GLSD Official-Response Full-Pipeline Audit

## Protocol identity

- Official repository commit: `{git_commit}`
- Cache SHA-256: `{manifest["cache_sha256"]}`
- Script SHA-256: `{manifest["script_sha256"]}`
- Subjects: {EXPECTED_SUBJECTS}
- Videos: {EXPECTED_VIDEOS}
- GT events in this official-response reconstruction: {EXPECTED_GT}
- GLSD configurations: 90
- Outer folds: 94
- Inner validation subjects per outer fold: 93

## Important protocol note

This run contains 853 GT events. It is not relabeled as the separate
858-event controlled CAS(ME)3 cache used by earlier decoder experiments.

## Native exact replay hard gate

- Raw: `{EXPECTED_NATIVE_RAW}`, F1 = `{native_raw_m["f1"]:.10f}`
- Full: `{EXPECTED_NATIVE_FULL}`, F1 = `{native_full_m["f1"]:.10f}`
- Spot predictions exact: `{native["prediction_exact"]}`
- Recognition exact: `{native["recognition_exact"]}`
- Matched-GT exact: `{native["matched_gt_exact"]}`
- Native exact replay: `PASS`

## GLSD-90

- Reference scales: `{REFERENCE_SCALES}`
- Local radii: `{LOCAL_RADII}`
- Thresholds: `{THRESHOLDS}`
- Selection: pooled inner Raw Spotting F1
- Tie-break: F1 -> precision -> fewer FP -> fixed grid order
- Outer subject excluded from its own inner pool: enforced by assertion

## GLSD final outer-test result

- Raw TP/FP/FN: `{final_raw}`
- Raw Precision: `{raw_metrics["precision"]:.10f}`
- Raw Recall: `{raw_metrics["recall"]:.10f}`
- Raw F1: `{raw_metrics["f1"]:.10f}`

- Full TP/FP/FN: `{final_full}`
- Full Precision: `{full_metrics["precision"]:.10f}`
- Full Recall: `{full_metrics["recall"]:.10f}`
- Full F1: `{full_metrics["f1"]:.10f}`

- Delta Raw vs Native: `{delta_raw:+.10f}`
- Delta Full vs Native: `{delta_full:+.10f}`
- GLSD Full minus published-context 0.0997: `{full_metrics["f1"] - 0.0997:+.10f}`

## Persisted evidence

- `glsd_search_all_90x94.csv`
- `glsd_search_inner_fold_counts.csv`
- `outer_selected_configs.csv`
- `outer_selected_configs.json`
- `outer_selection_trace.json`
- `per_subject_counts.csv`
- `per_video_evaluation.csv`
- `per_prediction_evaluation.csv`
- `glsd_final_predictions.pkl`
- `glsd_final_predictions_auditable.pkl`
- `run_manifest.json`
- `run_stdout_stderr.log`

## Runtime verdict

`BOOSTING_CASME3_GLSD_FULL_VALIDATED = PASS`

Note: stdout/stderr SHA-256 is finalized after process exit.
"""

    (
        out
        / "BOOSTING_CASME3_GLSD_FULL_AUDIT.md"
    ).write_text(
        audit,
        encoding="utf-8",
    )

    print()
    print("=" * 70)
    print("CASME3 GLSD OFFICIAL-RESPONSE")
    print("=" * 70)

    print(
        "Native Raw :",
        EXPECTED_NATIVE_RAW,
        "F1 =",
        f'{native_raw_m["f1"]:.10f}',
    )

    print(
        "GLSD Raw   :",
        final_raw,
        "F1 =",
        f'{raw_metrics["f1"]:.10f}',
    )

    print(
        "Native Full:",
        EXPECTED_NATIVE_FULL,
        "F1 =",
        f'{native_full_m["f1"]:.10f}',
    )

    print(
        "GLSD Full  :",
        final_full,
        "F1 =",
        f'{full_metrics["f1"]:.10f}',
    )

    print(
        "Delta Raw  =",
        f"{delta_raw:+.10f}",
    )

    print(
        "Delta Full =",
        f"{delta_full:+.10f}",
    )

    print(
        "GLSD Full - published 0.0997 =",
        f"{full_metrics['f1'] - 0.0997:+.10f}",
    )

    print()
    print(
        "BOOSTING_CASME3_GLSD_FULL_VALIDATED = PASS"
    )


if __name__ == "__main__":
    main()


In [ ]:
!mkdir -p /content/transfer_bundle


In [ ]:
%%writefile /content/transfer_bundle/transfer_common.py
#!/usr/bin/env python3
"""Shared, dependency-light protocol utilities for GLSD configuration transfer."""

from __future__ import annotations

import csv
import json
from pathlib import Path

import numpy as np


SEED = 100
BOOTSTRAP_ITERATIONS = 10_000
THETA_FIELDS = ("reference_scale", "local_radius", "threshold")
FORBIDDEN_SOURCE_FIELDS = {
    "k", "k_p", "target_k", "source_k", "event_interval", "interval_strategy",
    "frame_skip", "boundary", "boundary_rule", "candidate_order", "conflict_rule",
}


DATASETS = {
    "sammlv": {"subjects": 29, "videos": 79, "gt": 159},
    "casme3": {"subjects": 94, "videos": 462, "gt": 853},
}


LOCKED = {
    "metst_sammlv": {
        "pipeline": "ME-TST+", "dataset": "sammlv",
        "native": (52, 171, 107), "tuned": (47, 121, 112),
    },
    "metst_casme3": {
        "pipeline": "ME-TST+", "dataset": "casme3",
        "native": (76, 736, 777), "tuned": (90, 898, 763),
    },
    "boosting_sammlv": {
        "pipeline": "BoostingVRME", "dataset": "sammlv",
        "native": (51, 141, 108), "tuned": (44, 82, 115),
    },
    "boosting_casme3": {
        "pipeline": "BoostingVRME", "dataset": "casme3",
        "native": (84, 786, 769), "tuned": (120, 1110, 733),
    },
}


DIRECTIONS = {
    "metst_to_boosting_sammlv": {
        "dataset": "sammlv", "source": "metst_sammlv", "target": "boosting_sammlv",
        "relative_output": "sammlv/metst_to_boosting",
    },
    "boosting_to_metst_sammlv": {
        "dataset": "sammlv", "source": "boosting_sammlv", "target": "metst_sammlv",
        "relative_output": "sammlv/boosting_to_metst",
    },
    "metst_to_boosting_casme3": {
        "dataset": "casme3", "source": "metst_casme3", "target": "boosting_casme3",
        "relative_output": "casme3/metst_to_boosting",
    },
    "boosting_to_metst_casme3": {
        "dataset": "casme3", "source": "boosting_casme3", "target": "metst_casme3",
        "relative_output": "casme3/boosting_to_metst",
    },
}


def f1(counts) -> float:
    tp, fp, fn = (int(value) for value in counts)
    denominator = 2 * tp + fp + fn
    return 2 * tp / denominator if denominator else 0.0


def metrics(counts) -> dict:
    tp, fp, fn = (int(value) for value in counts)
    return {"TP": tp, "FP": fp, "FN": fn, "F1": f1((tp, fp, fn))}


def write_json(path: Path, value) -> None:
    path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        raise RuntimeError(f"refusing to write empty CSV: {path}")
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def load_csv(path: Path) -> list[dict]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        raise RuntimeError(f"empty CSV: {path}")
    return rows


def canonical_subject(dataset: str, value) -> str:
    text = str(value).strip()
    if not text:
        raise RuntimeError("empty subject ID")
    try:
        number = int(text)
    except ValueError as exc:
        raise RuntimeError(f"non-numeric {dataset} subject ID: {text!r}") from exc
    return f"{number:03d}" if dataset == "sammlv" else str(number)


def field_name(row: dict, candidates: tuple[str, ...], label: str) -> str:
    lowered = {name.lower(): name for name in row}
    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]
    raise RuntimeError(f"cannot resolve {label}; fields={list(row)}")


def selected_config_rows(path: Path, dataset: str) -> dict[str, dict]:
    rows = load_csv(path)
    subject_field = field_name(rows[0], ("outer_subject", "subject", "subject_id"), "subject")
    a0_field = field_name(rows[0], ("reference_scale", "a0"), "a0")
    rho_field = field_name(rows[0], ("local_radius", "rho"), "rho")
    tau_field = field_name(rows[0], ("threshold", "tau"), "tau")
    forbidden = {name.lower() for name in rows[0]} & FORBIDDEN_SOURCE_FIELDS
    if forbidden:
        raise RuntimeError(f"source selected-config table contains forbidden transfer fields: {sorted(forbidden)}")
    result = {}
    for row in rows:
        subject = canonical_subject(dataset, row[subject_field])
        if subject in result:
            raise RuntimeError(f"duplicate selected config for subject {subject}")
        result[subject] = {
            "source_subject_raw": str(row[subject_field]),
            "source_outer_fold": str(row.get("outer_subject_index", row[subject_field])),
            "a0": float(row[a0_field]),
            "rho": float(row[rho_field]),
            "tau": float(row[tau_field]),
            "source_config_origin": str(path),
        }
    return result


def normalize_json(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): normalize_json(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [normalize_json(item) for item in value]
    return value


def pooled(rows: list[dict], prefix: str) -> tuple[int, int, int]:
    return tuple(sum(int(row[f"{prefix}_{name}"]) for row in rows) for name in ("TP", "FP", "FN"))


def paired_bootstrap(
    native: np.ndarray, tuned: np.ndarray, transferred: np.ndarray,
    iterations: int = BOOTSTRAP_ITERATIONS, seed: int = SEED,
) -> dict:
    arrays = [np.asarray(value, dtype=np.int64) for value in (native, tuned, transferred)]
    if len({array.shape for array in arrays}) != 1 or arrays[0].ndim != 2 or arrays[0].shape[1] != 3:
        raise RuntimeError("paired bootstrap expects three aligned N x 3 count arrays")
    n_subjects = arrays[0].shape[0]
    rng = np.random.default_rng(seed)
    delta_native = np.empty(iterations, dtype=float)
    delta_tuned = np.empty(iterations, dtype=float)
    for index in range(iterations):
        draw = rng.integers(0, n_subjects, size=n_subjects)
        sampled = [array[draw].sum(axis=0) for array in arrays]
        delta_native[index] = f1(sampled[2]) - f1(sampled[0])
        delta_tuned[index] = f1(sampled[2]) - f1(sampled[1])
    point_native = f1(arrays[2].sum(axis=0)) - f1(arrays[0].sum(axis=0))
    point_tuned = f1(arrays[2].sum(axis=0)) - f1(arrays[1].sum(axis=0))
    return {
        "iterations": iterations,
        "seed": seed,
        "unit": "subject",
        "aggregation": "pooled TP/FP/FN before F1",
        "paired_resampling": True,
        "vs_native": {
            "point_estimate": point_native,
            "percentile_2_5": float(np.percentile(delta_native, 2.5)),
            "percentile_97_5": float(np.percentile(delta_native, 97.5)),
        },
        "vs_tuned": {
            "point_estimate": point_tuned,
            "percentile_2_5": float(np.percentile(delta_tuned, 2.5)),
            "percentile_97_5": float(np.percentile(delta_tuned, 97.5)),
        },
    }


In [ ]:
%%writefile /content/transfer_bundle/transfer_paths.py
#!/usr/bin/env python3
"""Single source of truth for sealed Colab/Drive paths."""

from pathlib import Path


DRIVE = Path("/content/drive/MyDrive")

EVIDENCE = {
    "metst_sammlv": DRIVE / "GLSD_METST_OFFICIAL/SAMMLV_FINAL_EVIDENCE/results_v1",
    "metst_casme3": DRIVE / "GLSD_METST_OFFICIAL/CASME3_FINAL_EVIDENCE/results_v1",
    "boosting_sammlv": DRIVE / "GLSD_BoostingVRME_Full/results_evidence_v2",
    "boosting_casme3": DRIVE / "GLSD_BoostingVRME_Full/CASME3/glsd_full_evidence_v1",
}

RESPONSES = {
    "metst_sammlv": DRIVE / "ME-TST_OFFICIAL_DUMP/SAMMLV_method1_strategy1",
    "metst_casme3": DRIVE / "ME-TST_OFFICIAL_DUMP/CASME_3_method1_strategy1",
    "boosting_sammlv": DRIVE / "GLSD_BoostingVRME_Full/sammlv_official_full_responses.pkl",
    "boosting_casme3": DRIVE / (
        "GLSD_BoostingVRME_Full/CASME3/official_response_cache/"
        "casme3_official_full_responses.pkl"
    ),
}

RUNNERS = {
    "metst_sammlv": DRIVE / "GLSD_METST_OFFICIAL/SAMMLV_FINAL_EVIDENCE/metst_official_glds_full.py",
    "metst_casme3": DRIVE / (
        "GLSD_METST_OFFICIAL/CASME3_FINAL_EVIDENCE/source/"
        "metst_casme3_official_glds_full.py"
    ),
    "boosting_sammlv": Path("/content/BoostingVRME/boosting_official_glds_full.py"),
    "boosting_casme3": Path("/content/BoostingVRME/boosting_official_glds_full_casme3.py"),
}

CORES = {
    "metst_sammlv": DRIVE / (
        "GLSD_METST_OFFICIAL/SAMMLV_FINAL_EVIDENCE/"
        "boosting_official_glds_full_LOCKED.py"
    ),
    "metst_casme3": DRIVE / (
        "GLSD_METST_OFFICIAL/CASME3_FINAL_EVIDENCE/source/"
        "boosting_official_glds_full_LOCKED.py"
    ),
}


In [ ]:
%%writefile /content/transfer_bundle/transfer_audit.py
#!/usr/bin/env python3
"""Read-only subject/video/configuration audit before any transfer execution."""

from __future__ import annotations

import argparse
import ast
import json
import re
from pathlib import Path

from transfer_common import DATASETS, LOCKED, canonical_subject, field_name, load_csv, selected_config_rows, write_csv, write_json
from transfer_paths import EVIDENCE, RESPONSES, RUNNERS, CORES


def normalize_video(dataset: str, subject: str, raw_value) -> str:
    value = Path(str(raw_value).strip()).name.lower()
    while "." in value and value.rsplit(".", 1)[1] in {"avi", "mp4", "mov", "pkl", "npy", "csv"}:
        value = value.rsplit(".", 1)[0]
    if dataset == "casme3":
        value = re.sub(r"^spno\.?0*\d+[_-]", "", value, flags=re.IGNORECASE)
    else:
        value = re.sub(rf"^0*{int(subject)}[_-]", "", value)
    if not value:
        raise RuntimeError(f"empty canonical video ID from {raw_value!r}")
    return value


def video_inventory(path: Path, dataset: str) -> dict[tuple[str, str], str]:
    rows = load_csv(path)
    subject_field = field_name(rows[0], ("outer_subject", "subject", "subject_id"), "video subject")
    video_field = field_name(
        rows[0], ("video", "video_name", "video_id", "video_path"), "explicit video ID"
    )
    inventory = {}
    for row in rows:
        subject = canonical_subject(dataset, row[subject_field])
        video = normalize_video(dataset, subject, row[video_field])
        key = subject, video
        raw = str(row[video_field])
        if key in inventory and inventory[key] != raw:
            raise RuntimeError(f"ambiguous canonical video key {key}: {inventory[key]!r} vs {raw!r}")
        inventory[key] = raw
    return inventory


def required_inputs() -> list[Path]:
    paths = []
    for setting in LOCKED:
        paths.extend([
            EVIDENCE[setting],
            EVIDENCE[setting] / "outer_selected_configs.csv",
            EVIDENCE[setting] / "per_subject_counts.csv",
            EVIDENCE[setting] / "per_video_evaluation.csv",
            EVIDENCE[setting] / "run_manifest.json",
            RESPONSES[setting],
            RUNNERS[setting],
        ])
    paths.extend(CORES.values())
    return paths


def parse_subject_list(value) -> list:
    text = str(value).strip()
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
        except (ValueError, SyntaxError, json.JSONDecodeError):
            continue
        if isinstance(parsed, list):
            return parsed
    if ";" in text:
        return [item for item in text.split(";") if item]
    raise RuntimeError(f"cannot parse inner_subjects list: {text[:120]!r}")


def validate_selection_provenance(path: Path, dataset: str, all_subjects: set[str]) -> None:
    rows = load_csv(path)
    subject_field = field_name(rows[0], ("outer_subject", "subject", "subject_id"), "outer subject")
    inner_field = field_name(rows[0], ("inner_subjects", "training_subjects"), "inner subjects")
    count_field = field_name(rows[0], ("inner_subject_count", "training_subject_count"), "inner subject count")
    if len(rows) != len(all_subjects):
        raise RuntimeError(f"selection provenance row count mismatch: {path}")
    for row in rows:
        outer = canonical_subject(dataset, row[subject_field])
        inner = {canonical_subject(dataset, value) for value in parse_subject_list(row[inner_field])}
        expected = all_subjects - {outer}
        if outer in inner or inner != expected or int(row[count_field]) != len(expected):
            raise RuntimeError(
                f"TRANSFER_PROTOCOL_FAIL: {path} outer fold {outer} is not exact Train_s"
            )


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-root", type=Path, required=True)
    args = parser.parse_args()
    output = args.output_root.resolve() / "audit"
    output.mkdir(parents=True, exist_ok=True)

    missing = [str(path) for path in required_inputs() if not path.exists()]
    if missing:
        raise RuntimeError("TRANSFER_PROTOCOL_FAIL: missing sealed input(s):\n" + "\n".join(missing))

    subject_rows = []
    video_rows = []
    universe_lines = []
    for dataset in ("sammlv", "casme3"):
        left_name = f"metst_{dataset}"
        right_name = f"boosting_{dataset}"
        left_configs = selected_config_rows(EVIDENCE[left_name] / "outer_selected_configs.csv", dataset)
        right_configs = selected_config_rows(EVIDENCE[right_name] / "outer_selected_configs.csv", dataset)
        expected = DATASETS[dataset]
        if len(left_configs) != expected["subjects"] or len(right_configs) != expected["subjects"]:
            raise RuntimeError(f"TRANSFER_PROTOCOL_FAIL: {dataset} subject cardinality mismatch")
        if set(left_configs) != set(right_configs):
            raise RuntimeError(f"TRANSFER_PROTOCOL_FAIL: {dataset} subject ID sets differ")
        validate_selection_provenance(
            EVIDENCE[left_name] / "outer_selected_configs.csv", dataset, set(left_configs)
        )
        validate_selection_provenance(
            EVIDENCE[right_name] / "outer_selected_configs.csv", dataset, set(right_configs)
        )
        for subject in sorted(left_configs, key=int):
            subject_rows.append({
                "dataset": dataset,
                "canonical_subject": subject,
                "metst_subject": left_configs[subject]["source_subject_raw"],
                "boosting_subject": right_configs[subject]["source_subject_raw"],
                "mapping_status": "EXACT_CANONICAL_MATCH",
            })

        left_videos = video_inventory(EVIDENCE[left_name] / "per_video_evaluation.csv", dataset)
        right_videos = video_inventory(EVIDENCE[right_name] / "per_video_evaluation.csv", dataset)
        if len(left_videos) != expected["videos"] or len(right_videos) != expected["videos"]:
            raise RuntimeError(
                f"TRANSFER_PROTOCOL_FAIL: {dataset} video cardinality mismatch "
                f"ME-TST+={len(left_videos)}, BoostingVRME={len(right_videos)}, expected={expected['videos']}"
            )
        if set(left_videos) != set(right_videos):
            only_left = sorted(set(left_videos) - set(right_videos))[:20]
            only_right = sorted(set(right_videos) - set(left_videos))[:20]
            raise RuntimeError(
                f"TRANSFER_PROTOCOL_FAIL: {dataset} video IDs differ; "
                f"only ME-TST+={only_left}; only BoostingVRME={only_right}"
            )
        for subject, video in sorted(left_videos, key=lambda item: (int(item[0]), item[1])):
            video_rows.append({
                "dataset": dataset, "canonical_subject": subject, "canonical_video": video,
                "metst_video": left_videos[(subject, video)],
                "boosting_video": right_videos[(subject, video)], "mapping_status": "MATCH",
            })
        if dataset == "casme3" and ("15", "j") in left_videos:
            raise RuntimeError("TRANSFER_PROTOCOL_FAIL: forbidden excluded CAS(ME)3 video spNO.15_j is present")
        universe_lines.append(
            f"{dataset}: subjects={expected['subjects']} videos={expected['videos']} "
            f"GT={expected['gt']} mapping=PASS"
        )

    write_csv(output / "subject_mapping.csv", subject_rows)
    write_csv(output / "video_mapping.csv", video_rows)
    (output / "universe_audit.txt").write_text(
        "\n".join(universe_lines + [
            "CAS(ME)3 explicit exclusion: spNO.15_j / skip index 74 preserved.",
            "UNIVERSE_AND_MAPPING_AUDIT = PASS",
        ]) + "\n",
        encoding="utf-8",
    )
    write_json(output / "mapping_gate.json", {
        "status": "PASS", "subject_mapping_complete": True,
        "video_mapping_complete": True, "casme3_spNO_15_j_excluded": True,
        "source_outer_fold_train_sets_exact": True,
        "source_test_subject_excluded_from_selection": True,
        "target_side_tuning_used": False, "transferred_fields": ["a0", "rho", "tau"],
        "forbidden_transferred_fields": sorted({"k", "event_geometry", "boundary_rule", "candidate_order", "conflict_rule"}),
    })
    print("UNIVERSE_AND_MAPPING_AUDIT = PASS")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/transfer_bundle/consolidate_preflight.py
#!/usr/bin/env python3
"""Consolidate four independently executed exact replays into one hard gate."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

from transfer_common import LOCKED, f1, load_csv, write_csv, write_json


def one_generic(root: Path, setting: str) -> dict:
    candidates = list(root.glob("preflight_*/locked_replay_summary.csv"))
    if len(candidates) != 1:
        raise RuntimeError(f"expected exactly one preflight summary below {root}, got {candidates}")
    rows = load_csv(candidates[0])
    if len(rows) != 1:
        raise RuntimeError(f"expected one setting in {candidates[0]}")
    row = rows[0]
    expected = LOCKED[setting]
    native = tuple(int(row[f"Native_{name}"]) for name in ("TP", "FP", "FN"))
    tuned = tuple(int(row[f"GLSD_{name}"]) for name in ("TP", "FP", "FN"))
    if native != expected["native"] or tuned != expected["tuned"]:
        raise RuntimeError(f"TRANSFER_PROTOCOL_FAIL: {setting} exact replay mismatch")
    return {"setting": setting, "native": native, "tuned": tuned, "source": str(candidates[0])}


def casme3(root: Path) -> dict:
    path = root / "recovery_manifest.json"
    data = json.loads(path.read_text(encoding="utf-8"))
    setting = "boosting_casme3"
    expected = LOCKED[setting]
    native = tuple(int(x) for x in data["native"]["full_counts"])
    tuned = tuple(int(x) for x in data["glsd_90"]["full_counts"])
    if data.get("status") != "PASS" or not data.get("all_functional_checks_passed"):
        raise RuntimeError("TRANSFER_PROTOCOL_FAIL: BoostingVRME/CAS(ME)3 replay did not PASS")
    if native != expected["native"] or tuned != expected["tuned"]:
        raise RuntimeError("TRANSFER_PROTOCOL_FAIL: BoostingVRME/CAS(ME)3 exact counts mismatch")
    return {"setting": setting, "native": native, "tuned": tuned, "source": str(path)}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-root", type=Path, required=True)
    parser.add_argument("--metst-sammlv", type=Path, required=True)
    parser.add_argument("--metst-casme3", type=Path, required=True)
    parser.add_argument("--boosting-sammlv", type=Path, required=True)
    parser.add_argument("--boosting-casme3", type=Path, required=True)
    args = parser.parse_args()
    mapping_gate = args.output_root / "audit/mapping_gate.json"
    mapping = json.loads(mapping_gate.read_text(encoding="utf-8"))
    if mapping.get("status") != "PASS":
        raise RuntimeError("TRANSFER_PROTOCOL_FAIL: mapping gate is not PASS")
    records = [
        one_generic(args.metst_sammlv, "metst_sammlv"),
        one_generic(args.metst_casme3, "metst_casme3"),
        one_generic(args.boosting_sammlv, "boosting_sammlv"),
        casme3(args.boosting_casme3),
    ]
    native_rows, tuned_rows = [], []
    for record in records:
        spec = LOCKED[record["setting"]]
        native_rows.append({
            "setting": record["setting"], "pipeline": spec["pipeline"], "dataset": spec["dataset"],
            "TP": record["native"][0], "FP": record["native"][1], "FN": record["native"][2],
            "F1": f1(record["native"]), "exact_replay": "PASS", "source": record["source"],
        })
        tuned_rows.append({
            "setting": record["setting"], "pipeline": spec["pipeline"], "dataset": spec["dataset"],
            "TP": record["tuned"][0], "FP": record["tuned"][1], "FN": record["tuned"][2],
            "F1": f1(record["tuned"]), "exact_replay": "PASS", "source": record["source"],
        })
    write_csv(args.output_root / "audit/native_replay.csv", native_rows)
    write_csv(args.output_root / "audit/tuned_glsd90_replay.csv", tuned_rows)
    gate = {
        "status": "PASS", "native_exact_replays": 4, "tuned_glsd90_exact_replays": 4,
        "mapping_gate": str(mapping_gate), "transfer_allowed": True,
    }
    write_json(args.output_root / "audit/preflight_gate.json", gate)
    print("FOUR_SETTING_EXACT_REPLAY_GATE = PASS")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/transfer_bundle/run_transfer_target.py
#!/usr/bin/env python3
"""Run one target-side transfer direction with no target-side selection."""

from __future__ import annotations

import argparse
import importlib.util
import json
import sys
from pathlib import Path

import numpy as np

from transfer_common import (
    DATASETS, DIRECTIONS, LOCKED, canonical_subject, f1, load_csv, metrics,
    normalize_json, paired_bootstrap, selected_config_rows, write_csv, write_json,
)
from transfer_paths import EVIDENCE, RESPONSES, RUNNERS


def load_module(path: Path, name: str, repo_root: Path | None = None):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"cannot import module: {path}")
    module = importlib.util.module_from_spec(spec)
    old_path = list(sys.path)
    additions = [str(path.parent)]
    if repo_root is not None:
        additions.insert(0, str(repo_root))
    sys.path[:0] = additions
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    finally:
        sys.path[:] = old_path
    return module


def theta_of(config) -> tuple[float, float, float]:
    if isinstance(config, dict):
        return tuple(float(config[name]) for name in ("reference_scale", "local_radius", "threshold"))
    return tuple(float(getattr(config, name)) for name in ("reference_scale", "local_radius", "threshold"))


def target_config(grid, theta: tuple[float, float, float]):
    matches = [config for config in grid if theta_of(config) == theta]
    if len(matches) != 1:
        raise RuntimeError(f"source theta {theta} does not map uniquely into target locked GLSD-90 grid")
    return matches[0]


def assert_source_k_never_passed(source: dict) -> None:
    assert "source_k" not in source and "k" not in source and "k_p" not in source, (
        "source_k is never passed into target decoder execution"
    )


def sealed_tuned_counts(path: Path, subjects: list[str], dataset: str) -> dict[str, tuple[int, int, int]]:
    rows = load_csv(path)
    first = rows[0]
    subject_name = next((name for name in first if name.lower() in {"outer_subject", "subject", "subject_id"}), None)
    if subject_name is None:
        raise RuntimeError(f"sealed per-subject table lacks subject field: {path}")
    lowered = {name.lower(): name for name in first}
    result = {}
    for row in rows:
        subject = canonical_subject(dataset, row[subject_name])
        counts = []
        for metric_name in ("tp", "fp", "fn"):
            candidates = (f"full_{metric_name}", f"full_{metric_name.upper()}")
            key = next((lowered[name.lower()] for name in candidates if name.lower() in lowered), None)
            if key is None:
                raise RuntimeError(f"sealed table lacks full {metric_name}: {path}")
            counts.append(int(row[key]))
        result[subject] = tuple(counts)
    if set(result) != set(subjects):
        raise RuntimeError("target sealed tuned counts do not match target subject IDs")
    return result


def generic_target(setting: str, source_configs: dict[str, dict]):
    import official_response_component_ablation as ora

    spec = ora.SPECS[setting]
    ora.verify_sealed_inputs(spec)
    context = ora.metst_context(spec) if spec["kind"] == "metst" else ora.boosting_context(spec)
    if spec["kind"] == "metst":
        runner, base, metric_class, official, records, _paths, raw_subjects, _native = context
        grid = ora.locked_grid(base)
    else:
        runner, cache, grouped, raw_subjects, metric_class, official, _native = context
        base = runner
        grid = ora.locked_grid(base)
    dataset = LOCKED[setting]["dataset"]
    subjects = [canonical_subject(dataset, value) for value in raw_subjects]
    if len(subjects) != len(set(subjects)):
        raise RuntimeError("duplicate canonical target subject IDs")
    if set(subjects) != set(source_configs):
        raise RuntimeError("source/target canonical subject mapping is incomplete")
    tuned = sealed_tuned_counts(EVIDENCE[setting] / "per_subject_counts.csv", subjects, dataset)

    rows, configs, prediction_rows = [], [], []
    for index, subject in enumerate(subjects):
        source = source_configs[subject]
        assert_source_k_never_passed(source)
        theta = (source["a0"], source["rho"], source["tau"])
        config = target_config(grid, theta)
        native_raw, native_full = ora.native_subject_counts(spec["kind"], context, index)
        if spec["kind"] == "metst":
            target_k = int(records[index]["k_p"])
            raw, predictions, pred_list, gt_list, *_ = runner.decode_glsd_subject(
                records, index, config, base, metric_class, official, True,
            )
            full = runner.full_counts_from_official_synergy(raw, pred_list, gt_list)
        else:
            target_k = int(cache["config"]["k_p"])
            samples, _ = runner.samples_and_emotions(grouped)
            raw, predictions, pred_list, gt_list, *_ = runner.decode_glsd_subject(
                grouped, samples, index, config, official, metric_class, cache["config"], True,
            )
            full = runner.full_counts_from_official_synergy(raw, pred_list, gt_list, cache["config"])
        raw = tuple(int(value) for value in raw)
        full = tuple(int(value) for value in full)
        tuned_counts = tuned[subject]
        row = {
            "dataset": dataset, "subject": subject,
            "source_pipeline": LOCKED[DIRECTION["source"]]["pipeline"],
            "target_pipeline": LOCKED[setting]["pipeline"],
            "source_a0": source["a0"], "source_rho": source["rho"], "source_tau": source["tau"],
            "source_config_origin": source["source_config_origin"],
            "source_outer_fold": source["source_outer_fold"], "target_k": target_k,
            "target_native_TP": native_full[0], "target_native_FP": native_full[1], "target_native_FN": native_full[2],
            "target_tuned_TP": tuned_counts[0], "target_tuned_FP": tuned_counts[1], "target_tuned_FN": tuned_counts[2],
            "transfer_TP": full[0], "transfer_FP": full[1], "transfer_FN": full[2],
            "target_native_F1_subject": f1(native_full), "target_tuned_F1_subject": f1(tuned_counts),
            "transfer_F1_subject": f1(full),
        }
        rows.append(row)
        configs.append({key: row[key] for key in (
            "dataset", "subject", "source_pipeline", "target_pipeline", "source_a0", "source_rho",
            "source_tau", "source_config_origin", "source_outer_fold", "target_k",
        )})
        prediction_rows.append({
            "dataset": dataset, "subject": subject, "target_pipeline": LOCKED[setting]["pipeline"],
            "theta_source": {"a0": source["a0"], "rho": source["rho"], "tau": source["tau"]},
            "target_k": target_k, "raw_counts": list(raw), "full_counts": list(full),
            "predictions": normalize_json(predictions),
        })
    return rows, configs, prediction_rows


def casme3_boosting_target(source_configs: dict[str, dict]):
    import boosting_casme3_official_phase2 as helper

    setting = "boosting_casme3"
    dataset = "casme3"
    runner = load_module(RUNNERS[setting], "transfer_boosting_casme3_runner", Path("/content/BoostingVRME"))
    cache, videos = runner.load_cache(RESPONSES[setting])
    grouped = runner.group_videos(videos)
    raw_subjects = [str(subject[0]["subject"]) for subject in grouped]
    subjects = [canonical_subject(dataset, value) for value in raw_subjects]
    expected = DATASETS[dataset]
    gt = sum(len(video["gt_intervals"]) for video in videos)
    if (len(subjects), len(videos), gt) != (expected["subjects"], expected["videos"], expected["gt"]):
        raise RuntimeError("BoostingVRME/CAS(ME)3 target universe mismatch")
    if set(subjects) != set(source_configs):
        raise RuntimeError("source/target canonical subject mapping is incomplete")
    cache_config = cache["config"]
    target_k = int(cache_config["k_p"])
    final_samples = [[video["gt_intervals"] for video in subject] for subject in grouped]
    final_emotions = [[video["emotion_labels"] for video in subject] for subject in grouped]
    feature_bank = [
        [runner.GLSDFeatures(video["spot_response"], target_k) for video in subject]
        for subject in grouped
    ]
    grid = list(runner.build_grid())
    if len(grid) != 90:
        raise RuntimeError("BoostingVRME/CAS(ME)3 target grid is not locked GLSD-90")
    tuned_raw, tuned_full = helper.load_locked_glsd_counts(
        EVIDENCE[setting] / "per_subject_counts.csv", raw_subjects,
    )
    rows, configs, prediction_rows = [], [], []
    for index, subject in enumerate(subjects):
        source = source_configs[subject]
        assert_source_k_never_passed(source)
        theta = (source["a0"], source["rho"], source["tau"])
        config = target_config(grid, theta)
        native_raw, native_full = helper.native_subject_counts(
            runner, cache_config, grouped, final_samples, final_emotions, index,
        )
        decoded = runner.decode_subject(
            index, config, grouped, feature_bank, final_samples, final_emotions,
            cache_config, need_recognition=True,
        )
        raw = tuple(int(value) for value in decoded["raw_counts"])
        full = tuple(int(value) for value in decoded["full_counts"])
        tuned_counts = tuple(int(value) for value in tuned_full[index])
        row = {
            "dataset": dataset, "subject": subject,
            "source_pipeline": LOCKED[DIRECTION["source"]]["pipeline"],
            "target_pipeline": LOCKED[setting]["pipeline"],
            "source_a0": source["a0"], "source_rho": source["rho"], "source_tau": source["tau"],
            "source_config_origin": source["source_config_origin"],
            "source_outer_fold": source["source_outer_fold"], "target_k": target_k,
            "target_native_TP": int(native_full[0]), "target_native_FP": int(native_full[1]), "target_native_FN": int(native_full[2]),
            "target_tuned_TP": tuned_counts[0], "target_tuned_FP": tuned_counts[1], "target_tuned_FN": tuned_counts[2],
            "transfer_TP": full[0], "transfer_FP": full[1], "transfer_FN": full[2],
            "target_native_F1_subject": f1(native_full), "target_tuned_F1_subject": f1(tuned_counts),
            "transfer_F1_subject": f1(full),
        }
        rows.append(row)
        configs.append({key: row[key] for key in (
            "dataset", "subject", "source_pipeline", "target_pipeline", "source_a0", "source_rho",
            "source_tau", "source_config_origin", "source_outer_fold", "target_k",
        )})
        prediction_rows.append({
            "dataset": dataset, "subject": subject, "target_pipeline": LOCKED[setting]["pipeline"],
            "theta_source": {"a0": source["a0"], "rho": source["rho"], "tau": source["tau"]},
            "target_k": target_k, "raw_counts": list(raw), "full_counts": list(full),
            "predictions": normalize_json(decoded["preds"]),
        })
    return rows, configs, prediction_rows


def validate_and_write(output: Path, rows: list[dict], configs: list[dict], predictions: list[dict]) -> None:
    target_setting = DIRECTION["target"]
    expected = LOCKED[target_setting]
    native = np.asarray([[row[f"target_native_{name}"] for name in ("TP", "FP", "FN")] for row in rows])
    tuned = np.asarray([[row[f"target_tuned_{name}"] for name in ("TP", "FP", "FN")] for row in rows])
    transferred = np.asarray([[row[f"transfer_{name}"] for name in ("TP", "FP", "FN")] for row in rows])
    if tuple(native.sum(axis=0)) != expected["native"]:
        raise RuntimeError("TRANSFER_PROTOCOL_FAIL: target Native per-subject replay mismatch")
    if tuple(tuned.sum(axis=0)) != expected["tuned"]:
        raise RuntimeError("TRANSFER_PROTOCOL_FAIL: target tuned GLSD-90 evidence mismatch")
    if output.exists():
        raise RuntimeError(f"refusing to overwrite transfer output: {output}")
    output.mkdir(parents=True)
    write_csv(output / "transferred_configs.csv", configs)
    write_csv(output / "per_subject_counts.csv", rows)
    write_json(output / "predictions.json", predictions)
    bootstrap = paired_bootstrap(native, tuned, transferred)
    write_json(output / "bootstrap.json", bootstrap)
    native_counts = tuple(int(x) for x in native.sum(axis=0))
    tuned_counts = tuple(int(x) for x in tuned.sum(axis=0))
    transfer_counts = tuple(int(x) for x in transferred.sum(axis=0))
    result = {
        "direction": DIRECTION_NAME, "dataset": DIRECTION["dataset"],
        "source_pipeline": LOCKED[DIRECTION["source"]]["pipeline"],
        "target_pipeline": LOCKED[DIRECTION["target"]]["pipeline"],
        "target_native": metrics(native_counts), "target_tuned_glsd90": metrics(tuned_counts),
        "transferred": metrics(transfer_counts),
        "delta_vs_native": f1(transfer_counts) - f1(native_counts),
        "delta_vs_tuned": f1(transfer_counts) - f1(tuned_counts),
        "bootstrap_95_ci_vs_native": [bootstrap["vs_native"]["percentile_2_5"], bootstrap["vs_native"]["percentile_97_5"]],
        "bootstrap_95_ci_vs_tuned": [bootstrap["vs_tuned"]["percentile_2_5"], bootstrap["vs_tuned"]["percentile_97_5"]],
    }
    write_json(output / "metrics.json", result)
    write_json(output / "protocol_checks.json", {
        "status": "PASS", "source_config_fields": ["a0", "rho", "tau"],
        "source_k_passed_to_target": False, "target_k_origin": "target official-response context",
        "target_side_configuration_search": False,
        "target_event_geometry_preserved": True, "target_boundary_candidate_conflict_rules_preserved": True,
        "target_official_evaluator_preserved": True, "target_recognition_result_synergy_preserved": True,
    })


def main() -> None:
    global DIRECTION_NAME, DIRECTION
    parser = argparse.ArgumentParser()
    parser.add_argument("--direction", choices=tuple(DIRECTIONS), required=True)
    parser.add_argument("--output-root", type=Path, required=True)
    args = parser.parse_args()
    DIRECTION_NAME = args.direction
    DIRECTION = DIRECTIONS[DIRECTION_NAME]
    gate = json.loads((args.output_root / "audit/preflight_gate.json").read_text(encoding="utf-8"))
    if gate.get("status") != "PASS" or not gate.get("transfer_allowed"):
        raise RuntimeError("TRANSFER_PROTOCOL_FAIL: four-setting exact replay gate is not PASS")
    source_path = EVIDENCE[DIRECTION["source"]] / "outer_selected_configs.csv"
    source_configs = selected_config_rows(source_path, DIRECTION["dataset"])
    if DIRECTION["target"] == "boosting_casme3":
        rows, configs, predictions = casme3_boosting_target(source_configs)
    else:
        rows, configs, predictions = generic_target(DIRECTION["target"], source_configs)
    expected_subjects = DATASETS[DIRECTION["dataset"]]["subjects"]
    if len(rows) != expected_subjects:
        raise RuntimeError("TRANSFER_PROTOCOL_FAIL: transfer subject count mismatch")
    output = args.output_root / DIRECTION["relative_output"]
    validate_and_write(output, rows, configs, predictions)
    print(f"TRANSFER_DIRECTION_PASS = {DIRECTION_NAME}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/transfer_bundle/finalize_transfer.py
#!/usr/bin/env python3
"""Validate four completed directions and create paper-facing summaries."""

from __future__ import annotations

import argparse
import json
from pathlib import Path

from transfer_common import DIRECTIONS, write_csv, write_json


ORDER = (
    "metst_to_boosting_sammlv",
    "metst_to_boosting_casme3",
    "boosting_to_metst_sammlv",
    "boosting_to_metst_casme3",
)


def fmt(value: float) -> str:
    return f"{float(value):.6f}"


def ci(values) -> str:
    return f"[{float(values[0]):.6f}, {float(values[1]):.6f}]"


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--output-root", type=Path, required=True)
    args = parser.parse_args()
    root = args.output_root.resolve()
    gate = json.loads((root / "audit/preflight_gate.json").read_text(encoding="utf-8"))
    if gate.get("status") != "PASS":
        raise RuntimeError("TRANSFER_PROTOCOL_FAIL: preflight gate is not PASS")
    summary_dir = root / "summary"
    if summary_dir.exists():
        raise RuntimeError(f"refusing to overwrite summary: {summary_dir}")
    summary_dir.mkdir(parents=True)

    results = []
    for name in ORDER:
        spec = DIRECTIONS[name]
        direction_dir = root / spec["relative_output"]
        checks = json.loads((direction_dir / "protocol_checks.json").read_text(encoding="utf-8"))
        if checks.get("status") != "PASS" or checks.get("source_k_passed_to_target"):
            raise RuntimeError(f"TRANSFER_PROTOCOL_FAIL: protocol check failed for {name}")
        result = json.loads((direction_dir / "metrics.json").read_text(encoding="utf-8"))
        results.append(result)

    csv_rows, paper_rows = [], []
    for result in results:
        native = result["target_native"]
        tuned = result["target_tuned_glsd90"]
        transfer = result["transferred"]
        row = {
            "direction": result["direction"], "dataset": result["dataset"],
            "source_pipeline": result["source_pipeline"], "target_pipeline": result["target_pipeline"],
            "native_TP": native["TP"], "native_FP": native["FP"], "native_FN": native["FN"], "native_F1": native["F1"],
            "tuned_TP": tuned["TP"], "tuned_FP": tuned["FP"], "tuned_FN": tuned["FN"], "tuned_F1": tuned["F1"],
            "transfer_TP": transfer["TP"], "transfer_FP": transfer["FP"], "transfer_FN": transfer["FN"], "transfer_F1": transfer["F1"],
            "delta_vs_native": result["delta_vs_native"],
            "ci_low_vs_native": result["bootstrap_95_ci_vs_native"][0],
            "ci_high_vs_native": result["bootstrap_95_ci_vs_native"][1],
            "delta_vs_tuned": result["delta_vs_tuned"],
            "ci_low_vs_tuned": result["bootstrap_95_ci_vs_tuned"][0],
            "ci_high_vs_tuned": result["bootstrap_95_ci_vs_tuned"][1],
        }
        csv_rows.append(row)
        short = "M -> B" if result["source_pipeline"] == "ME-TST+" else "B -> M"
        paper_rows.append({
            "Transfer": f"{short} / {result['dataset'].upper()}",
            "Target Native": fmt(native["F1"]),
            "Target Tuned GLSD-90": fmt(tuned["F1"]),
            "Transferred": fmt(transfer["F1"]),
            "Delta vs Native": fmt(result["delta_vs_native"]),
            "95% CI vs Native": ci(result["bootstrap_95_ci_vs_native"]),
            "Delta vs Tuned": fmt(result["delta_vs_tuned"]),
            "95% CI vs Tuned": ci(result["bootstrap_95_ci_vs_tuned"]),
        })

    write_csv(summary_dir / "transfer_summary.csv", csv_rows)
    write_json(summary_dir / "transfer_summary.json", results)
    write_csv(summary_dir / "paper_table.csv", paper_rows)
    positives = [row for row in csv_rows if float(row["delta_vs_native"]) > 0]
    ci_positive = [row for row in csv_rows if float(row["ci_low_vs_native"]) > 0]
    negative = [row for row in csv_rows if float(row["delta_vs_native"]) < 0]
    negative_names = ", ".join(row["direction"] for row in negative) if negative else "None"
    lines = [
        "CURRENT PROTOCOL CROSS-PIPELINE TRANSFER FINAL AUDIT",
        "",
        "1. Current official-response GLSD-90 main experiments found: PASS",
        "2. Four Native exact replays: PASS",
        "3. Four tuned GLSD-90 exact replays: PASS",
        "4. SAMMLV universe 29 subjects / 79 videos / 159 GT: PASS",
        "5. CAS(ME)3 universe 94 subjects / 462 videos / 853 GT: PASS",
        "6. Source/target subject and video mapping complete: PASS",
        "7. Transferred fields are only (a0, rho, tau): PASS",
        "8. Source k was not transferred: PASS",
        "9. Target event geometry/native output rules preserved: PASS",
        "10. Target-side tuning or test leakage detected: NO",
        "",
        "Pooled results:",
    ]
    for row in csv_rows:
        lines.append(
            f"- {row['direction']}: transfer TP/FP/FN/F1="
            f"{row['transfer_TP']}/{row['transfer_FP']}/{row['transfer_FN']}/{float(row['transfer_F1']):.12f}; "
            f"Delta vs Native={float(row['delta_vs_native']):+.12f} "
            f"[{float(row['ci_low_vs_native']):+.12f}, {float(row['ci_high_vs_native']):+.12f}]; "
            f"Delta vs Tuned={float(row['delta_vs_tuned']):+.12f} "
            f"[{float(row['ci_low_vs_tuned']):+.12f}, {float(row['ci_high_vs_tuned']):+.12f}]"
        )
    lines.extend([
        "",
        f"15. Point estimates above target Native: {len(positives)}/4",
        f"16. CIs completely above zero vs Native: {len(ci_positive)}/4",
        f"17. Negative transfer exists: {'YES' if negative else 'NO'}",
        f"18. Negative-transfer directions: {negative_names}",
        "19. Interpretation: descriptive only; no claim of universal transfer is made.",
        "",
        "FINAL_STATUS = PASS",
    ])
    (summary_dir / "FINAL_AUDIT.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
    (root / "audit/protocol_audit.txt").write_text(
        "Only (a0, rho, tau) crossed pipelines. Target response, k, event geometry, "
        "boundary search, candidate order, conflict handling, recognition/result synergy, "
        "and evaluator remained target-owned. No target-side search was invoked.\n"
        "PROTOCOL_AUDIT = PASS\n",
        encoding="utf-8",
    )
    print("CURRENT_PROTOCOL_CROSS_PIPELINE_TRANSFER = PASS")
    print("OUTPUT_ROOT =", root)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/transfer_bundle/test_protocol.py
#!/usr/bin/env python3
"""Dependency-light smoke tests; these do not read official responses."""

from __future__ import annotations

import csv
import tempfile
from pathlib import Path

import numpy as np

from transfer_common import canonical_subject, paired_bootstrap, selected_config_rows
from run_transfer_target import assert_source_k_never_passed, target_config, theta_of


def test_canonical_subjects() -> None:
    assert canonical_subject("sammlv", "6") == "006"
    assert canonical_subject("sammlv", "006") == "006"
    assert canonical_subject("casme3", "015") == "15"


def test_selected_config_rejects_k() -> None:
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "bad.csv"
        with path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=["outer_subject", "reference_scale", "local_radius", "threshold", "k_p"])
            writer.writeheader()
            writer.writerow({"outer_subject": "006", "reference_scale": 1, "local_radius": 2, "threshold": 0.3, "k_p": 17})
        try:
            selected_config_rows(path, "sammlv")
        except RuntimeError as exc:
            assert "forbidden transfer fields" in str(exc)
        else:
            raise AssertionError("source k was not rejected")


def test_target_config_is_target_owned() -> None:
    grid = [
        {"reference_scale": 1.0, "local_radius": 2.0, "threshold": 0.3, "target_only": 7},
        {"reference_scale": 1.5, "local_radius": 2.0, "threshold": 0.3, "target_only": 8},
    ]
    selected = target_config(grid, (1.5, 2.0, 0.3))
    assert selected["target_only"] == 8
    assert theta_of(selected) == (1.5, 2.0, 0.3)


def test_explicit_source_k_assertion() -> None:
    assert_source_k_never_passed({"a0": 1.0, "rho": 2.0, "tau": 0.3})
    try:
        assert_source_k_never_passed({"a0": 1.0, "rho": 2.0, "tau": 0.3, "source_k": 17})
    except AssertionError:
        pass
    else:
        raise AssertionError("explicit source-k assertion did not fire")


def test_bootstrap_is_paired_and_deterministic() -> None:
    native = np.asarray([[1, 2, 3], [2, 1, 2], [0, 1, 1]])
    tuned = np.asarray([[1, 1, 3], [2, 1, 2], [0, 1, 1]])
    transferred = np.asarray([[2, 1, 2], [2, 1, 2], [0, 1, 1]])
    first = paired_bootstrap(native, tuned, transferred, iterations=100, seed=100)
    second = paired_bootstrap(native, tuned, transferred, iterations=100, seed=100)
    assert first == second
    assert first["paired_resampling"] is True
    assert first["unit"] == "subject"


if __name__ == "__main__":
    test_canonical_subjects()
    test_selected_config_rejects_k()
    test_target_config_is_target_owned()
    test_explicit_source_k_assertion()
    test_bootstrap_is_paired_and_deterministic()
    print("TRANSFER_PROTOCOL_SMOKE_TESTS = PASS")


In [ ]:
%%writefile /content/transfer_bundle/official_response_component_ablation.py
"""Colab-only matched-budget component ablation for sealed official responses.

Run after Drive is mounted.  This program never imports a backbone, reads a
video, generates an official response, or writes inside a locked evidence
directory.  It imports the exact decoder/evaluator source recorded by each
notebook and changes only candidate scores in memory.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import importlib.util
import json
import sys
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np


DRIVE = Path("/content/drive/MyDrive")
MODES = ("H", "G", "L", "GL")
MODE_LABELS = {"H": "H", "G": "G", "L": "L", "GL": "matched-budget-GL-30"}
SCALES = (1.0, 1.5, 2.0)
RHO = 2.0
SEED = 100
REPEATS = 10_000


@dataclass(frozen=True)
class ComponentConfig:
    reference_scale: float
    local_radius: float
    threshold: float

    @property
    def identifier(self) -> str:
        return (
            f"reference_scale={self.reference_scale:g}|"
            f"local_radius={self.local_radius:g}|threshold={self.threshold:g}"
        )


# These locations are transcribed from the three supplied final Colab notebooks.
# The Boosting source programs live in the Colab checkout used by those notebooks;
# their SHA-256 is checked against the sealed result manifest after Drive mount.
SPECS = {
    "metst_sammlv": {
        "label": "ME-TST+ / SAMMLV",
        "kind": "metst",
        "dump": DRIVE / "ME-TST_OFFICIAL_DUMP/SAMMLV_method1_strategy1",
        "evidence": DRIVE / "GLSD_METST_OFFICIAL/SAMMLV_FINAL_EVIDENCE",
        "run_evidence": DRIVE / "GLSD_METST_OFFICIAL/SAMMLV_FINAL_EVIDENCE/results_v1",
        "results": DRIVE / "GLSD_METST_OFFICIAL/results_v1",
        "locked_results": DRIVE / "GLSD_METST_OFFICIAL/SAMMLV_FINAL_EVIDENCE/results_v1",
        "source": DRIVE / "GLSD_METST_OFFICIAL/SAMMLV_FINAL_EVIDENCE/metst_official_glds_full.py",
        "core": DRIVE / "GLSD_METST_OFFICIAL/SAMMLV_FINAL_EVIDENCE/boosting_official_glds_full_LOCKED.py",
        "subjects": 29,
        "videos": 79,
        "gt": 159,
        "native_full": (52, 171, 107),
        "glsd_full": (47, 121, 112),
        "native_reported_f1": "0.2723",
        "glsd_reported_f1": "0.2875",
    },
    "metst_casme3": {
        "label": "ME-TST+ / CAS(ME)3",
        "kind": "metst",
        "dump": DRIVE / "ME-TST_OFFICIAL_DUMP/CASME_3_method1_strategy1",
        "evidence": DRIVE / "GLSD_METST_OFFICIAL/CASME3_FINAL_EVIDENCE",
        "run_evidence": DRIVE / "GLSD_METST_OFFICIAL/CASME3_FINAL_EVIDENCE/results_v1",
        "results": DRIVE / "GLSD_METST_OFFICIAL/CASME3/results_v1",
        "locked_results": DRIVE / "GLSD_METST_OFFICIAL/CASME3_FINAL_EVIDENCE/results_v1",
        "source": DRIVE / "GLSD_METST_OFFICIAL/CASME3_FINAL_EVIDENCE/source/metst_casme3_official_glds_full.py",
        "core": DRIVE / "GLSD_METST_OFFICIAL/CASME3_FINAL_EVIDENCE/source/boosting_official_glds_full_LOCKED.py",
        "subjects": 94,
        "videos": 462,
        "gt": 853,
        "native_full": (76, 736, 777),
        "glsd_full": (90, 898, 763),
        "native_reported_f1": "0.0912912913",
        "glsd_reported_f1": "0.0977729495",
    },
    "boosting_sammlv": {
        "label": "BoostingVRME / SAMMLV",
        "kind": "boosting",
        "cache": DRIVE / "GLSD_BoostingVRME_Full/sammlv_official_full_responses.pkl",
        "evidence": DRIVE / "GLSD_BoostingVRME_Full/results_evidence_v2",
        "run_evidence": DRIVE / "GLSD_BoostingVRME_Full/results_evidence_v2",
        "locked_results": DRIVE / "GLSD_BoostingVRME_Full/results_evidence_v2",
        "source": Path("/content/BoostingVRME/boosting_official_glds_full.py"),
        "subjects": 29,
        "videos": 79,
        "gt": 159,
        "native_full": (51, 141, 108),
        "glsd_full": (44, 82, 115),
        "native_reported_f1": "0.2906",
        "glsd_reported_f1": "0.3088",
    },
    "boosting_casme3": {
        "label": "BoostingVRME / CAS(ME)3",
        "kind": "boosting",
        "cache": DRIVE / "GLSD_BoostingVRME_Full/CASME3/official_response_cache/casme3_official_full_responses.pkl",
        "evidence": DRIVE / "GLSD_BoostingVRME_Full/CASME3/glsd_full_evidence_v1",
        "run_evidence": DRIVE / "GLSD_BoostingVRME_Full/CASME3/glsd_full_evidence_v1",
        "locked_results": DRIVE / "GLSD_BoostingVRME_Full/CASME3/glsd_full_evidence_v1",
        "source": Path("/content/BoostingVRME/boosting_official_glds_full_casme3.py"),
        "subjects": 94,
        "videos": 462,
        "gt": 853,
        "native_full": (84, 786, 769),
        "glsd_full": (120, 1110, 733),
        "native_reported_f1": "0.0975043529",
        "glsd_reported_f1": "0.1152184349",
    },
}


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def f1(counts) -> float:
    tp, fp, fn = (int(value) for value in counts)
    return 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0


def metrics(counts) -> dict:
    tp, fp, fn = (int(value) for value in counts)
    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": tp / (tp + fp) if tp + fp else 0.0,
        "recall": tp / (tp + fn) if tp + fn else 0.0,
        "F1": f1((tp, fp, fn)),
    }


def write_json(path: Path, value) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        raise RuntimeError(f"refusing to write empty CSV: {path.name}")
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def load_module(path: Path, name: str):
    spec = importlib.util.spec_from_file_location(name, str(path))
    if spec is None or spec.loader is None:
        raise RuntimeError(f"cannot import sealed source: {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    sys.path.insert(0, str(path.parent))
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    finally:
        sys.path.pop(0)
    return module


def required_paths(spec: dict) -> list[Path]:
    evidence = spec["evidence"]
    run_evidence = spec["run_evidence"]
    common = [
        run_evidence / "per_subject_counts.csv",
        run_evidence / "run_manifest.json",
        evidence / "evidence_bundle_sha256.txt",
        run_evidence / "outer_selected_configs.csv",
        spec["source"],
    ]
    if spec["kind"] == "metst":
        common.extend([spec["dump"], spec["core"]])
    else:
        common.extend([spec["cache"], evidence / "glsd_search_inner_fold_counts.csv"])
    return common


def verify_sealed_inputs(spec: dict) -> dict:
    missing = [str(path) for path in required_paths(spec) if not path.exists()]
    if missing:
        raise RuntimeError("Drive preflight failed; missing sealed input(s):\n" + "\n".join(missing))
    manifest = json.loads((spec["run_evidence"] / "run_manifest.json").read_text(encoding="utf-8"))
    text = (spec["evidence"] / "evidence_bundle_sha256.txt").read_text(encoding="utf-8")
    if not text.strip():
        raise RuntimeError("sealed bundle checksum list is empty")
    for key, expected in (("subjects", spec["subjects"]), ("videos", spec["videos"])):
        if key in manifest and int(manifest[key]) != expected:
            raise RuntimeError(f"{spec['label']}: manifest {key} mismatch")
    for key in ("glsd_full_counts", "native_full_counts"):
        if key in manifest:
            got = tuple(int(x) for x in manifest[key])
            expected = spec["glsd_full"] if key.startswith("glsd") else spec["native_full"]
            if got != expected:
                raise RuntimeError(f"{spec['label']}: {key} does not match its locked result")
    return manifest


def component_grid(thresholds) -> list[ComponentConfig]:
    values = tuple(float(t) for t in thresholds)
    if len(values) != 10 or len(set(values)) != 10:
        raise RuntimeError("locked tau grid is not exactly ten unique values")
    return [ComponentConfig(scale, RHO, tau) for scale in SCALES for tau in values]


def make_component_features(base_features, mode: str):
    """Return a class that preserves the locked candidates and G/L construction.

    The only replacement is the scalar score used to threshold those candidates.
    H follows the locked protocol's normalized reference-curve amplitude:
    max(0, (peak - mean) / (max - mean)).
    """
    class ComponentFeatures(base_features):
        def selected_peaks(self, config):
            peaks, evidence = self.evidence(config.reference_scale, RHO)
            peaks = np.asarray(peaks, dtype=int)
            if not len(peaks):
                return peaks
            global_values, local_values = np.asarray(evidence, dtype=float).T
            if mode == "G":
                score = global_values
            elif mode == "L":
                score = local_values
            elif mode == "GL":
                score = (global_values + local_values) / 2.0
            else:
                # scale_data is the sealed core's reference-curve cache.  It is
                # inspected, never modified, and is the same curve that produced G.
                _peaks, curve, _spread, _prominence = self.scale_data[config.reference_scale]
                centre = float(np.mean(curve))
                # Exact inherited H definition from
                # my_method/gl_saliency_skill/evidence.py:99:
                # maximum(0, (smooth[peaks]-mean)/maximum(smooth.max()-mean, 1e-12)).
                score = np.maximum(
                    0.0,
                    (np.asarray(curve)[peaks] - centre)
                    / max(float(np.max(curve) - centre), 1e-12),
                )
            return peaks[np.asarray(score) >= float(config.threshold)]
    return ComponentFeatures


class ComponentCore:
    def __init__(self, base, mode: str, grid: list[ComponentConfig]):
        if not hasattr(base.GLSDFeatures, "evidence"):
            raise RuntimeError("locked GLSD interface has no evidence() method; abort rather than reimplement it")
        self.GLSDFeatures = make_component_features(base.GLSDFeatures, mode)
        self._grid = grid
        self.selection_order = base.selection_order

    def configuration_grid(self):
        return list(self._grid)


def outer_selection(raw_table: np.ndarray, subjects: list[str], selection_order):
    selected, traces, outer_rows = {}, [], []
    for outer, subject in enumerate(subjects):
        inner = [index for index in range(len(subjects)) if index != outer]
        pooled = raw_table[:, inner, :].sum(axis=1)
        order = np.asarray(selection_order(pooled), dtype=int)
        winner = int(order[0])
        ranks = np.empty(len(order), dtype=int)
        ranks[order] = np.arange(1, len(order) + 1)
        selected[subject] = winner
        candidates = []
        for config_id, values in enumerate(pooled):
            candidates.append({
                "config_id": config_id,
                "TP": int(values[0]), "FP": int(values[1]), "FN": int(values[2]),
                **metrics(values), "grid_order": config_id,
                "rank": int(ranks[config_id]), "selected": config_id == winner,
            })
        traces.append({
            "outer_subject": subject,
            "inner_subjects": [subjects[i] for i in inner],
            "winner_config_id": winner,
            "tie_break": ["higher F1", "higher precision", "fewer FP", "fixed grid order"],
            "candidates": candidates,
        })
        outer_rows.append((outer, subject, inner, winner, pooled[winner]))
    return selected, traces, outer_rows


def metst_context(spec: dict):
    runner = load_module(spec["source"], "sealed_metst_runner")
    base_core = load_module(spec["core"], "sealed_metst_core")
    metric, official = runner.load_official_metst()
    dump_paths, records = runner.load_records()
    native = runner.native_gate(records, metric, official)
    subjects = [str(x) for x in runner.subject_names(records)]
    if len(subjects) != spec["subjects"]:
        raise RuntimeError("ME-TST subject cardinality mismatch")
    return runner, base_core, metric, official, records, dump_paths, subjects, native


def boosting_context(spec: dict):
    runner = load_module(spec["source"], "sealed_boosting_runner")
    cache = runner.load_cache(spec["cache"])
    subjects, grouped = runner.group_videos(cache["videos"])
    metric, official = runner.official_functions()
    native = runner.native_exact_replay(cache, grouped)
    if len(subjects) != spec["subjects"]:
        raise RuntimeError("BoostingVRME subject cardinality mismatch")
    return runner, cache, grouped, [str(x) for x in subjects], metric, official, native


def native_subject_counts(kind: str, context, subject_index: int):
    if kind == "metst":
        runner, _core, metric, official, records, _paths, _subjects, _native = context
        raw, full, *_ = runner.decode_native_subject(records, subject_index, metric, official)
        return tuple(raw), tuple(full)
    runner, cache, grouped, _subjects, metric, official, _native = context
    videos = grouped[subject_index]
    samples, emotions = runner.samples_and_emotions(grouped)
    result_all = [np.asarray(video["spot_response"]) for video in videos]
    sequences = [np.asarray(video["recognition_sequence"]) for video in videos]
    map_cls, official_code = metric, official
    metric_final = map_cls(num_classes=1)
    predictions, _, total_gt, metric_video, metric_final = official_code.spotting(
        samples, subject_index, result_all, 0, metric_final,
        cache["config"]["k_p"], cache["config"]["interval_strategy"], cache["dataset"],
    )
    raw = tuple(int(x) for x in official_code.sequence_evaluation(total_gt, metric_final))
    pred_list, _, gt_tp_list, _, _ = official_code.recognition(
        sequences, predictions, metric_video, emotions, subject_index, [], [], samples, [], [],
        cache["config"]["frame_skip"], cache["config"]["penalty_strategy"],
    )
    return raw, tuple(runner.full_counts_from_official_synergy(raw, pred_list, gt_tp_list, cache["config"]))


def decode_component(kind: str, context, core, subject_index: int, config, with_recognition: bool):
    if kind == "metst":
        runner, _base, metric, official, records, _paths, _subjects, _native = context
        raw, _pred, pred_list, gt_tp_list, *_ = runner.decode_glsd_subject(
            records, subject_index, config, core, metric, official, with_recognition
        )
        full = None if not with_recognition else runner.full_counts_from_official_synergy(raw, pred_list, gt_tp_list)
        return tuple(raw), None if full is None else tuple(full)
    runner, cache, grouped, _subjects, metric, official, _native = context
    original = runner.GLSDFeatures
    runner.GLSDFeatures = core.GLSDFeatures
    try:
        samples, _emotions = runner.samples_and_emotions(grouped)
        raw, _pred, pred_list, gt_tp_list, *_ = runner.decode_glsd_subject(
            grouped, samples, subject_index, config, official, metric, cache["config"], with_recognition
        )
    finally:
        runner.GLSDFeatures = original
    full = None if not with_recognition else runner.full_counts_from_official_synergy(raw, pred_list, gt_tp_list, cache["config"])
    return tuple(raw), None if full is None else tuple(full)


def existing_glsd_subject_counts(path: Path, expected_subjects: int) -> dict[str, tuple[int, int, int]]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != expected_subjects:
        raise RuntimeError(f"sealed GLSD per-subject table has {len(rows)}, expected {expected_subjects}")
    field = {name.lower(): name for name in rows[0]}
    def get(row, name):
        key = field.get(name.lower())
        if key is None:
            raise RuntimeError(f"sealed per_subject_counts.csv lacks {name}")
        return int(row[key])
    result = {}
    for row in rows:
        subject = str(row[field.get("outer_subject", field.get("subject", ""))])
        if not subject:
            raise RuntimeError("sealed per-subject table lacks a subject identifier")
        result[subject] = (get(row, "full_TP"), get(row, "full_FP"), get(row, "full_FN"))
    return result


def bootstrap(setting: str, native: dict, glsd: dict, output: Path) -> None:
    subjects = sorted(native)
    if subjects != sorted(glsd):
        raise RuntimeError("Native/GLSD subject IDs differ; paired bootstrap is invalid")
    n = np.asarray([native[s] for s in subjects], dtype=np.int64)
    g = np.asarray([glsd[s] for s in subjects], dtype=np.int64)
    rng = np.random.default_rng(SEED)
    draws = rng.integers(0, len(subjects), size=(REPEATS, len(subjects)))
    def sampled(values):
        totals = values[draws].sum(axis=1).astype(float)
        return np.divide(2 * totals[:, 0], 2 * totals[:, 0] + totals[:, 1] + totals[:, 2], out=np.zeros(REPEATS), where=(2 * totals[:, 0] + totals[:, 1] + totals[:, 2]) > 0)
    samples = sampled(g) - sampled(n)
    np.save(output / "native_vs_glsd_bootstrap_samples.npy", samples)
    write_csv(output / "native_vs_glsd_bootstrap_summary.csv", [{
        "setting": setting,
        "statistic": "F1_GLSD_minus_F1_Native",
        "point_estimate": f1(g.sum(axis=0)) - f1(n.sum(axis=0)),
        "percentile_2_5": float(np.quantile(samples, 0.025)),
        "percentile_97_5": float(np.quantile(samples, 0.975)),
        "resamples": REPEATS, "seed": SEED, "unit": "subject", "metric": "Full Spotting F1",
    }])
    write_json(output / "bootstrap_manifest.json", {
        "setting": setting, "seed": SEED, "resamples": REPEATS,
        "unit": "subject", "same_resampled_subject_ids_for_native_and_glsd": True,
        "aggregation": "sum TP/FP/FN then compute F1", "statistic": "F1_GLSD - F1_Native",
        "subject_ids": subjects,
    })


def locked_grid(base):
    if hasattr(base, "configuration_grid"):
        return list(base.configuration_grid())
    if hasattr(base, "build_grid"):
        return list(base.build_grid())
    raise RuntimeError("sealed GLSD source exposes neither configuration_grid() nor build_grid()")


def selected_locked_configs(path: Path, grid, subjects: list[str]) -> dict[str, object]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != len(subjects):
        raise RuntimeError("locked outer_selected_configs.csv has the wrong fold count")
    fields = {name.lower(): name for name in rows[0]}
    subject_field = fields.get("outer_subject", fields.get("subject"))
    config_field = fields.get("config_id")
    if subject_field is None or config_field is None:
        raise RuntimeError("locked selection table lacks outer subject or config id")
    selected = {}
    for row in rows:
        subject = str(row[subject_field])
        config_id = int(row[config_field])
        if not 0 <= config_id < len(grid):
            raise RuntimeError("locked selected config is outside its GLSD-90 grid")
        selected[subject] = grid[config_id]
    if set(selected) != set(subjects):
        raise RuntimeError("locked selection subject IDs differ from official response subject IDs")
    return selected


def locked_glsd_subject_counts(kind: str, context, base, selected: dict[str, object], subjects: list[str]):
    result = {}
    for subject_index, subject in enumerate(subjects):
        config = selected[subject]
        if kind == "metst":
            runner, _core, metric, official, records, _paths, _subjects, _native = context
            raw, _pred, pred_list, gt_tp_list, *_ = runner.decode_glsd_subject(
                records, subject_index, config, base, metric, official, True
            )
            full = runner.full_counts_from_official_synergy(raw, pred_list, gt_tp_list)
        else:
            runner, cache, grouped, _subjects, metric, official, _native = context
            samples, _emotions = runner.samples_and_emotions(grouped)
            raw, _pred, pred_list, gt_tp_list, *_ = runner.decode_glsd_subject(
                grouped, samples, subject_index, config, official, metric, cache["config"], True
            )
            full = runner.full_counts_from_official_synergy(raw, pred_list, gt_tp_list, cache["config"])
        result[subject] = (tuple(int(x) for x in raw), tuple(int(x) for x in full))
    return result


def context_video_and_gt_counts(kind: str, context) -> tuple[int, int]:
    if kind == "metst":
        runner, _core, _metric, _official, records, _paths, _subjects, _native = context
        final_samples = runner.final_samples(records)
        return (
            sum(len(subject_videos) for subject_videos in final_samples),
            sum(len(video_gt) for subject_videos in final_samples for video_gt in subject_videos),
        )
    _runner, cache, _grouped, _subjects, _metric, _official, _native = context
    return (
        len(cache["videos"]),
        sum(len(video["gt_intervals"]) for video in cache["videos"]),
    )


def stage_description(kind: str) -> dict:
    if kind == "metst":
        return {
            "locked_table_stage": "official recognition/result-synergy final predictions (full counts)",
            "raw_call_chain": "official.spotting -> sequence_counts_quiet / official sequence evaluator",
            "full_call_chain": "raw counts -> official.recognition(result1_all) -> full_counts_from_official_synergy",
        }
    return {
        "locked_table_stage": "official recognition/result-synergy final predictions (full counts)",
        "raw_call_chain": "official.spotting -> official.sequence_evaluation",
        "full_call_chain": "raw counts -> official.recognition(recognition_sequence) -> full_counts_from_official_synergy",
    }


def run_preflight_setting(name: str, output: Path) -> dict:
    spec = SPECS[name]
    sealed_manifest = verify_sealed_inputs(spec)
    if spec["kind"] == "metst":
        context = metst_context(spec)
        runner, base, _metric, _official, _records, _paths, subjects, native_gate = context
    else:
        context = boosting_context(spec)
        runner, _cache, _grouped, subjects, _metric, _official, native_gate = context
        base = runner
    video_count, gt_count = context_video_and_gt_counts(spec["kind"], context)
    if video_count != spec["videos"]:
        raise RuntimeError(f"{spec['label']}: videos={video_count}, expected {spec['videos']}")
    if gt_count != spec["gt"]:
        raise RuntimeError(f"{spec['label']}: GT={gt_count}, expected {spec['gt']}")
    native_rows, native_raw, native_full = [], np.zeros(3, dtype=np.int64), np.zeros(3, dtype=np.int64)
    for index, subject in enumerate(subjects):
        raw, full = native_subject_counts(spec["kind"], context, index)
        native_raw += raw
        native_full += full
        native_rows.append({"outer_subject": subject, "method": "Native", "raw_TP": raw[0], "raw_FP": raw[1], "raw_FN": raw[2], "full_TP": full[0], "full_FP": full[1], "full_FN": full[2]})
    if tuple(native_raw) != tuple(native_gate["raw_counts"]) or tuple(native_full) != spec["native_full"]:
        raise RuntimeError("Native direct per-subject replay fails its locked aggregate")
    grid = locked_grid(base)
    if len(grid) != 90:
        raise RuntimeError("locked main GLSD grid is not 90 configurations")
    selected = selected_locked_configs(spec["locked_results"] / "outer_selected_configs.csv", grid, subjects)
    glsd = locked_glsd_subject_counts(spec["kind"], context, base, selected, subjects)
    glsd_raw = np.asarray([value[0] for value in glsd.values()]).sum(axis=0)
    glsd_full = np.asarray([value[1] for value in glsd.values()]).sum(axis=0)
    if tuple(glsd_full) != spec["glsd_full"]:
        raise RuntimeError("locked GLSD direct per-subject replay fails its locked aggregate")
    sealed_subject = existing_glsd_subject_counts(spec["run_evidence"] / "per_subject_counts.csv", spec["subjects"])
    if {subject: values[1] for subject, values in glsd.items()} != sealed_subject:
        raise RuntimeError("locked GLSD replay differs from sealed per-subject full counts")
    glsd_rows = [{"outer_subject": subject, "method": "GLSD-90", "raw_TP": raw[0], "raw_FP": raw[1], "raw_FN": raw[2], "full_TP": full[0], "full_FP": full[1], "full_FN": full[2]} for subject, (raw, full) in glsd.items()]
    stage = stage_description(spec["kind"])
    row = {
        "setting": spec["label"], "evaluation_stage_for_locked_table_f1": stage["locked_table_stage"],
        "subjects": len(subjects), "videos": spec["videos"], "GT_events": gt_count,
        "Native_raw_TP": int(native_raw[0]), "Native_raw_FP": int(native_raw[1]), "Native_raw_FN": int(native_raw[2]), "Native_raw_F1": f1(native_raw),
        "Native_TP": int(native_full[0]), "Native_FP": int(native_full[1]), "Native_FN": int(native_full[2]), "Native_replay_F1": f1(native_full), "Native_locked_F1": f1(spec["native_full"]), "Native_locked_reported_F1": spec["native_reported_f1"], "Native_absolute_difference": abs(f1(native_full) - f1(spec["native_full"])),
        "GLSD_raw_TP": int(glsd_raw[0]), "GLSD_raw_FP": int(glsd_raw[1]), "GLSD_raw_FN": int(glsd_raw[2]), "GLSD_raw_F1": f1(glsd_raw),
        "GLSD_TP": int(glsd_full[0]), "GLSD_FP": int(glsd_full[1]), "GLSD_FN": int(glsd_full[2]), "GLSD_replay_F1": f1(glsd_full), "GLSD_locked_F1": f1(spec["glsd_full"]), "GLSD_locked_reported_F1": spec["glsd_reported_f1"], "GLSD_absolute_difference": abs(f1(glsd_full) - f1(spec["glsd_full"])),
    }
    write_csv(output / f"{name}_replay_per_subject.csv", native_rows + glsd_rows)
    write_json(output / f"{name}_preflight_detail.json", {
        "setting": spec["label"], "sealed_manifest": sealed_manifest, "stage": stage,
        "raw_counts_mean": "raw spotting evaluator counts before official recognition/result synergy",
        "full_counts_mean": "same official spot predictions after official recognition/result synergy conversion",
        "table_2_bootstrap_stage": "full counts; the locked Table 2 F1 values equal full-count F1",
        "H_lineage": "Exact matched-budget H formula inherited verbatim from gl_saliency_skill/evidence.py:99; no new normalization choice",
        "score_formulas": {"H": "max(0, (smooth[peak]-mean)/(smooth.max()-mean))", "G": "global prominence / reference smoothed range", "L": "median aligned local saliency / per-scale range, missing scale=0", "GL": "(G+L)/2"},
        "matched_budget": {"modes": list(MODES), "configs_per_mode": 30, "a0": list(SCALES), "tau_source": "locked GLSD ten-value grid", "rho": {"L": RHO, "GL": RHO, "H": "not searched", "G": "not searched"}},
        "summary": row,
    })
    return row


def run_preflight(settings: list[str], root: Path) -> None:
    root.mkdir(parents=True, exist_ok=False)
    rows = [run_preflight_setting(name, root) for name in settings]
    if any(row["Native_absolute_difference"] > 1e-12 or row["GLSD_absolute_difference"] > 1e-12 for row in rows):
        raise RuntimeError("locked-result replay tolerance failed")
    write_csv(root / "locked_replay_summary.csv", rows)
    write_json(root / "preflight_manifest.json", {"preflight_only": True, "ablation_search_started": False, "bootstrap_started": False, "settings": settings, "stage": "full counts / official recognition-result-synergy final predictions"})
    audit_lines = [
        "# Official-response locked-result preflight",
        "",
        "This run replayed sealed Native and locked GLSD-90 selections only. It did not construct a component-score grid, run H/G/L/GL selection, or run bootstrap.",
        "",
        "## Evaluation stage",
        "",
        "All four Table-2 locked F1 values are **full-count F1**: official spotting is evaluated first to form raw spotting counts, then the same spotting predictions pass through the locked official recognition/result-synergy call and `full_counts_from_official_synergy`. Bootstrap and any subsequent H/G/L/GL comparison must use these full counts.",
        "",
        "- ME-TST+: `official.spotting -> sequence_counts_quiet` (raw), then `official.recognition(result1_all) -> full_counts_from_official_synergy` (full).",
        "- BoostingVRME: `official.spotting -> official.sequence_evaluation` (raw), then `official.recognition(recognition_sequence) -> full_counts_from_official_synergy` (full).",
        "",
        "## Locked replay totals",
        "",
        "| Setting | Subjects | Videos | GT | Native full TP/FP/FN | Native replay / locked F1 | GLSD-90 full TP/FP/FN | GLSD-90 replay / locked F1 | Absolute differences |",
        "|---|---:|---:|---:|---|---|---|---|---:|",
    ]
    for row in rows:
        audit_lines.append(
            f"| {row['setting']} | {row['subjects']} | {row['videos']} | {row['GT_events']} | "
            f"{row['Native_TP']}/{row['Native_FP']}/{row['Native_FN']} | {row['Native_replay_F1']:.12f} / {row['Native_locked_reported_F1']} | "
            f"{row['GLSD_TP']}/{row['GLSD_FP']}/{row['GLSD_FN']} | {row['GLSD_replay_F1']:.12f} / {row['GLSD_locked_reported_F1']} | "
            f"Native={row['Native_absolute_difference']:.1e}; GLSD={row['GLSD_absolute_difference']:.1e} |"
        )
    audit_lines.extend([
        "",
        "## Future matched-budget component scores (not executed in this preflight)",
        "",
        "`H` directly inherits the exact prior matched-budget normalized reference-peak formula in `make_component_features()` (source lineage: `gl_saliency_skill/evidence.py:99`): `max(0, (smooth[peak] - mean) / max(smooth.max() - mean, 1e-12))`. `G` is the locked global normalized prominence, `L` is aligned multi-scale local saliency, and `GL` is the fixed `(G+L)/2`.",
        "",
        "Each future mode has exactly 30 configurations: 3 `a0` values × the locked 10-value `tau` grid. `rho=2` is fixed only for L/GL; H/G have no rho parameter. The future output label is `GL` / `matched-budget-GL-30`, never `GLSD-90`.",
        "",
    ])
    (root / "preflight_stage_audit.md").write_text("\n".join(audit_lines), encoding="utf-8")
    print("OFFICIAL_RESPONSE_PREFLIGHT = PASS")
    print("PREFLIGHT_OUTPUT =", root)


def run_setting(name: str, root: Path) -> None:
    spec = SPECS[name]
    sealed_manifest = verify_sealed_inputs(spec)
    output = root / name
    if output.exists():
        raise RuntimeError(f"refusing to overwrite: {output}")
    output.mkdir(parents=True)
    print(f"=== {spec['label']} ===")
    print("sealed input preflight = PASS")
    if spec["kind"] == "metst":
        context = metst_context(spec)
        runner, base, _metric, _official, _records, _paths, subjects, native_gate = context
        thresholds = base.configuration_grid()[0:10]
        thresholds = [cfg.threshold for cfg in thresholds]
    else:
        context = boosting_context(spec)
        runner, _cache, _grouped, subjects, _metric, _official, native_gate = context
        base = runner
        thresholds = [cfg.threshold for cfg in runner.configuration_grid()[0:10]] if hasattr(runner, "configuration_grid") else list(runner.THRESHOLDS)
    if tuple(native_gate["full_counts"]) != spec["native_full"]:
        raise RuntimeError("native exact replay does not equal sealed final result")
    grid = component_grid(thresholds)
    if len(grid) != 30:
        raise RuntimeError("matched budget is not 30 configurations")
    native_subject = {}
    for index, subject in enumerate(subjects):
        raw, full = native_subject_counts(spec["kind"], context, index)
        native_subject[subject] = full
    if tuple(np.asarray(list(native_subject.values())).sum(axis=0)) != spec["native_full"]:
        raise RuntimeError("per-subject Native recount fails sealed aggregate")
    locked_glsd_subject = existing_glsd_subject_counts(spec["run_evidence"] / "per_subject_counts.csv", spec["subjects"])
    if tuple(np.asarray(list(locked_glsd_subject.values())).sum(axis=0)) != spec["glsd_full"]:
        raise RuntimeError("sealed GLSD per-subject table fails locked aggregate")
    bootstrap(spec["label"], native_subject, locked_glsd_subject, output)
    summary, outer_rows, counts_rows, selections = [], [], [], {}
    search_rows, inner_rows = [], []
    for mode in MODES:
        print(f"mode={MODE_LABELS[mode]}: search {len(grid)} configs x {len(subjects)} outer subjects")
        core = ComponentCore(base, mode, grid)
        raw_table = np.zeros((len(grid), len(subjects), 3), dtype=np.int64)
        for subject_index in range(len(subjects)):
            for config_id, config in enumerate(grid):
                raw, _ = decode_component(spec["kind"], context, core, subject_index, config, False)
                raw_table[config_id, subject_index] = raw
        selected, traces, folds = outer_selection(raw_table, subjects, core.selection_order)
        selections[mode] = {subject: {"config_id": cid, **asdict(grid[cid])} for subject, cid in selected.items()}
        for outer, subject in enumerate(subjects):
            inner = [index for index in range(len(subjects)) if index != outer]
            pooled = raw_table[:, inner, :].sum(axis=1)
            winner = selected[subject]
            for config_id, config in enumerate(grid):
                row_metrics = metrics(pooled[config_id])
                search_rows.append({
                    "setting": spec["label"], "score_mode": mode, "score_mode_label": MODE_LABELS[mode], "outer_subject": subject,
                    "config_id": config_id, "a0": config.reference_scale, "rho": RHO if mode in ("L", "GL") else "not searched",
                    "tau": config.threshold, "pooled_inner_TP": row_metrics["TP"], "pooled_inner_FP": row_metrics["FP"],
                    "pooled_inner_FN": row_metrics["FN"], "precision": row_metrics["precision"], "recall": row_metrics["recall"],
                    "F1": row_metrics["F1"], "grid_order": config_id, "selected": int(config_id == winner),
                })
                for inner_index in inner:
                    values = raw_table[config_id, inner_index]
                    inner_rows.append({
                        "setting": spec["label"], "score_mode": mode, "score_mode_label": MODE_LABELS[mode], "outer_subject": subject,
                        "inner_validation_subject": subjects[inner_index], "config_id": config_id,
                        "a0": config.reference_scale, "rho": RHO if mode in ("L", "GL") else "not searched", "tau": config.threshold,
                        "TP": int(values[0]), "FP": int(values[1]), "FN": int(values[2]),
                    })
        total = np.zeros(3, dtype=np.int64)
        for subject_index, subject, inner, config_id, inner_counts in folds:
            raw, full = decode_component(spec["kind"], context, core, subject_index, grid[config_id], True)
            if raw != tuple(raw_table[config_id, subject_index]):
                raise RuntimeError("selected outer subject does not replay its pre-selection count")
            total += raw
            outer_rows.append({
                "setting": spec["label"], "score_mode": mode, "score_mode_label": MODE_LABELS[mode], "outer_subject": subject,
                "config_id": config_id, "selected_a0": grid[config_id].reference_scale,
                "fixed_rho": RHO if mode in ("L", "GL") else "not searched",
                "selected_tau": grid[config_id].threshold,
                "inner_subject_count": len(inner), "inner_TP": int(inner_counts[0]),
                "inner_FP": int(inner_counts[1]), "inner_FN": int(inner_counts[2]),
                "raw_TP": raw[0], "raw_FP": raw[1], "raw_FN": raw[2],
                "full_TP": full[0], "full_FP": full[1], "full_FN": full[2],
            })
            counts_rows.append({"setting": spec["label"], "score_mode": mode, "score_mode_label": MODE_LABELS[mode], "outer_subject": subject, "TP": raw[0], "FP": raw[1], "FN": raw[2], "full_TP": full[0], "full_FP": full[1], "full_FN": full[2]})
        m = metrics(total)
        summary.append({"setting": spec["label"], "score_mode": mode, "score_mode_label": MODE_LABELS[mode], **m})
        trace_name = "selection_trace_matched_budget_GL_30.json" if mode == "GL" else f"selection_trace_{mode}.json"
        write_json(output / trace_name, traces)
    write_csv(output / "ablation_summary.csv", summary)
    write_csv(output / "ablation_outer_folds.csv", outer_rows)
    write_json(output / "ablation_selected_configs.json", selections)
    write_csv(output / "ablation_per_subject_counts.csv", counts_rows)
    write_csv(output / "ablation_search_all.csv", search_rows)
    write_csv(output / "ablation_inner_fold_counts.csv", inner_rows)
    write_json(output / "ablation_manifest.json", {
        "setting": spec["label"], "created_utc": datetime.now(timezone.utc).isoformat(),
        "input_mode": "sealed official-response evidence only", "historical_controlled_cache_used": False,
        "source": str(spec["source"]), "source_sha256": sha256(spec["source"]),
        "sealed_evidence": str(spec["evidence"]), "sealed_manifest": sealed_manifest,
        "score_modes": {"H": "normalized reference peak-height amplitude", "G": "global normalized prominence", "L": "aligned multi-scale local saliency", "GL": "matched-budget-GL-30: fixed (G+L)/2; not locked GLSD-90"},
        "grid": {"a0": list(SCALES), "tau": list(map(float, thresholds)), "rho": {"L": RHO, "GL": RHO, "H": "not searched", "G": "not searched"}},
        "configurations_per_score_mode": 30, "selection": "outer LOSO; F1 -> precision -> fewer FP -> fixed grid order",
        "invariants": ["official response read only", "official candidate geometry and post-processing reused", "official evaluator reused", "no backbone", "no response generation", "no locked-evidence writes"],
    })
    print(f"{spec['label']}: ablation + paired bootstrap = PASS")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--setting", choices=(*SPECS, "all"), default="all")
    parser.add_argument("--output-root", type=Path, default=DRIVE / "official_response_ablation")
    parser.add_argument(
        "--preflight-only",
        action="store_true",
        help="replay locked Native/GLSD-90 only; do not start ablation or bootstrap",
    )
    args = parser.parse_args()
    prefix = "preflight" if args.preflight_only else "ablation"
    run_id = datetime.now(timezone.utc).strftime(prefix + "_%Y%m%dT%H%M%SZ")
    root = args.output_root / run_id
    if root.exists():
        raise RuntimeError(f"refusing to overwrite: {root}")
    settings = SPECS if args.setting == "all" else {args.setting: SPECS[args.setting]}
    if args.preflight_only:
        run_preflight(list(settings), root)
        return
    root.mkdir(parents=True)
    for setting in settings:
        run_setting(setting, root)
    print("OFFICIAL_RESPONSE_COMPONENT_ABLATION = PASS")
    print("OUTPUT_ROOT =", root)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/transfer_bundle/boosting_casme3_official_response_preflight.py
#!/usr/bin/env python3
"""Fail-closed functional preflight for BoostingVRME / CAS(ME)3 GLSD-90.

This phase-one adapter uses only APIs present in
``boosting_official_glds_full_casme3.py``. It does not run component
ablation or bootstrap. It writes recovery evidence only after every locked
functional replay and sealed-evidence comparison passes.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import importlib.util
import inspect
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Iterator

import numpy as np


DEFAULT_RUNNER = Path("/content/BoostingVRME/boosting_official_glds_full_casme3.py")
DEFAULT_OFFICIAL_REPO_ROOT = Path("/content/BoostingVRME")
DEFAULT_CACHE = Path(
    "/content/drive/MyDrive/GLSD_BoostingVRME_Full/CASME3/"
    "official_response_cache/casme3_official_full_responses.pkl"
)
DEFAULT_EVIDENCE_DIR = Path(
    "/content/drive/MyDrive/GLSD_BoostingVRME_Full/CASME3/glsd_full_evidence_v1"
)

EXPECTED_SUBJECTS = 94
EXPECTED_VIDEOS = 462
EXPECTED_GT = 853
FORBIDDEN_CONTROLLED_GT = 858
EXPECTED_CONFIGS = 90
EXPECTED_OUTER_FOLDS = 94
EXPECTED_INNER_SUBJECTS = 93
EXPECTED_INNER_ROWS = 94 * 90 * 93
EXPECTED_SEARCH_ROWS = 94 * 90
EXPECTED_NATIVE_RAW = (84, 793, 769)
EXPECTED_NATIVE_FULL = (84, 786, 769)
EXPECTED_GLSD_RAW = (120, 1113, 733)
EXPECTED_GLSD_FULL = (120, 1110, 733)

INNER_FIELDS = [
    "outer_subject_index", "outer_subject", "config_id", "reference_scale",
    "local_radius", "threshold", "inner_subject_index", "inner_subject",
    "tp", "fp", "fn",
]
SEARCH_FIELDS = [
    "outer_subject_index", "outer_subject", "config_id", "reference_scale",
    "local_radius", "threshold", "grid_order", "pooled_inner_tp",
    "pooled_inner_fp", "pooled_inner_fn", "precision", "recall", "f1",
    "rank", "final_rank", "selected",
]
SELECTED_FIELDS = [
    "outer_subject_index", "outer_subject", "inner_subject_count",
    "inner_subject_indices", "inner_subjects", "config_id", "reference_scale",
    "local_radius", "threshold", "grid_order", "pooled_inner_tp",
    "pooled_inner_fp", "pooled_inner_fn", "precision", "recall", "f1", "rank",
]
SUBJECT_FIELDS = [
    "outer_subject_index", "outer_subject", "config_id", "reference_scale",
    "local_radius", "threshold", "raw_tp", "raw_fp", "raw_fn",
    "raw_precision", "raw_recall", "raw_f1", "full_tp", "full_fp",
    "full_fn", "full_precision", "full_recall", "full_f1",
]
VIDEO_FIELDS = [
    "subject_index", "subject", "video_index_within_subject",
    "video_index_global", "video", "config_id", "gt_count", "prediction_count",
    "raw_tp", "raw_fp", "raw_fn", "neutral_tp", "neutral_fp", "full_tp",
    "full_fp", "full_fn",
]
PREDICTION_FIELDS = [
    "subject_index", "subject", "video_index_within_subject",
    "video_index_global", "video", "config_id", "prediction_index", "onset",
    "offset", "peak", "matched_gt_index", "matched_gt", "best_iou_inclusive",
    "matched_iou_inclusive", "raw_status", "recognition_prediction",
    "recognition_target", "neutral", "synergy_action",
]


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def require_file(path: Path, label: str) -> None:
    if not path.is_file():
        raise RuntimeError(f"missing locked {label}: {path}")


def as_counts(value, label: str) -> tuple[int, int, int]:
    if isinstance(value, dict):
        lowered = {str(key).lower(): item for key, item in value.items()}
        try:
            value = (lowered["tp"], lowered["fp"], lowered["fn"])
        except KeyError as exc:
            raise RuntimeError(f"{label} is not a TP/FP/FN mapping") from exc
    result = tuple(int(item) for item in value)
    if len(result) != 3:
        raise RuntimeError(f"{label} must contain TP/FP/FN, got {result}")
    return result


def require_equal(actual, expected, label: str) -> None:
    if actual != expected:
        raise RuntimeError(f"{label}: actual={actual!r}, expected={expected!r}")


def require_signature(function, parameters: tuple[str, ...], label: str) -> None:
    actual = tuple(inspect.signature(function).parameters)
    require_equal(actual, parameters, f"runner API signature {label}")


def import_runner(path: Path, official_repo_root: Path):
    spec = importlib.util.spec_from_file_location("locked_boosting_casme3_runner", path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"cannot import runner: {path}")
    module = importlib.util.module_from_spec(spec)
    old_path = list(sys.path)
    sys.path[:0] = [str(official_repo_root), str(path.parent)]
    sys.modules[spec.name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(spec.name, None)
        raise
    finally:
        sys.path[:] = old_path
    return module


def validate_real_runner_api(runner) -> None:
    required = {
        "load_cache": ("path",),
        "group_videos": ("videos",),
        "native_exact_replay": ("cache", "grouped"),
        "build_grid": (),
        "decode_subject": (
            "subject_index", "config_item", "grouped", "feature_bank",
            "final_samples", "final_emotions", "cache_config", "need_recognition",
        ),
        "subject_prediction_evidence": (
            "subject_index", "subject_name", "config_item", "decoded", "grouped",
            "final_samples", "cache_config",
        ),
        "metrics": ("counts",),
        "add_counts": ("items",),
        "official_metric_counts": ("metric", "total_gt"),
        "normalize": ("obj",),
        "patched_candidate_source": ("selected_per_video",),
    }
    for name, parameters in required.items():
        function = getattr(runner, name, None)
        if not callable(function):
            raise RuntimeError(f"real runner API is missing callable {name}")
        require_signature(function, parameters, name)

    feature_class = getattr(runner, "GLSDFeatures", None)
    if not inspect.isclass(feature_class):
        raise RuntimeError("real runner API is missing GLSDFeatures")
    require_signature(feature_class, ("response", "k"), "GLSDFeatures")
    require_signature(
        feature_class.local_values, ("self", "width", "rho"),
        "GLSDFeatures.local_values",
    )
    require_signature(
        feature_class.scores, ("self", "reference_scale", "rho"),
        "GLSDFeatures.scores",
    )
    require_signature(
        feature_class.selected_peaks,
        ("self", "reference_scale", "rho", "tau"),
        "GLSDFeatures.selected_peaks",
    )

def validate_manifest(manifest_path: Path, cache_sha: str, runner_sha: str) -> dict:
    manifest = load_json(manifest_path)
    if not isinstance(manifest, dict):
        raise RuntimeError("run_manifest.json must contain a JSON object")

    exact = {
        "dataset": "CASME_3",
        "subject_count": EXPECTED_SUBJECTS,
        "video_count": EXPECTED_VIDEOS,
        "gt_event_count": EXPECTED_GT,
        "glsd_grid_size": EXPECTED_CONFIGS,
        "inner_evidence_rows": EXPECTED_INNER_ROWS,
        "glsd_search_rows": EXPECTED_SEARCH_ROWS,
    }
    for key, expected in exact.items():
        if key not in manifest:
            raise RuntimeError(f"run_manifest.json lacks required field {key}")
        require_equal(manifest[key], expected, f"manifest {key}")

    require_equal(
        manifest.get("selection_tie_break"),
        ["higher_F1", "higher_precision", "fewer_FP", "fixed_grid_order"],
        "manifest selection_tie_break",
    )
    require_equal(
        manifest.get("glsd_grid_constants"),
        {
            "reference_scales": [1.0, 1.5, 2.0],
            "local_radii": [1.0, 2.0, 3.0],
            "thresholds": [0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.75],
        },
        "manifest glsd_grid_constants",
    )
    require_equal(
        as_counts(manifest.get("native_expected_counts", {}).get("raw", ()),
                  "manifest native raw"),
        EXPECTED_NATIVE_RAW,
        "manifest Native raw counts",
    )
    require_equal(
        as_counts(manifest.get("native_expected_counts", {}).get("full", ()),
                  "manifest native full"),
        EXPECTED_NATIVE_FULL,
        "manifest Native full counts",
    )
    require_equal(
        as_counts(manifest.get("glsd_final_raw_counts", ()), "manifest GLSD raw"),
        EXPECTED_GLSD_RAW,
        "manifest GLSD raw counts",
    )
    require_equal(
        as_counts(manifest.get("glsd_final_full_counts", ()), "manifest GLSD full"),
        EXPECTED_GLSD_FULL,
        "manifest GLSD full counts",
    )

    historical_cache_sha = str(manifest.get("cache_sha256", "")).lower()
    if not historical_cache_sha:
        raise RuntimeError("run_manifest.json lacks cache_sha256")
    if cache_sha.lower() != historical_cache_sha:
        raise RuntimeError(
            "HARD FAIL: cache_sha256 mismatch: "
            f"current={cache_sha}, historical={historical_cache_sha}"
        )

    historical_core_sha = str(manifest.get("core_script_sha256", "")).lower()
    if not historical_core_sha:
        raise RuntimeError("run_manifest.json lacks core_script_sha256")
    historical_wrapper_sha = str(manifest.get("script_sha256", "")).lower()
    if not historical_wrapper_sha:
        raise RuntimeError("run_manifest.json lacks historical wrapper script_sha256")

    return {
        "manifest": manifest,
        "historical_cache_sha256": historical_cache_sha,
        "historical_core_runner_sha256": historical_core_sha,
        "historical_execution_wrapper_sha256": historical_wrapper_sha,
        "current_runner_sha256": runner_sha.lower(),
        "runner_sha_match": runner_sha.lower() == historical_core_sha,
    }


def validate_universe(cache: dict, videos: list[dict], grouped: list[list[dict]]):
    require_equal(cache.get("dataset"), "CASME_3", "cache dataset")
    require_equal(int(cache.get("subject_count", -1)), EXPECTED_SUBJECTS,
                  "cache subject_count")
    require_equal(int(cache.get("video_count", -1)), EXPECTED_VIDEOS,
                  "cache video_count")
    require_equal(len(videos), EXPECTED_VIDEOS, "loaded video count")
    require_equal(len(grouped), EXPECTED_SUBJECTS, "grouped subject count")
    require_equal(sum(len(subject) for subject in grouped), EXPECTED_VIDEOS,
                  "grouped video count")

    global_indices = [int(video["video_index_global"]) for video in videos]
    require_equal(global_indices, list(range(EXPECTED_VIDEOS)),
                  "global video ordering")
    subject_names = []
    for subject_index, subject in enumerate(grouped):
        if not subject:
            raise RuntimeError(f"empty official subject bucket {subject_index}")
        indices = {int(video["subject_index"]) for video in subject}
        require_equal(indices, {subject_index}, f"subject bucket {subject_index}")
        names = {str(video["subject"]) for video in subject}
        if len(names) != 1:
            raise RuntimeError(f"subject bucket {subject_index} has inconsistent names")
        subject_names.append(str(subject[0]["subject"]))
    require_equal(len(set(subject_names)), EXPECTED_SUBJECTS,
                  "unique official subject names")

    gt_count = sum(len(video["gt_intervals"]) for video in videos)
    if gt_count == FORBIDDEN_CONTROLLED_GT:
        raise RuntimeError("HARD FAIL: forbidden 858-event controlled cache detected")
    require_equal(gt_count, EXPECTED_GT, "official GT count")
    return subject_names, gt_count


def validate_native(runner, cache: dict, grouped: list[list[dict]]) -> dict:
    native = runner.native_exact_replay(cache, grouped)
    if not isinstance(native, dict):
        raise RuntimeError("native_exact_replay did not return its audit dictionary")
    required = {
        "passed", "raw_counts", "full_counts", "prediction_exact",
        "recognition_exact", "matched_gt_exact",
    }
    missing = sorted(required.difference(native))
    if missing:
        raise RuntimeError(f"native_exact_replay result lacks fields {missing}")
    for key in ("passed", "prediction_exact", "recognition_exact", "matched_gt_exact"):
        if native[key] is not True:
            raise RuntimeError(f"Native exact replay gate {key}=False")
    require_equal(as_counts(native["raw_counts"], "Native raw"), EXPECTED_NATIVE_RAW,
                  "Native raw counts")
    require_equal(as_counts(native["full_counts"], "Native full"), EXPECTED_NATIVE_FULL,
                  "Native full counts")
    return native


def build_main_state(runner, cache: dict, grouped: list[list[dict]]):
    cache_config = cache["config"]
    k = int(cache_config["k_p"])
    if k < 1:
        raise RuntimeError(f"invalid official cache k_p={k}")
    final_samples = [
        [video["gt_intervals"] for video in subject] for subject in grouped
    ]
    final_emotions = [
        [video["emotion_labels"] for video in subject] for subject in grouped
    ]
    feature_bank = [
        [runner.GLSDFeatures(video["spot_response"], k) for video in subject]
        for subject in grouped
    ]
    return cache_config, final_samples, final_emotions, feature_bank


def replay_subject_config_table(
    runner, grid: list[dict], grouped: list[list[dict]], feature_bank,
    final_samples, final_emotions, cache_config,
) -> np.ndarray:
    table = np.empty((EXPECTED_CONFIGS, EXPECTED_SUBJECTS, 3), dtype=np.int64)
    for config_item in grid:
        config_id = int(config_item["config_id"])
        for subject_index in range(EXPECTED_SUBJECTS):
            decoded = runner.decode_subject(
                subject_index, config_item, grouped, feature_bank, final_samples,
                final_emotions, cache_config, need_recognition=False,
            )
            table[config_id, subject_index] = as_counts(
                decoded["raw_counts"],
                f"raw config={config_id} subject={subject_index}",
            )
        print(f"functional raw replay config {config_id + 1:02d}/90")
    require_equal(tuple(table.shape), (90, 94, 3), "subject/config table shape")
    return table


def inner_rows(
    subject_names: list[str], grid: list[dict], table: np.ndarray,
) -> Iterator[dict]:
    for outer in range(EXPECTED_OUTER_FOLDS):
        for config_item in grid:
            config_id = int(config_item["config_id"])
            for inner in range(EXPECTED_SUBJECTS):
                if inner == outer:
                    continue
                tp, fp, fn = (int(value) for value in table[config_id, inner])
                yield {
                    "outer_subject_index": outer,
                    "outer_subject": subject_names[outer],
                    "config_id": config_id,
                    "reference_scale": config_item["reference_scale"],
                    "local_radius": config_item["local_radius"],
                    "threshold": config_item["threshold"],
                    "inner_subject_index": inner,
                    "inner_subject": subject_names[inner],
                    "tp": tp,
                    "fp": fp,
                    "fn": fn,
                }


def rebuild_outer_selection(runner, subject_names, grid, table):
    search_rows = []
    selected_rows = []
    selection_trace = []
    selected_by_outer = {}

    for outer in range(EXPECTED_OUTER_FOLDS):
        inner_subjects = [index for index in range(EXPECTED_SUBJECTS) if index != outer]
        require_equal(len(inner_subjects), EXPECTED_INNER_SUBJECTS,
                      f"outer {outer} inner subject count")
        if outer in inner_subjects:
            raise RuntimeError(f"outer subject {outer} leaked into inner LOSO")

        candidates = []
        for config_item in grid:
            config_id = int(config_item["config_id"])
            pooled = runner.add_counts(
                [tuple(table[config_id, inner]) for inner in inner_subjects]
            )
            metric = runner.metrics(pooled)
            candidates.append({
                **config_item,
                "pooled_inner_tp": metric["tp"],
                "pooled_inner_fp": metric["fp"],
                "pooled_inner_fn": metric["fn"],
                "precision": metric["precision"],
                "recall": metric["recall"],
                "f1": metric["f1"],
            })

        ranked = sorted(
            candidates,
            key=lambda row: (
                -float(row["f1"]), -float(row["precision"]),
                int(row["pooled_inner_fp"]), int(row["grid_order"]),
            ),
        )
        for rank, row in enumerate(ranked, start=1):
            row["rank"] = rank
        winner = ranked[0]
        selected_by_outer[outer] = dict(winner)
        rank_lookup = {
            int(row["config_id"]): int(row["rank"]) for row in ranked
        }

        for row in candidates:
            config_id = int(row["config_id"])
            search_rows.append({
                "outer_subject_index": outer,
                "outer_subject": subject_names[outer],
                **row,
                "final_rank": rank_lookup[config_id],
                "selected": int(config_id == int(winner["config_id"])),
            })

        selected_rows.append({
            "outer_subject_index": outer,
            "outer_subject": subject_names[outer],
            "inner_subject_count": len(inner_subjects),
            "inner_subject_indices": json.dumps(inner_subjects),
            "inner_subjects": json.dumps([subject_names[i] for i in inner_subjects]),
            **winner,
        })
        selection_trace.append({
            "outer_subject_index": outer,
            "outer_subject": subject_names[outer],
            "inner_subject_indices": inner_subjects,
            "inner_subjects": [subject_names[i] for i in inner_subjects],
            "tie_break": [
                "higher_F1", "higher_precision", "fewer_FP", "fixed_grid_order",
            ],
            "winner_config_id": int(winner["config_id"]),
            "winner_rank": int(winner["rank"]),
            "candidates": ranked,
        })

    require_equal(len(search_rows), EXPECTED_SEARCH_ROWS, "GLSD search rows")
    require_equal(len(selected_rows), EXPECTED_OUTER_FOLDS, "selected outer rows")
    require_equal(len(selection_trace), EXPECTED_OUTER_FOLDS, "selection trace folds")
    return search_rows, selected_rows, selection_trace, selected_by_outer


def csv_cell(value) -> str:
    if value is None:
        return ""
    return str(value)


def compare_csv_exact(
    path: Path, expected_rows: Iterable[dict], fields: list[str], label: str,
) -> dict:
    require_file(path, label)
    count = 0
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        require_equal(reader.fieldnames, fields, f"{label} header")
        for count, expected in enumerate(expected_rows, start=1):
            try:
                actual = next(reader)
            except StopIteration as exc:
                raise RuntimeError(f"{label} ended before row {count}") from exc
            expected_text = {field: csv_cell(expected[field]) for field in fields}
            if actual != expected_text:
                differences = {
                    field: (actual.get(field), expected_text[field])
                    for field in fields if actual.get(field) != expected_text[field]
                }
                raise RuntimeError(
                    f"{label} row {count} mismatch: {differences}"
                )
        try:
            extra = next(reader)
        except StopIteration:
            extra = None
        if extra is not None:
            raise RuntimeError(f"{label} has extra row after {count}: {extra}")
    return {
        "path": str(path.resolve()), "sha256": sha256_file(path),
        "rows": count, "fields": fields, "exact_match": True,
    }


def compare_json_exact(path: Path, expected, label: str, runner) -> dict:
    require_file(path, label)
    actual = load_json(path)
    normalized_expected = runner.normalize(expected)
    if actual != normalized_expected:
        raise RuntimeError(f"{label} differs from functional replay")
    return {
        "path": str(path.resolve()), "sha256": sha256_file(path),
        "exact_match": True,
    }


def replay_selected_outer(
    runner, subject_names, grouped, feature_bank, final_samples, final_emotions,
    cache_config, selected_by_outer, table,
):
    per_subject_rows = []
    per_video_rows = []
    per_prediction_rows = []
    outer_raw_counts = []
    outer_full_counts = []

    for outer in range(EXPECTED_OUTER_FOLDS):
        config_item = selected_by_outer[outer]
        decoded = runner.decode_subject(
            outer, config_item, grouped, feature_bank, final_samples,
            final_emotions, cache_config, need_recognition=True,
        )
        raw = as_counts(decoded["raw_counts"], f"outer {outer} raw")
        full = as_counts(decoded["full_counts"], f"outer {outer} full")
        config_id = int(config_item["config_id"])
        require_equal(raw, tuple(int(x) for x in table[config_id, outer]),
                      f"outer {outer} selected raw replay")

        prediction_rows, video_rows, _video_records = runner.subject_prediction_evidence(
            outer, subject_names[outer], config_item, decoded, grouped,
            final_samples, cache_config,
        )
        video_raw = (
            sum(int(row["raw_tp"]) for row in video_rows),
            sum(int(row["raw_fp"]) for row in video_rows),
            sum(int(row["raw_fn"]) for row in video_rows),
        )
        video_full = (
            sum(int(row["full_tp"]) for row in video_rows),
            sum(int(row["full_fp"]) for row in video_rows),
            sum(int(row["full_fn"]) for row in video_rows),
        )
        require_equal(video_raw, raw, f"outer {outer} independent raw recount")
        require_equal(video_full, full, f"outer {outer} independent full recount")

        raw_metric = runner.metrics(raw)
        full_metric = runner.metrics(full)
        per_subject_rows.append({
            "outer_subject_index": outer,
            "outer_subject": subject_names[outer],
            "config_id": config_id,
            "reference_scale": config_item["reference_scale"],
            "local_radius": config_item["local_radius"],
            "threshold": config_item["threshold"],
            "raw_tp": raw_metric["tp"],
            "raw_fp": raw_metric["fp"],
            "raw_fn": raw_metric["fn"],
            "raw_precision": raw_metric["precision"],
            "raw_recall": raw_metric["recall"],
            "raw_f1": raw_metric["f1"],
            "full_tp": full_metric["tp"],
            "full_fp": full_metric["fp"],
            "full_fn": full_metric["fn"],
            "full_precision": full_metric["precision"],
            "full_recall": full_metric["recall"],
            "full_f1": full_metric["f1"],
        })
        per_video_rows.extend(video_rows)
        per_prediction_rows.extend(prediction_rows)
        outer_raw_counts.append(raw)
        outer_full_counts.append(full)

    require_equal(len(per_subject_rows), EXPECTED_SUBJECTS, "per-subject replay rows")
    require_equal(len(per_video_rows), EXPECTED_VIDEOS, "per-video replay rows")
    final_raw = as_counts(runner.add_counts(outer_raw_counts), "GLSD aggregate raw")
    final_full = as_counts(runner.add_counts(outer_full_counts), "GLSD aggregate full")
    require_equal(final_raw, EXPECTED_GLSD_RAW, "GLSD-90 raw counts")
    require_equal(final_full, EXPECTED_GLSD_FULL, "GLSD-90 full counts")
    return (
        per_subject_rows, per_video_rows, per_prediction_rows, final_raw, final_full,
    )


def validate_grid(runner) -> list[dict]:
    grid = runner.build_grid()
    if not isinstance(grid, list):
        raise RuntimeError("build_grid() did not return a list")
    require_equal(len(grid), EXPECTED_CONFIGS, "configuration count")
    fields = {"config_id", "reference_scale", "local_radius", "threshold", "grid_order"}
    for index, config_item in enumerate(grid):
        if not isinstance(config_item, dict):
            raise RuntimeError(f"grid config {index} is not a dict")
        require_equal(set(config_item), fields, f"grid config {index} fields")
        require_equal(int(config_item["config_id"]), index, f"grid config {index} id")
        require_equal(int(config_item["grid_order"]), index,
                      f"grid config {index} order")
    return grid


def write_recovery_outputs(
    output: Path, recovery: dict, table: np.ndarray, selected_rows: list[dict],
    per_subject_rows: list[dict], runner,
) -> None:
    output.mkdir(parents=True, exist_ok=False)
    table_path = output / "functional_glsd90_subject_config_raw_counts.npy"
    selected_path = output / "functional_outer_selected_configs.json"
    subject_path = output / "functional_per_subject_counts.csv"
    manifest_path = output / "recovery_manifest.json"

    np.save(table_path, table)
    selected_path.write_text(
        json.dumps(runner.normalize(selected_rows), indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )
    with subject_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=SUBJECT_FIELDS)
        writer.writeheader()
        writer.writerows(per_subject_rows)

    recovery["recovery_artifacts"] = {
        table_path.name: {
            "sha256": sha256_file(table_path), "shape": list(table.shape),
            "dtype": str(table.dtype),
        },
        selected_path.name: {
            "sha256": sha256_file(selected_path), "rows": len(selected_rows),
        },
        subject_path.name: {
            "sha256": sha256_file(subject_path), "rows": len(per_subject_rows),
        },
    }
    manifest_path.write_text(
        json.dumps(recovery, indent=2, ensure_ascii=False) + "\n",
        encoding="utf-8",
    )


def run_preflight(args) -> None:
    runner_path = args.runner.resolve()
    repo_root = args.official_repo_root.resolve()
    cache_path = args.cache.resolve()
    evidence_dir = args.evidence_dir.resolve()

    required = {
        "runner": runner_path,
        "cache": cache_path,
        "manifest": evidence_dir / "run_manifest.json",
        "per_subject": evidence_dir / "per_subject_counts.csv",
        "selected_json": evidence_dir / "outer_selected_configs.json",
        "trace": evidence_dir / "outer_selection_trace.json",
        "inner": evidence_dir / "glsd_search_inner_fold_counts.csv",
    }
    for label in ("runner", "cache", "manifest"):
        require_file(required[label], label)
    if not repo_root.is_dir():
        raise RuntimeError(f"official repository root is not a directory: {repo_root}")

    runner_sha = sha256_file(runner_path)
    cache_sha = sha256_file(cache_path)
    provenance = validate_manifest(required["manifest"], cache_sha, runner_sha)
    print(f"cache path = {cache_path}")
    print(f"cache SHA256 = {cache_sha}")
    print(
        "historical core runner SHA256 = "
        f"{provenance['historical_core_runner_sha256']}"
    )
    print(f"current runner SHA256 = {runner_sha}")
    if not provenance["runner_sha_match"]:
        print("runner SHA mismatch: continuing complete functional exact replay")

    for label in ("per_subject", "selected_json", "trace", "inner"):
        require_file(required[label], label)

    # A runner SHA mismatch is evidence, not a bypass: the complete replay below still runs.
    runner = import_runner(runner_path, repo_root)
    validate_real_runner_api(runner)
    cache, videos = runner.load_cache(cache_path)
    grouped = runner.group_videos(videos)
    subject_names, gt_count = validate_universe(cache, videos, grouped)
    native = validate_native(runner, cache, grouped)
    grid = validate_grid(runner)
    cache_config, final_samples, final_emotions, feature_bank = build_main_state(
        runner, cache, grouped
    )
    table = replay_subject_config_table(
        runner, grid, grouped, feature_bank, final_samples, final_emotions,
        cache_config,
    )

    evidence_checks = {}
    evidence_checks["inner_fold_counts"] = compare_csv_exact(
        required["inner"], inner_rows(subject_names, grid, table), INNER_FIELDS,
        "glsd_search_inner_fold_counts.csv",
    )
    require_equal(evidence_checks["inner_fold_counts"]["rows"], EXPECTED_INNER_ROWS,
                  "inner evidence row count")

    search_rows, selected_rows, trace, selected_by_outer = rebuild_outer_selection(
        runner, subject_names, grid, table
    )
    evidence_checks["outer_selected_configs_json"] = compare_json_exact(
        required["selected_json"], selected_rows, "outer_selected_configs.json", runner
    )
    evidence_checks["outer_selection_trace"] = compare_json_exact(
        required["trace"], trace, "outer_selection_trace.json", runner
    )

    selected_csv = evidence_dir / "outer_selected_configs.csv"
    if selected_csv.is_file():
        evidence_checks["outer_selected_configs_csv"] = compare_csv_exact(
            selected_csv, selected_rows, SELECTED_FIELDS,
            "outer_selected_configs.csv",
        )
    search_csv = evidence_dir / "glsd_search_all_90x94.csv"
    if search_csv.is_file():
        evidence_checks["glsd_search_all_90x94"] = compare_csv_exact(
            search_csv, search_rows, SEARCH_FIELDS, "glsd_search_all_90x94.csv"
        )

    (
        per_subject_rows, per_video_rows, per_prediction_rows, glsd_raw, glsd_full,
    ) = replay_selected_outer(
        runner, subject_names, grouped, feature_bank, final_samples, final_emotions,
        cache_config, selected_by_outer, table,
    )
    evidence_checks["per_subject_counts"] = compare_csv_exact(
        required["per_subject"], per_subject_rows, SUBJECT_FIELDS,
        "per_subject_counts.csv",
    )
    per_video_path = evidence_dir / "per_video_evaluation.csv"
    if per_video_path.is_file():
        evidence_checks["per_video_evaluation"] = compare_csv_exact(
            per_video_path, per_video_rows, VIDEO_FIELDS, "per_video_evaluation.csv"
        )
    per_prediction_path = evidence_dir / "per_prediction_evaluation.csv"
    if per_prediction_path.is_file():
        evidence_checks["per_prediction_evaluation"] = compare_csv_exact(
            per_prediction_path, per_prediction_rows, PREDICTION_FIELDS,
            "per_prediction_evaluation.csv",
        )

    manifest = provenance["manifest"]
    manifest_native = manifest["native_exact_replay"]
    require_equal(runner.normalize(native), manifest_native,
                  "manifest Native exact replay evidence")
    require_equal(list(glsd_raw), manifest["glsd_final_raw_counts"],
                  "manifest GLSD raw replay")
    require_equal(list(glsd_full), manifest["glsd_final_full_counts"],
                  "manifest GLSD full replay")

    native_full = as_counts(native["full_counts"], "Native full")
    native_metric = runner.metrics(native_full)
    glsd_metric = runner.metrics(glsd_full)
    require_equal(native_metric["f1"], runner.metrics(EXPECTED_NATIVE_FULL)["f1"],
                  "Native full-count F1")
    require_equal(glsd_metric["f1"], manifest["glsd_final_full_metrics"]["f1"],
                  "GLSD full-count F1 vs manifest")

    runner_status = (
        "EXACT_MATCH" if provenance["runner_sha_match"]
        else "SHA_MISMATCH_FUNCTIONALLY_REPLAYED"
    )
    input_hashes = {
        label: {"path": str(path), "sha256": sha256_file(path)}
        for label, path in required.items()
    }
    recovery = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "status": "PASS",
        "preflight_only": True,
        "ablation_started": False,
        "bootstrap_started": False,
        "historical_manifest_modified": False,
        "input_hashes": input_hashes,
        "provenance": {
            "historical_cache_sha256": provenance["historical_cache_sha256"],
            "current_cache_sha256": cache_sha,
            "cache": "EXACT_MATCH",
            "historical_execution_wrapper_sha256":
                provenance["historical_execution_wrapper_sha256"],
            "historical_core_runner_sha256":
                provenance["historical_core_runner_sha256"],
            "current_runner_sha256": runner_sha,
            "runner_sha_match": provenance["runner_sha_match"],
            "runner": runner_status,
        },
        "universe": {
            "subjects": len(subject_names), "videos": len(videos), "gt": gt_count,
            "forbidden_controlled_cache_gt": FORBIDDEN_CONTROLLED_GT,
        },
        "protocol": {
            "configs": len(grid), "outer_folds": EXPECTED_OUTER_FOLDS,
            "inner_subjects_per_fold": EXPECTED_INNER_SUBJECTS,
            "inner_evidence_rows": EXPECTED_INNER_ROWS,
            "selection_metric": "pooled inner raw Spotting F1",
            "paper_metric": "full-count Spotting F1 after official recognition synergy",
            "tie_break": [
                "higher_F1", "higher_precision", "fewer_FP", "fixed_grid_order",
            ],
        },
        "native": {
            "raw_counts": list(as_counts(native["raw_counts"], "Native raw")),
            "full_counts": list(native_full), "full_f1": native_metric["f1"],
            "locked_delta_counts": [0, 0, 0], "locked_delta_f1": 0.0,
            "exact_replay": runner.normalize(native),
        },
        "glsd_90": {
            "raw_counts": list(glsd_raw), "full_counts": list(glsd_full),
            "full_f1": glsd_metric["f1"], "locked_delta_counts": [0, 0, 0],
            "locked_delta_f1": 0.0,
        },
        "evidence_exact_comparisons": evidence_checks,
        "all_functional_checks_passed": True,
    }
    write_recovery_outputs(
        args.output.resolve(), recovery, table, selected_rows, per_subject_rows, runner
    )

    print(f"cache path = {cache_path}")
    print(f"cache SHA256 = {cache_sha}")
    print(
        "historical execution wrapper SHA256 = "
        f"{provenance['historical_execution_wrapper_sha256']}"
    )
    print(
        "historical core runner SHA256 = "
        f"{provenance['historical_core_runner_sha256']}"
    )
    print(f"current runner SHA256 = {runner_sha}")
    print(f"subjects = {len(subject_names)}")
    print(f"videos = {len(videos)}")
    print(f"GT = {gt_count}")
    print(
        "Native full-count TP/FP/FN/F1 = "
        f"{native_full[0]}/{native_full[1]}/{native_full[2]}/{native_metric['f1']:.10f}"
    )
    print(
        "GLSD-90 full-count TP/FP/FN/F1 = "
        f"{glsd_full[0]}/{glsd_full[1]}/{glsd_full[2]}/{glsd_metric['f1']:.10f}"
    )
    print("Native locked delta TP/FP/FN/F1 = (0, 0, 0)/+0.000000000000e+00")
    print("GLSD-90 locked delta TP/FP/FN/F1 = (0, 0, 0)/+0.000000000000e+00")
    print("BOOSTING_CASME3_FUNCTIONAL_PREFLIGHT = PASS")
    print("CACHE_PROVENANCE = EXACT_MATCH")
    print(f"RUNNER_PROVENANCE = {runner_status}")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--runner", type=Path, default=DEFAULT_RUNNER)
    parser.add_argument(
        "--official-repo-root", type=Path, default=DEFAULT_OFFICIAL_REPO_ROOT,
        help="directory containing the official training_utils.py",
    )
    parser.add_argument("--cache", type=Path, default=DEFAULT_CACHE)
    parser.add_argument("--evidence-dir", type=Path, default=DEFAULT_EVIDENCE_DIR)
    parser.add_argument("--output", type=Path, required=True)
    parser.add_argument("--preflight-only", action="store_true")
    args = parser.parse_args()
    if not args.preflight_only:
        parser.error(
            "phase one exposes --preflight-only only; ablation/bootstrap remain locked"
        )
    if args.output.exists():
        parser.error(f"refusing to overwrite output path: {args.output}")
    try:
        run_preflight(args)
    except Exception as exc:
        print(f"BOOSTING_CASME3_FUNCTIONAL_PREFLIGHT = HARD_FAIL: {exc}", file=sys.stderr)
        raise


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/transfer_bundle/boosting_casme3_official_phase2.py
#!/usr/bin/env python3
"""Official-response Phase 2 for BoostingVRME / CAS(ME)3."""

from __future__ import annotations

import argparse
import contextlib
import csv
import hashlib
import importlib.util
import io
import json
import sys
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace

import numpy as np

from phase2_common import (
    FIXED_RHO,
    MODES,
    component_grid,
    component_scores,
    component_summaries,
    nested_loso_selection,
    paired_subject_bootstrap,
    run_protocol_smoke_assertions,
    threshold_candidates,
)


EXPECTED_SUBJECTS = 94
EXPECTED_VIDEOS = 462
EXPECTED_GT = 853
EXPECTED_NATIVE_FULL = (84, 786, 769)
EXPECTED_GLSD_RAW = (120, 1113, 733)
EXPECTED_GLSD_FULL = (120, 1110, 733)

DEFAULT_RUNNER = Path("/content/BoostingVRME/boosting_official_glds_full_casme3.py")
DEFAULT_REPO_ROOT = Path("/content/BoostingVRME")
DEFAULT_CACHE = Path(
    "/content/drive/MyDrive/GLSD_BoostingVRME_Full/CASME3/"
    "official_response_cache/casme3_official_full_responses.pkl"
)
DEFAULT_EVIDENCE = Path(
    "/content/drive/MyDrive/GLSD_BoostingVRME_Full/CASME3/glsd_full_evidence_v1"
)
DEFAULT_PREFLIGHT = Path(__file__).with_name(
    "boosting_casme3_official_response_preflight.py"
)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path: Path, value) -> None:
    path.write_text(
        json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
        encoding="utf-8",
    )


def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        raise RuntimeError(f"refusing to write empty CSV: {path}")
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def load_module(path: Path, name: str, repo_root: Path | None = None):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"cannot import module: {path}")
    module = importlib.util.module_from_spec(spec)
    old_path = list(sys.path)
    additions = [str(path.parent)]
    if repo_root is not None:
        additions.insert(0, str(repo_root))
    sys.path[:0] = additions
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except Exception:
        sys.modules.pop(name, None)
        raise
    finally:
        sys.path[:] = old_path
    return module


def cas_component_candidates(feature, config, mode: str) -> list[int]:
    evidence = feature.scores(config.reference_scale, FIXED_RHO)
    peaks = np.asarray(evidence["peaks"], dtype=int)
    if not len(peaks):
        return []
    width = feature.scale_to_width[float(config.reference_scale)]
    reference_peaks = np.asarray(feature.peaks[width], dtype=int)
    curve = np.asarray(feature.curves[width], dtype=float)
    if not np.array_equal(peaks, reference_peaks):
        raise AssertionError("CAS(ME)3 component changed the runner candidate universe")
    scores = component_scores(
        mode, curve, peaks, evidence["G"], evidence["L"]
    )
    return [
        int(value) for value in threshold_candidates(peaks, scores, config.threshold)
    ]


class CASComponentFeature:
    def __init__(self, base, mode: str):
        self.base = base
        self.mode = mode

    def selected_peaks(self, reference_scale, _rho, threshold):
        config = SimpleNamespace(
            reference_scale=float(reference_scale), threshold=float(threshold)
        )
        return cas_component_candidates(self.base, config, self.mode)


def component_config_dict(config) -> dict:
    return {
        "config_id": int(config.config_id),
        "reference_scale": float(config.reference_scale),
        "local_radius": FIXED_RHO,
        "threshold": float(config.threshold),
        "grid_order": int(config.config_id),
    }


def full_counts_from_recognition(raw, pred_list, gt_list, cache_config):
    neutral = int(cache_config["emotion_type"]) - 1
    neutral_tp = 0
    neutral_fp = 0
    for prediction, target in zip(pred_list, gt_list):
        if int(prediction) == neutral:
            if int(target) == -1:
                neutral_fp += 1
            else:
                neutral_tp += 1
    return (
        int(raw[0]) - neutral_tp,
        int(raw[1]) - neutral_fp,
        int(raw[2]) + neutral_tp,
    )


def native_subject_counts(
    runner, cache_config, grouped, final_samples, final_emotions, subject_index,
):
    videos = grouped[subject_index]
    responses = [np.asarray(video["spot_response"], dtype=float) for video in videos]
    sequences = [np.asarray(video["recognition_sequence"]) for video in videos]
    metric_final = runner.tu.MeanAveragePrecision2d(num_classes=1)
    predictions, _, total_gt, metric_video, metric_final = runner.tu.spotting(
        final_samples, subject_index, responses, 0, metric_final,
        int(cache_config["k_p"]), int(cache_config["interval_strategy"]), runner.DATASET,
    )
    raw = tuple(int(value) for value in runner.official_metric_counts(metric_final, total_gt))
    with contextlib.redirect_stdout(io.StringIO()):
        pred_list, _, gt_list, _, _ = runner.tu.recognition(
            sequences, predictions, metric_video, final_emotions, subject_index,
            [], [], final_samples, [], [], int(cache_config["frame_skip"]),
            int(cache_config["penalty_strategy"]),
        )
    full = full_counts_from_recognition(raw, pred_list, gt_list, cache_config)
    return raw, full


def decode_component(
    runner, grouped, feature_bank, final_samples, final_emotions,
    cache_config, subject_index, config, with_recognition,
):
    config_item = component_config_dict(config)
    decoded = runner.decode_subject(
        subject_index, config_item, grouped, feature_bank, final_samples,
        final_emotions, cache_config, need_recognition=with_recognition,
    )
    raw = tuple(int(value) for value in decoded["raw_counts"])
    if not with_recognition:
        return raw, None
    if "full_counts" not in decoded:
        raise RuntimeError("CAS(ME)3 full counts did not come from official recognition")
    full = tuple(int(value) for value in decoded["full_counts"])
    _, video_rows, _ = runner.subject_prediction_evidence(
        subject_index, str(grouped[subject_index][0]["subject"]), config_item,
        decoded, grouped, final_samples, cache_config,
    )
    video_raw = tuple(sum(int(row[f"raw_{name}"]) for row in video_rows) for name in ("tp", "fp", "fn"))
    video_full = tuple(sum(int(row[f"full_{name}"]) for row in video_rows) for name in ("tp", "fp", "fn"))
    if video_raw != raw or video_full != full:
        raise RuntimeError("CAS(ME)3 official evidence recount mismatch")
    return raw, full


def load_locked_glsd_counts(path: Path, subjects: list[str]):
    with path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != len(subjects):
        raise RuntimeError("CAS(ME)3 sealed per-subject row count mismatch")
    raw, full = [], []
    for index, row in enumerate(rows):
        if int(row["outer_subject_index"]) != index or str(row["outer_subject"]) != subjects[index]:
            raise RuntimeError(f"CAS(ME)3 sealed subject order mismatch at {index}")
        raw.append(tuple(int(row[f"raw_{name}"]) for name in ("tp", "fp", "fn")))
        full.append(tuple(int(row[f"full_{name}"]) for name in ("tp", "fp", "fn")))
    return np.asarray(raw, dtype=np.int64), np.asarray(full, dtype=np.int64)


def run_locked_preflight(args, output: Path) -> Path:
    preflight_path = args.preflight_script.resolve()
    if not preflight_path.is_file():
        raise RuntimeError(f"missing successful CAS(ME)3 preflight implementation: {preflight_path}")
    preflight = load_module(preflight_path, "boosting_casme3_phase2_preflight")
    preflight_output = output / "locked_preflight"
    preflight.run_preflight(SimpleNamespace(
        runner=args.runner.resolve(),
        official_repo_root=args.official_repo_root.resolve(),
        cache=args.cache.resolve(),
        evidence_dir=args.evidence_dir.resolve(),
        output=preflight_output,
    ))
    recovery_path = preflight_output / "recovery_manifest.json"
    if not recovery_path.is_file():
        raise RuntimeError("CAS(ME)3 preflight did not produce recovery_manifest.json")
    recovery = json.loads(recovery_path.read_text(encoding="utf-8"))
    if recovery.get("status") != "PASS" or recovery.get("all_functional_checks_passed") is not True:
        raise RuntimeError("CAS(ME)3 functional preflight did not pass")
    return recovery_path


def raw_table_rows(mode, subjects, grid, table):
    rows = []
    for config_id, config in enumerate(grid):
        for subject_index, subject in enumerate(subjects):
            tp, fp, fn = (int(value) for value in table[config_id, subject_index])
            rows.append({
                "component": mode, "config_id": config_id,
                "reference_scale": config.reference_scale,
                "rho": FIXED_RHO if mode in ("L", "G+L") else "not_searched",
                "threshold": config.threshold, "subject_index": subject_index,
                "subject": subject, "raw_TP": tp, "raw_FP": fp, "raw_FN": fn,
            })
    return rows


def run_phase2(args) -> None:
    output = args.output.resolve()
    if output.exists():
        raise RuntimeError(f"refusing to overwrite output: {output}")
    recovery_path = run_locked_preflight(args, output)
    print("BOOSTING_CASME3_LOCKED_PREFLIGHT_GATE = PASS")

    runner = load_module(
        args.runner.resolve(), "boosting_casme3_phase2_runner",
        args.official_repo_root.resolve(),
    )
    cache, videos = runner.load_cache(args.cache.resolve())
    grouped = runner.group_videos(videos)
    subjects = [str(subject[0]["subject"]) for subject in grouped]
    gt = sum(len(video["gt_intervals"]) for video in videos)
    if len(subjects) != EXPECTED_SUBJECTS or len(videos) != EXPECTED_VIDEOS or gt != EXPECTED_GT:
        raise RuntimeError("CAS(ME)3 official universe mismatch after preflight")
    if len(set(subjects)) != EXPECTED_SUBJECTS:
        raise RuntimeError("duplicate CAS(ME)3 subject IDs")

    cache_config = cache["config"]
    final_samples = [[video["gt_intervals"] for video in subject] for subject in grouped]
    final_emotions = [[video["emotion_labels"] for video in subject] for subject in grouped]
    k = int(cache_config["k_p"])
    base_feature_bank = [
        [runner.GLSDFeatures(video["spot_response"], k) for video in subject]
        for subject in grouped
    ]

    native_raw_rows, native_full_rows = [], []
    for subject_index in range(EXPECTED_SUBJECTS):
        raw, full = native_subject_counts(
            runner, cache_config, grouped, final_samples, final_emotions, subject_index
        )
        native_raw_rows.append(raw)
        native_full_rows.append(full)
    native_raw = np.asarray(native_raw_rows, dtype=np.int64)
    native_full = np.asarray(native_full_rows, dtype=np.int64)
    if tuple(native_full.sum(axis=0)) != EXPECTED_NATIVE_FULL:
        raise RuntimeError("CAS(ME)3 Native per-subject full aggregate mismatch")

    sealed_subject_path = args.evidence_dir.resolve() / "per_subject_counts.csv"
    glsd_raw, glsd_full = load_locked_glsd_counts(sealed_subject_path, subjects)
    if tuple(glsd_raw.sum(axis=0)) != EXPECTED_GLSD_RAW:
        raise RuntimeError("CAS(ME)3 locked GLSD-90 raw aggregate mismatch")
    if tuple(glsd_full.sum(axis=0)) != EXPECTED_GLSD_FULL:
        raise RuntimeError("CAS(ME)3 locked GLSD-90 full aggregate mismatch")

    grid = component_grid()
    if len(grid) != 30:
        raise AssertionError("CAS(ME)3 component grid is not 30 configs")
    pooled_raw_rows = []
    pooled_full_rows = []
    for mode in MODES:
        slug = mode.replace("+", "_plus_")
        mode_dir = output / f"component_{slug}"
        mode_dir.mkdir()
        feature_bank = [
            [CASComponentFeature(feature, mode) for feature in subject]
            for subject in base_feature_bank
        ]
        raw_table = np.zeros((30, EXPECTED_SUBJECTS, 3), dtype=np.int64)
        for subject_index in range(EXPECTED_SUBJECTS):
            for config_id, config in enumerate(grid):
                raw, _ = decode_component(
                    runner, grouped, feature_bank, final_samples, final_emotions,
                    cache_config, subject_index, config, False,
                )
                raw_table[config_id, subject_index] = raw
        winners, winner_rows, traces = nested_loso_selection(raw_table, subjects, grid)

        subject_rows = []
        raw_subject_counts = []
        full_subject_counts = []
        for subject_index, subject in enumerate(subjects):
            config_id = winners[subject_index]
            raw, full = decode_component(
                runner, grouped, feature_bank, final_samples, final_emotions,
                cache_config, subject_index, grid[config_id], True,
            )
            expected_raw = tuple(int(value) for value in raw_table[config_id, subject_index])
            if raw != expected_raw:
                raise RuntimeError(f"{mode} selected outer raw cell mismatch at {subject}")
            if full is None:
                raise RuntimeError("full counts did not come from official recognition/synergy")
            raw_subject_counts.append(raw)
            full_subject_counts.append(full)
            subject_rows.append({
                "component": mode,
                "component_label": "matched-budget-G+L-30" if mode == "G+L" else mode,
                "subject_index": subject_index, "subject": subject,
                "config_id": config_id,
                "reference_scale": grid[config_id].reference_scale,
                "rho": FIXED_RHO if mode in ("L", "G+L") else "not_searched",
                "threshold": grid[config_id].threshold,
                "raw_TP": raw[0], "raw_FP": raw[1], "raw_FN": raw[2],
                "full_TP": full[0], "full_FP": full[1], "full_FN": full[2],
            })
        raw_summary, full_summary = component_summaries(
            raw_subject_counts, full_subject_counts
        )
        expected_full = tuple(int(value) for value in np.sum(full_subject_counts, axis=0))
        if (full_summary["TP"], full_summary["FP"], full_summary["FN"]) != expected_full:
            raise AssertionError("CAS(ME)3 full summary reused raw aggregate")
        label = "matched-budget-G+L-30" if mode == "G+L" else mode
        pooled_raw_rows.append({"component": mode, "component_label": label, **raw_summary})
        pooled_full_rows.append({"component": mode, "component_label": label, **full_summary})
        np.save(mode_dir / "raw_counts_30x94x3.npy", raw_table)
        write_csv(mode_dir / "raw_counts_30x94.csv", raw_table_rows(mode, subjects, grid, raw_table))
        write_csv(mode_dir / "outer_winner_configs.csv", winner_rows)
        write_json(mode_dir / "selection_trace.json", traces)
        write_csv(mode_dir / "per_subject_raw_full_counts.csv", subject_rows)

    write_csv(output / "component_pooled_raw_summary.csv", pooled_raw_rows)
    write_csv(output / "component_pooled_full_summary.csv", pooled_full_rows)

    locked_rows = []
    for index, subject in enumerate(subjects):
        native = native_full[index]
        glsd = glsd_full[index]
        locked_rows.append({
            "subject_index": index, "subject": subject,
            "Native_full_TP": int(native[0]), "Native_full_FP": int(native[1]),
            "Native_full_FN": int(native[2]), "GLSD90_full_TP": int(glsd[0]),
            "GLSD90_full_FP": int(glsd[1]), "GLSD90_full_FN": int(glsd[2]),
        })
    write_csv(output / "native_glsd90_per_subject_full_counts.csv", locked_rows)
    bootstrap = paired_subject_bootstrap(native_full, glsd_full)
    np.save(output / "paired_bootstrap_subject_indexes.npy", bootstrap["indexes"])
    np.save(output / "paired_bootstrap_native_f1_samples.npy", bootstrap["native_f1_samples"])
    np.save(output / "paired_bootstrap_glsd_f1_samples.npy", bootstrap["glsd_f1_samples"])
    np.save(output / "paired_bootstrap_delta_f1_samples.npy", bootstrap["delta_f1_samples"])
    write_csv(output / "paired_bootstrap_summary.csv", [bootstrap["summary"]])

    script_path = Path(__file__).resolve()
    common_path = script_path.with_name("phase2_common.py")
    manifest = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "dataset": "CASME_3", "status": "PASS", "locked_preflight": "PASS",
        "universe": {"subjects": 94, "videos": 462, "gt_events": 853},
        "sha256": {
            "script": sha256_file(script_path), "common": sha256_file(common_path),
            "runner": sha256_file(args.runner.resolve()),
            "cache": sha256_file(args.cache.resolve()),
            "preflight_script": sha256_file(args.preflight_script.resolve()),
            "preflight_recovery_manifest": sha256_file(recovery_path),
            "sealed_per_subject": sha256_file(sealed_subject_path),
        },
        "locked_results": {
            "Native_full": list(EXPECTED_NATIVE_FULL),
            "GLSD90_raw": list(EXPECTED_GLSD_RAW),
            "GLSD90_full": list(EXPECTED_GLSD_FULL),
        },
        "component_protocol": {
            "modes": list(MODES), "configs_per_mode": 30,
            "reference_scales": [1.0, 1.5, 2.0],
            "thresholds": [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.55, 0.60, 0.65, 0.75],
            "rho": {"H": "not_searched", "G": "not_searched", "L": 2.0, "G+L": 2.0},
            "grid_order": "a0-major,tau-minor",
            "selection": "pooled inner raw F1, precision, fewer FP, grid order",
            "paper_metric": "pooled full-count Spotting F1",
            "G+L_label": "matched-budget-G+L-30; not locked GLSD-90",
        },
        "bootstrap": bootstrap["summary"],
    }
    write_json(output / "run_manifest.json", manifest)
    print("BOOSTING_CASME3_OFFICIAL_PHASE2 = PASS")
    print("OUTPUT =", output)


def run_smoke_test() -> None:
    run_protocol_smoke_assertions()

    class FakeCASFeature:
        def __init__(self):
            self._peaks = np.asarray([1, 3])
            self._curve = np.asarray([0.0, 2.0, 0.0, 1.0, 0.0])
            self.scale_to_width = {1.0: 2, 1.5: 2, 2.0: 2}
            self.peaks = {2: self._peaks}
            self.curves = {2: self._curve}

        def scores(self, reference, rho):
            assert rho == FIXED_RHO
            return {
                "peaks": self._peaks,
                "G": np.asarray([0.8, 0.4]),
                "L": np.asarray([0.2, 0.6]),
                "S": np.asarray([0.5, 0.5]),
            }

    feature = FakeCASFeature()
    config = component_grid()[5]  # tau=0.5 separates the four score paths.
    outputs = {
        mode: cas_component_candidates(feature, config, mode) for mode in MODES
    }
    expected = {"H": [1], "G": [1], "L": [3], "G+L": [1, 3]}
    if outputs != expected:
        raise AssertionError(f"CAS(ME)3 component score path mismatch: {outputs}")
    print("BOOSTING_CASME3_PHASE2_SMOKE = PASS")


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--runner", type=Path, default=DEFAULT_RUNNER)
    parser.add_argument("--official-repo-root", type=Path, default=DEFAULT_REPO_ROOT)
    parser.add_argument("--cache", type=Path, default=DEFAULT_CACHE)
    parser.add_argument("--evidence-dir", type=Path, default=DEFAULT_EVIDENCE)
    parser.add_argument("--preflight-script", type=Path, default=DEFAULT_PREFLIGHT)
    parser.add_argument("--output", type=Path)
    parser.add_argument("--smoke-test", action="store_true")
    args = parser.parse_args()
    if args.smoke_test:
        run_smoke_test()
        return
    if args.output is None:
        parser.error("--output is required for a formal Phase-2 run")
    run_phase2(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/transfer_bundle/phase2_common.py
"""Pure protocol helpers for BoostingVRME official-response Phase 2."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np


MODES = ("H", "G", "L", "G+L")
SCALES = (1.0, 1.5, 2.0)
THRESHOLDS = (0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.55, 0.60, 0.65, 0.75)
FIXED_RHO = 2.0
BOOTSTRAP_SEED = 100
BOOTSTRAP_ITERATIONS = 10_000


@dataclass(frozen=True)
class ComponentConfig:
    config_id: int
    reference_scale: float
    threshold: float
    local_radius: float = FIXED_RHO

    @property
    def identifier(self) -> str:
        return f"a0={self.reference_scale:g}|tau={self.threshold:g}"


def component_grid() -> list[ComponentConfig]:
    grid = [
        ComponentConfig(index, scale, threshold)
        for index, (scale, threshold) in enumerate(
            (scale, threshold) for scale in SCALES for threshold in THRESHOLDS
        )
    ]
    if len(grid) != 30:
        raise AssertionError("matched-budget component grid must contain 30 configs")
    return grid


def height_scores(curve, peaks) -> np.ndarray:
    curve = np.asarray(curve, dtype=float)
    peaks = np.asarray(peaks, dtype=int)
    if curve.ndim != 1 or np.any(peaks < 0) or np.any(peaks >= len(curve)):
        raise ValueError("invalid reference curve or candidate indexes")
    if not len(peaks):
        return np.empty(0, dtype=float)
    centre = float(np.mean(curve))
    denominator = max(float(np.max(curve) - centre), 1e-12)
    return np.maximum(0.0, (curve[peaks] - centre) / denominator)


def component_scores(mode: str, curve, peaks, global_values, local_values) -> np.ndarray:
    if mode not in MODES:
        raise ValueError(f"unknown component mode: {mode}")
    peaks = np.asarray(peaks, dtype=int)
    global_values = np.asarray(global_values, dtype=float)
    local_values = np.asarray(local_values, dtype=float)
    if not (len(peaks) == len(global_values) == len(local_values)):
        raise ValueError("H/G/L evidence cardinality mismatch")
    if mode == "H":
        return height_scores(curve, peaks)
    if mode == "G":
        return global_values
    if mode == "L":
        return local_values
    return (global_values + local_values) / 2.0


def threshold_candidates(peaks, scores, threshold: float) -> np.ndarray:
    peaks = np.asarray(peaks, dtype=int)
    scores = np.asarray(scores, dtype=float)
    if len(peaks) != len(scores):
        raise ValueError("candidate/score cardinality mismatch")
    return peaks[scores >= float(threshold)]


def counts_tuple(counts) -> tuple[int, int, int]:
    result = tuple(int(value) for value in counts)
    if len(result) != 3 or any(value < 0 for value in result):
        raise ValueError(f"invalid TP/FP/FN counts: {result}")
    return result


def pool_counts(rows) -> tuple[int, int, int]:
    array = np.asarray(list(rows), dtype=np.int64)
    if array.ndim != 2 or array.shape[1] != 3 or np.any(array < 0):
        raise ValueError("counts must have shape (N, 3) and be nonnegative")
    return counts_tuple(array.sum(axis=0))


def metrics(counts) -> dict:
    tp, fp, fn = counts_tuple(counts)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "F1": f1,
    }


def selection_order(pooled_raw_counts: np.ndarray) -> np.ndarray:
    counts = np.asarray(pooled_raw_counts, dtype=np.int64)
    if counts.ndim != 2 or counts.shape[1] != 3 or np.any(counts < 0):
        raise ValueError("pooled raw counts must have shape (configs, 3)")
    tp, fp, fn = counts.astype(float).T
    denominator = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, denominator, out=np.zeros_like(tp), where=denominator > 0)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    grid_order = np.arange(len(counts))
    return np.lexsort((grid_order, fp, -precision, -f1))


def nested_loso_selection(raw_table: np.ndarray, subjects: list[str], grid):
    table = np.asarray(raw_table, dtype=np.int64)
    expected = (len(grid), len(subjects), 3)
    if table.shape != expected or np.any(table < 0):
        raise ValueError(f"raw count table shape={table.shape}, expected={expected}")
    if len(grid) != 30:
        raise AssertionError("component selection requires exactly 30 configs")

    winners = []
    winner_rows = []
    traces = []
    for outer, subject in enumerate(subjects):
        inner = [index for index in range(len(subjects)) if index != outer]
        if len(inner) != len(subjects) - 1 or outer in inner:
            raise AssertionError("invalid nested LOSO inner subject universe")
        pooled = table[:, inner, :].sum(axis=1)
        order = selection_order(pooled)
        winner = int(order[0])
        ranks = np.empty(len(grid), dtype=np.int64)
        ranks[order] = np.arange(1, len(grid) + 1)
        winner_metric = metrics(pooled[winner])
        winners.append(winner)
        winner_rows.append({
            "outer_subject_index": outer,
            "outer_subject": subject,
            "inner_subject_count": len(inner),
            "config_id": winner,
            "reference_scale": grid[winner].reference_scale,
            "threshold": grid[winner].threshold,
            "rho": FIXED_RHO,
            "pooled_inner_raw_TP": winner_metric["TP"],
            "pooled_inner_raw_FP": winner_metric["FP"],
            "pooled_inner_raw_FN": winner_metric["FN"],
            "pooled_inner_raw_precision": winner_metric["precision"],
            "pooled_inner_raw_recall": winner_metric["recall"],
            "pooled_inner_raw_F1": winner_metric["F1"],
            "winner_rank": int(ranks[winner]),
        })
        candidates = []
        for config_id, config in enumerate(grid):
            value = metrics(pooled[config_id])
            candidates.append({
                "config_id": config_id,
                "reference_scale": config.reference_scale,
                "threshold": config.threshold,
                "rho": FIXED_RHO,
                "pooled_inner_raw_TP": value["TP"],
                "pooled_inner_raw_FP": value["FP"],
                "pooled_inner_raw_FN": value["FN"],
                "precision": value["precision"],
                "recall": value["recall"],
                "F1": value["F1"],
                "rank": int(ranks[config_id]),
                "selected": config_id == winner,
            })
        traces.append({
            "outer_subject_index": outer,
            "outer_subject": subject,
            "inner_subject_indices": inner,
            "inner_subjects": [subjects[index] for index in inner],
            "tie_break": [
                "higher_raw_F1",
                "higher_raw_precision",
                "fewer_raw_FP",
                "fixed_grid_order",
            ],
            "winner_config_id": winner,
            "candidates": candidates,
        })
    return winners, winner_rows, traces


def component_summaries(raw_rows, full_rows) -> tuple[dict, dict]:
    raw_total = pool_counts(raw_rows)
    full_total = pool_counts(full_rows)
    raw_summary = {"metric_stage": "raw_audit", **metrics(raw_total)}
    full_summary = {"metric_stage": "full_official_recognition_synergy", **metrics(full_total)}
    if full_summary["TP"] != full_total[0] or full_summary["FP"] != full_total[1]:
        raise AssertionError("full summary was not computed from full counts")
    return raw_summary, full_summary


def paired_subject_bootstrap(
    native_full: np.ndarray,
    glsd_full: np.ndarray,
    iterations: int = BOOTSTRAP_ITERATIONS,
    seed: int = BOOTSTRAP_SEED,
) -> dict:
    native = np.asarray(native_full, dtype=np.int64)
    glsd = np.asarray(glsd_full, dtype=np.int64)
    if native.shape != glsd.shape or native.ndim != 2 or native.shape[1] != 3:
        raise ValueError("paired full-count arrays must share shape (subjects, 3)")
    if len(native) < 2 or np.any(native < 0) or np.any(glsd < 0):
        raise ValueError("invalid subject-level full counts")
    if iterations != 10_000 or seed != 100:
        raise AssertionError("locked bootstrap requires iterations=10000 and seed=100")

    rng = np.random.default_rng(seed)
    indexes = rng.integers(0, len(native), size=(iterations, len(native)))
    native_totals = native[indexes].sum(axis=1)
    glsd_totals = glsd[indexes].sum(axis=1)

    def vector_f1(totals):
        totals = totals.astype(float)
        denominator = 2 * totals[:, 0] + totals[:, 1] + totals[:, 2]
        return np.divide(
            2 * totals[:, 0], denominator,
            out=np.zeros(len(totals), dtype=float), where=denominator > 0,
        )

    native_samples = vector_f1(native_totals)
    glsd_samples = vector_f1(glsd_totals)
    delta_samples = glsd_samples - native_samples
    point_delta = metrics(glsd.sum(axis=0))["F1"] - metrics(native.sum(axis=0))["F1"]
    return {
        "indexes": indexes,
        "native_f1_samples": native_samples,
        "glsd_f1_samples": glsd_samples,
        "delta_f1_samples": delta_samples,
        "summary": {
            "statistic": "pooled_full_F1_GLSD_minus_Native",
            "point_delta_F1": point_delta,
            "percentile_2_5": float(np.quantile(delta_samples, 0.025)),
            "percentile_97_5": float(np.quantile(delta_samples, 0.975)),
            "seed": seed,
            "iterations": iterations,
            "unit": "subject",
            "paired": True,
            "aggregation": "sample subjects, pool full TP/FP/FN, then compute F1",
        },
    }


def run_protocol_smoke_assertions() -> None:
    grid = component_grid()
    assert len(grid) == 30
    assert [(item.reference_scale, item.threshold) for item in grid] == [
        (scale, threshold) for scale in SCALES for threshold in THRESHOLDS
    ]

    curve = np.asarray([0.0, 1.0, 3.0, 1.0, 0.0])
    peaks = np.asarray([2])
    h = height_scores(curve, peaks)
    assert np.array_equal(h, np.asarray([1.0]))
    g = np.asarray([0.7])
    local = np.asarray([0.3])
    assert np.array_equal(component_scores("H", curve, peaks, g, local), h)
    assert np.array_equal(component_scores("G", curve, peaks, g, local), g)
    assert np.array_equal(component_scores("L", curve, peaks, g, local), local)
    assert np.array_equal(component_scores("G+L", curve, peaks, g, local), np.asarray([0.5]))

    subjects = ["s0", "s1", "s2"]
    table = np.full((30, 3, 3), (1, 9, 9), dtype=np.int64)
    table[0] = np.asarray([(2, 8, 8), (2, 8, 8), (2, 8, 8)])
    table[1] = np.asarray([(2, 7, 9), (2, 7, 9), (2, 7, 9)])
    winners, rows, _ = nested_loso_selection(table, subjects, grid)
    assert winners == [1, 1, 1]
    assert all(row["inner_subject_count"] == 2 for row in rows)

    raw = np.asarray([(2, 8, 8), (1, 9, 9), (0, 10, 10)])
    full = np.asarray([(1, 6, 9), (1, 7, 9), (0, 8, 10)])
    raw_summary, full_summary = component_summaries(raw, full)
    assert (raw_summary["TP"], raw_summary["FP"], raw_summary["FN"]) == (3, 27, 27)
    assert (full_summary["TP"], full_summary["FP"], full_summary["FN"]) == (2, 21, 28)
    assert full_summary["F1"] == metrics((2, 21, 28))["F1"]

    boot = paired_subject_bootstrap(full, full + np.asarray((1, 0, 0)))
    indexes = boot["indexes"]
    first_native = full[indexes[0]].sum(axis=0)
    first_glsd = (full + np.asarray((1, 0, 0)))[indexes[0]].sum(axis=0)
    expected_delta = metrics(first_glsd)["F1"] - metrics(first_native)["F1"]
    assert boot["delta_f1_samples"][0] == expected_delta
    assert boot["indexes"].shape == (10_000, len(full))
    assert boot["summary"]["paired"] is True
    assert boot["summary"]["unit"] == "subject"


## 4. Dependency-free protocol smoke tests

In [ ]:
%%bash
set -e
/content/micromamba/bin/micromamba run -p /content/boostvrme-env python /content/transfer_bundle/test_protocol.py


## 5. Phase 2: universe and explicit ID mapping audit

This reads only sealed evidence. CAS(ME)3 must retain the explicit `spNO.15_j` exclusion.


In [ ]:
%%bash
set -e
RUN_ROOT=$(cat /content/transfer_run_root.txt)
/content/micromamba/bin/micromamba run -p /content/boostvrme-env python /content/transfer_bundle/transfer_audit.py --output-root "$RUN_ROOT"


## 6. Phase 3: four exact replays

Run all four cells. Each is fail-closed and starts no transfer. The CAS(ME)3 Boosting replay reconstructs the complete locked 90 x 94 table, so it is expected to take substantially longer.


In [ ]:
%%bash
set -e
RUN_ROOT=$(cat /content/transfer_run_root.txt)
/content/bin/micromamba run -n ME-TST env PYTHONPATH=/content/drive/MyDrive/GLSD_METST_OFFICIAL/SAMMLV_FINAL_EVIDENCE:/content/ME-TST:/content/transfer_bundle python /content/transfer_bundle/official_response_component_ablation.py --setting metst_sammlv --preflight-only --output-root "$RUN_ROOT/preflight/metst_sammlv" 2>&1 | tee "$RUN_ROOT/audit/metst_sammlv_preflight.log"


In [ ]:
%%bash
set -e
RUN_ROOT=$(cat /content/transfer_run_root.txt)
/content/bin/micromamba run -n ME-TST env PYTHONPATH=/content/drive/MyDrive/GLSD_METST_OFFICIAL/CASME3_FINAL_EVIDENCE/source:/content/ME-TST:/content/transfer_bundle python /content/transfer_bundle/official_response_component_ablation.py --setting metst_casme3 --preflight-only --output-root "$RUN_ROOT/preflight/metst_casme3" 2>&1 | tee "$RUN_ROOT/audit/metst_casme3_preflight.log"


In [ ]:
%%bash
set -e
RUN_ROOT=$(cat /content/transfer_run_root.txt)
/content/micromamba/bin/micromamba run -p /content/boostvrme-env env PYTHONPATH=/content/BoostingVRME:/content/transfer_bundle python /content/transfer_bundle/official_response_component_ablation.py --setting boosting_sammlv --preflight-only --output-root "$RUN_ROOT/preflight/boosting_sammlv" 2>&1 | tee "$RUN_ROOT/audit/boosting_sammlv_preflight.log"


In [ ]:
%%bash
set -e
RUN_ROOT=$(cat /content/transfer_run_root.txt)
/content/micromamba/bin/micromamba run -p /content/boostvrme-env env PYTHONPATH=/content/BoostingVRME:/content/transfer_bundle python /content/transfer_bundle/boosting_casme3_official_response_preflight.py --preflight-only --runner /content/BoostingVRME/boosting_official_glds_full_casme3.py --official-repo-root /content/BoostingVRME --cache /content/drive/MyDrive/GLSD_BoostingVRME_Full/CASME3/official_response_cache/casme3_official_full_responses.pkl --evidence-dir /content/drive/MyDrive/GLSD_BoostingVRME_Full/CASME3/glsd_full_evidence_v1 --output "$RUN_ROOT/preflight/boosting_casme3" 2>&1 | tee "$RUN_ROOT/audit/boosting_casme3_preflight.log"


In [ ]:
%%bash
set -e
RUN_ROOT=$(cat /content/transfer_run_root.txt)
/content/micromamba/bin/micromamba run -p /content/boostvrme-env env PYTHONPATH=/content/BoostingVRME:/content/transfer_bundle python /content/transfer_bundle/consolidate_preflight.py --output-root "$RUN_ROOT" --metst-sammlv "$RUN_ROOT/preflight/metst_sammlv" --metst-casme3 "$RUN_ROOT/preflight/metst_casme3" --boosting-sammlv "$RUN_ROOT/preflight/boosting_sammlv" --boosting-casme3 "$RUN_ROOT/preflight/boosting_casme3"


## 7. Phases 4-7: configuration transfer and paired bootstrap

These cells cannot run unless the mapping audit and all eight Native/tuned replay checks passed. Each target is executed inside its own original environment. No target-side configuration search is called.


In [ ]:
%%bash
set -e
RUN_ROOT=$(cat /content/transfer_run_root.txt)
/content/micromamba/bin/micromamba run -p /content/boostvrme-env env PYTHONPATH=/content/BoostingVRME:/content/transfer_bundle python /content/transfer_bundle/run_transfer_target.py --direction metst_to_boosting_sammlv --output-root "$RUN_ROOT"


In [ ]:
%%bash
set -e
RUN_ROOT=$(cat /content/transfer_run_root.txt)
/content/bin/micromamba run -n ME-TST env PYTHONPATH=/content/drive/MyDrive/GLSD_METST_OFFICIAL/SAMMLV_FINAL_EVIDENCE:/content/ME-TST:/content/transfer_bundle python /content/transfer_bundle/run_transfer_target.py --direction boosting_to_metst_sammlv --output-root "$RUN_ROOT"


In [ ]:
%%bash
set -e
RUN_ROOT=$(cat /content/transfer_run_root.txt)
/content/micromamba/bin/micromamba run -p /content/boostvrme-env env PYTHONPATH=/content/BoostingVRME:/content/transfer_bundle python /content/transfer_bundle/run_transfer_target.py --direction metst_to_boosting_casme3 --output-root "$RUN_ROOT"


In [ ]:
%%bash
set -e
RUN_ROOT=$(cat /content/transfer_run_root.txt)
/content/bin/micromamba run -n ME-TST env PYTHONPATH=/content/drive/MyDrive/GLSD_METST_OFFICIAL/CASME3_FINAL_EVIDENCE/source:/content/ME-TST:/content/transfer_bundle python /content/transfer_bundle/run_transfer_target.py --direction boosting_to_metst_casme3 --output-root "$RUN_ROOT"


## 8. Phase 8: paper table and final audit

In [ ]:
%%bash
set -e
RUN_ROOT=$(cat /content/transfer_run_root.txt)
/content/micromamba/bin/micromamba run -p /content/boostvrme-env python /content/transfer_bundle/finalize_transfer.py --output-root "$RUN_ROOT"


In [ ]:
from pathlib import Path
from IPython.display import display
import pandas as pd

run_root = Path('/content/transfer_run_root.txt').read_text(encoding='utf-8').strip()
table = pd.read_csv(Path(run_root) / 'summary/paper_table.csv')
display(table)
print((Path(run_root) / 'summary/FINAL_AUDIT.txt').read_text(encoding='utf-8'))
